In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Create a folder in your Drive for Palace
DRIVE_PALACE_DIR = '/content/drive/MyDrive/Palace6'
os.makedirs(DRIVE_PALACE_DIR, exist_ok=True)

# Export the Python variable as an environment variable for subsequent bash cells
os.environ['DRIVE_PALACE_DIR'] = DRIVE_PALACE_DIR

Mounted at /content/drive


In [ ]:
# %%bash
# # Brajabidhu #Palace Installation
# # Set Google Drive target directory
# export DRIVE_PALACE_DIR="/content/drive/MyDrive/Palace6"

# LOCAL_BUILD_DIR="/content/Palace_build_tmp"
# BIN_DEST="$DRIVE_PALACE_DIR/build/bin"
# LIB_DEST="$DRIVE_PALACE_DIR/build/lib"

# # Use 2 parallel jobs to guarantee RAM stability in Colab
# BUILD_JOBS=2

# mkdir -p "$BIN_DEST" "$LIB_DEST" || exit 1

# # Clean old destination files
# rm -rf "$BIN_DEST"/*

# #-------------------------------
# # System Dependencies
# #-------------------------------
# echo "Installing build prerequisites..."
# apt-get update --fix-missing -qq && apt-get install -y -qq \
#     build-essential gfortran pkg-config ninja-build \
#     libopenmpi-dev openmpi-bin libblas-dev liblapack-dev cmake > /dev/null 2>&1

# #-------------------------------
# # Build Palace
# #-------------------------------
# echo "Cloning Palace repository..."
# rm -rf "$LOCAL_BUILD_DIR"
# mkdir -p "$LOCAL_BUILD_DIR"
# cd "$LOCAL_BUILD_DIR" || exit 1

# git clone --recursive https://github.com/awslabs/palace.git source || exit 1

# # Save commit and status
# git -C source rev-parse HEAD > "$DRIVE_PALACE_DIR/build/palace_commit.txt"
# git -C source status > "$DRIVE_PALACE_DIR/build/palace_git_status.txt"

# mkdir -p build && cd build || exit 1

# echo "Configuring CMake..."
# cmake ../source -DCMAKE_BUILD_TYPE=Release -DPALACE_WITH_OPENMP=ON > cmake_log.txt 2>&1

# if [ $? -ne 0 ]; then
#     echo "ERROR: CMake failed." >&2
#     cat cmake_log.txt >&2
#     exit 1
# fi

# echo "Compiling Palace..."
# cmake --build . --parallel "${BUILD_JOBS}" > make_log.txt 2>&1

# if [ $? -ne 0 ]; then
#     echo "ERROR: Build failed." >&2
#     tail -n 60 make_log.txt >&2
#     exit 1
# fi

# # Save logs
# cp CMakeCache.txt cmake_log.txt make_log.txt "$DRIVE_PALACE_DIR/build/"

# #--------------------------------------------------
# # Copy Executables and Libraries
# #--------------------------------------------------
# echo "Copying executables to Google Drive..."

# # Copy EVERYTHING inside the build bin/ directory (both wrapper and palace-x86_64.bin)
# cp -r "$LOCAL_BUILD_DIR/build/bin"/* "$BIN_DEST/"
# chmod +x "$BIN_DEST"/*

# # Verify primary binary
# ACTUAL_BINARY="$BIN_DEST/palace"
# ldd "$ACTUAL_BINARY" > "$BIN_DEST/palace_ldd.txt" 2>&1
# "$ACTUAL_BINARY" --help > "$BIN_DEST/palace_version.txt" 2>&1 || true

# # Copy shared libraries
# echo "Copying shared libraries..."
# while read -r lib_file; do
#     if [[ "$lib_file" =~ \.so($|\.[0-9]+(\.[0-9]+)*$) ]]; then
#         cp -df "$lib_file" "$LIB_DEST/"
#     fi
# done < <(find "$LOCAL_BUILD_DIR/build" -type f -name "*.so*")

# # Save directory tree
# find "$LOCAL_BUILD_DIR/build" > "$DRIVE_PALACE_DIR/build/build_tree.txt"

# # Save helper script
# cat > "$DRIVE_PALACE_DIR/build/use_palace.sh" <<'EOF'
# #!/bin/bash
# export PALACE_HOME="$(cd "$(dirname "$0")" && pwd)"
# export LD_LIBRARY_PATH="$PALACE_HOME/lib:$LD_LIBRARY_PATH"
# "$PALACE_HOME/bin/palace" "$@"
# EOF
# chmod +x "$DRIVE_PALACE_DIR/build/use_palace.sh"

# echo "========================================="
# echo "SUCCESS: Palace build and backup completed!"
# echo "Binaries backed up to: $BIN_DEST"
# echo "========================================="

Installing build prerequisites...
Cloning Palace repository...
Configuring CMake...
Compiling Palace...
Copying executables to Google Drive...
Copying shared libraries...
SUCCESS: Palace build and backup completed!
Binaries backed up to: /content/drive/MyDrive/Palace6/build/bin


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Cloning into 'source'...


In [ ]:
import os
import subprocess

print("Loading and verifying Palace from Google Drive...\n")

# 1. Mount Google Drive if not mounted
if not os.path.exists('/content/drive'):
    print("Mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print("Google Drive already mounted.")

# 2. Paths
DRIVE_PALACE_DIR = '/content/drive/MyDrive/Palace6'
BINARY_PATH = os.path.join(DRIVE_PALACE_DIR, 'build', 'bin', 'palace')
DRIVE_LIB_DIR = os.path.join(DRIVE_PALACE_DIR, 'build', 'lib')

print(f"Loading Palace binary from: {BINARY_PATH}")

# 3. Configure Runtime Environment
env = os.environ.copy()

# Add Drive dynamic libraries to LD_LIBRARY_PATH
current_ld = env.get('LD_LIBRARY_PATH', '')
env['LD_LIBRARY_PATH'] = f"{DRIVE_LIB_DIR}:{current_ld}" if current_ld else DRIVE_LIB_DIR
print(f"LD_LIBRARY_PATH set to: {env['LD_LIBRARY_PATH']}")

# Allow OpenMPI to execute under Colab's root user
env['OMPI_ALLOW_RUN_AS_ROOT'] = '1'
env['OMPI_ALLOW_RUN_AS_ROOT_CONFIRM'] = '1'

# 4. Check & Execute
if os.path.exists(BINARY_PATH):
    os.chmod(BINARY_PATH, 0o755)
    print("Executable permissions verified.")

    print("\nTesting Palace execution...")
    try:
        # Try --version, fallback to --help if needed
        try:
            cmd = [BINARY_PATH, '--version']
            result = subprocess.run(cmd, capture_output=True, text=True, check=True, env=env)
        except subprocess.CalledProcessError:
            cmd = [BINARY_PATH, '--help']
            result = subprocess.run(cmd, capture_output=True, text=True, check=True, env=env)

        print("\n=========================================")
        print("✅ Palace successfully loaded and ran!")
        print(f"Command output ({' '.join(cmd)}):\n")
        print(result.stdout if result.stdout else result.stderr)
        print("=========================================")

    except subprocess.CalledProcessError as e:
        print("\n=========================================")
        print(f"❌ ERROR: Palace failed to run! Exit code: {e.returncode}")
        print(f"Stderr:\n{e.stderr}")
        print("=========================================")
else:
    print(f"\n❌ ERROR: Palace binary not found at: {BINARY_PATH}")

Loading and verifying Palace from Google Drive...

Google Drive already mounted.
Loading Palace binary from: /content/drive/MyDrive/Palace6/build/bin/palace
LD_LIBRARY_PATH set to: /content/drive/MyDrive/Palace6/build/lib
Executable permissions verified.

Testing Palace execution...

✅ Palace successfully loaded and ran!
Command output (/content/drive/MyDrive/Palace6/build/bin/palace --help):

Usage: palace [OPTIONS] CONFIG_FILE

Wrapper for launching Palace using MPI

Options:
  -h, --help                       Show this help message and exit
  -dry-run, --dry-run              Parse configuration file for errors and exit
  -serial, --serial                Call Palace without MPI launcher, default is false
  -np, --np NUM_PROCS              How many MPI processes to use, default is 1
  -nt, --nt NUM_THREADS            Number of OpenMP threads to use for OpenMP builds, default is 1 or the value of OMP_NUM_THREADS in the environment
  -launcher, --launcher LAUNCHER   MPI launcher, defaul

First, let's create a placeholder configuration file that Palace can use for the dry run. For this example, it will be a simple empty file, but in a real scenario, this would contain your simulation settings.

In [ ]:
import os

config_file_path = os.path.join(DRIVE_PALACE_DIR, 'dummy_config.json')
with open(config_file_path, 'w') as f:
    # Providing a more structured JSON to satisfy Palace's parsing requirements
    f.write('''
{
  "Problem": {
    "Type": "Electrostatic",
    "Solver": {
      "Type": "GMRES",
      "RelTol": 1.0e-6,
      "AbsTol": 1.0e-12,
      "MaxIter": 100
    }
  },
  "Domain": {},
  "Boundaries": {}
}
''')

print(f"Updated dummy configuration file: {config_file_path} with a more structured JSON.")


Updated dummy configuration file: /content/drive/MyDrive/Palace6/dummy_config.json with a more structured JSON.


Now, let's run Palace with the `-dry-run` option, passing our dummy configuration file. This command will parse the configuration file for errors without executing a full simulation.

In [ ]:
import os
import glob

build_dir = '/content/drive/MyDrive/Palace6/build'

print("Searching for actual compiled binaries inside build directory...\n")
for root, dirs, files in os.walk(build_dir):
    for file in files:
        full_path = os.path.join(root, file)
        # Find executable files that are NOT shell scripts
        if os.access(full_path, os.X_OK) and not file.endswith('.sh') and not file.endswith('.txt'):
            print(f"Found binary: {full_path}")

Searching for actual compiled binaries inside build directory...

Found binary: /content/drive/MyDrive/Palace6/build/bin/palace
Found binary: /content/drive/MyDrive/Palace6/build/bin/palace-x86_64.bin
Found binary: /content/drive/MyDrive/Palace6/build/bin/validate-config
Found binary: /content/drive/MyDrive/Palace6/build/bin/schema/config-schema.json
Found binary: /content/drive/MyDrive/Palace6/build/bin/schema/ValidateConfig.jl
Found binary: /content/drive/MyDrive/Palace6/build/lib/libxsmm.so.1.17.0
Found binary: /content/drive/MyDrive/Palace6/build/lib/libceed.so
Found binary: /content/drive/MyDrive/Palace6/build/lib/libxsmmgen.so.1.17.0
Found binary: /content/drive/MyDrive/Palace6/build/lib/libxsmm.so.1
Found binary: /content/drive/MyDrive/Palace6/build/lib/libxsmm.so
Found binary: /content/drive/MyDrive/Palace6/build/lib/libxsmmgen.so
Found binary: /content/drive/MyDrive/Palace6/build/lib/libxsmmgen.so.1


In [ ]:
import os

LIB_DIR = '/content/drive/MyDrive/Palace6/build/lib'

# Libraries that require symlinks for runtime binding
symlinks_to_create = [
    ('libxsmm.so.1.17.0', 'libxsmm.so.1'),
    ('libxsmm.so.1.17.0', 'libxsmm.so'),
    ('libxsmmgen.so.1.17.0', 'libxsmmgen.so.1'),
    ('libxsmmgen.so.1.17.0', 'libxsmmgen.so')
]

print("Creating missing library symlinks in Google Drive...\n")

for target, link_name in symlinks_to_create:
    target_path = os.path.join(LIB_DIR, target)
    link_path = os.path.join(LIB_DIR, link_name)

    if os.path.exists(target_path):
        if os.path.islink(link_path) or os.path.exists(link_path):
            os.remove(link_path)
        os.symlink(target, link_path)
        print(f"  [CREATED] {link_name} -> {target}")
    else:
        print(f"  [SKIP] Target file {target} not found in {LIB_DIR}")

print("\n✅ Symlinks created successfully!")

Creating missing library symlinks in Google Drive...

  [CREATED] libxsmm.so.1 -> libxsmm.so.1.17.0
  [CREATED] libxsmm.so -> libxsmm.so.1.17.0
  [CREATED] libxsmmgen.so.1 -> libxsmmgen.so.1.17.0
  [CREATED] libxsmmgen.so -> libxsmmgen.so.1.17.0

✅ Symlinks created successfully!


In [ ]:
import os
import subprocess
import json

print("=========================================")
print("  PALACE INSTALLATION VERIFICATION CHECK ")
print("=========================================\n")

# 1. Mount Google Drive if not already mounted
if not os.path.exists('/content/drive'):
    print("Mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print("✓ Google Drive is mounted.")

DRIVE_PALACE_DIR = '/content/drive/MyDrive/Palace6'
BUILD_DIR = os.path.join(DRIVE_PALACE_DIR, 'build')
BIN_DIR = os.path.join(BUILD_DIR, 'bin')
LIB_DIR = os.path.join(BUILD_DIR, 'lib')

# 2. Check Required Artifacts
required_files = [
    (os.path.join(BIN_DIR, 'palace'), "Palace Launcher Wrapper"),
    (os.path.join(BIN_DIR, 'palace-x86_64.bin'), "Palace C++ Solver Engine"),
    (os.path.join(BIN_DIR, 'validate-config'), "Config Validator Tool"),
    (os.path.join(BIN_DIR, 'palace_ldd.txt'), "Dependency LDD Log"),
    (os.path.join(BIN_DIR, 'palace_version.txt'), "Version/Help Output Log"),
    (os.path.join(BUILD_DIR, 'palace_commit.txt'), "Git Commit Metadata"),
    (os.path.join(BUILD_DIR, 'palace_git_status.txt'), "Git Status Log"),
    (os.path.join(BUILD_DIR, 'CMakeCache.txt'), "CMake Cache"),
    (os.path.join(BUILD_DIR, 'cmake_log.txt'), "CMake Build Log"),
    (os.path.join(BUILD_DIR, 'make_log.txt'), "Make Build Log"),
    (os.path.join(BUILD_DIR, 'build_tree.txt'), "Build Directory Tree Log"),
    (os.path.join(BUILD_DIR, 'use_palace.sh'), "Restore Launcher Script")
]

all_passed = True
print("\n--- 1. Checking Critical Files ---")
for file_path, label in required_files:
    if os.path.exists(file_path):
        print(f"  [PASS] Found {label}: {os.path.basename(file_path)}")
    else:
        print(f"  [FAIL] MISSING {label}: {file_path}")
        all_passed = False

# 3. Check Shared Libraries (.so)
print("\n--- 2. Checking Shared Libraries ---")
if os.path.isdir(LIB_DIR):
    so_files = [f for f in os.listdir(LIB_DIR) if '.so' in f]
    print(f"  [PASS] Found {len(so_files)} shared library files in {LIB_DIR}")
    for so in so_files[:5]:  # Show first few libraries
        print(f"         - {so}")
    if len(so_files) == 0:
        print("  [WARN] No .so files found in lib directory!")
else:
    print(f"  [FAIL] Library directory missing: {LIB_DIR}")
    all_passed = False

# 4. Check Executable Permissions
print("\n--- 3. Checking Executable Permissions ---")
binaries_to_check = [
    os.path.join(BIN_DIR, 'palace'),
    os.path.join(BIN_DIR, 'palace-x86_64.bin'),
    os.path.join(BUILD_DIR, 'use_palace.sh')
]

for b in binaries_to_check:
    if os.path.exists(b):
        os.chmod(b, 0o755)
        print(f"  [PASS] Verified executable permissions for {os.path.basename(b)}")

# # 5. Execute Test Run (-dry-run)
# print("\n--- 4. Performing Execution Test ---")
# env = os.environ.copy()

# current_ld = env.get('LD_LIBRARY_PATH', '')
# env['LD_LIBRARY_PATH'] = f"{LIB_DIR}:{current_ld}" if current_ld else LIB_DIR
# env['OMPI_ALLOW_RUN_AS_ROOT'] = '1'
# env['OMPI_ALLOW_RUN_AS_ROOT_CONFIRM'] = '1'

# config_file_path = os.path.join(DRIVE_PALACE_DIR, 'dummy_config.json')

# if not os.path.exists(config_file_path):
#     dummy_config = {
#         "Problem": {"Type": "Electrostatic", "Verbose": 1},
#         "Model": {"Mesh": "mesh.msh"},
#         "Domains": {"Postprocessing": {"Energy": []}},
#         "Boundaries": {"Ground": [1]},
#         "Solver": {"Linear": {"Type": "AMS", "KSPType": "CG", "Tol": 1e-6, "MaxIter": 100}}
#     }
#     with open(config_file_path, 'w') as f:
#         json.dump(dummy_config, f, indent=2)

# try:
#     cmd = [os.path.join(BIN_DIR, 'palace'), '-dry-run', config_file_path]
#     result = subprocess.run(cmd, capture_output=True, text=True, check=True, env=env)
#     print("  [PASS] Palace -dry-run executed successfully!")
#     print("\nCommand Output:\n" + result.stdout)
# except subprocess.CalledProcessError as e:
#     print(f"  [FAIL] Palace -dry-run failed with exit code {e.returncode}")
#     print(f"Stderr:\n{e.stderr}")
#     all_passed = False
# except Exception as e:
#     print(f"  [FAIL] Execution error: {e}")
#     all_passed = False

# Final Summary
print("\n=========================================")
if all_passed:
    print("  SUCCESS: Palace installation is 100% complete and functional!")
else:
    print("  WARNING: One or more checks failed. Review output above.")
print("=========================================")

  PALACE INSTALLATION VERIFICATION CHECK 

✓ Google Drive is mounted.

--- 1. Checking Critical Files ---
  [PASS] Found Palace Launcher Wrapper: palace
  [PASS] Found Palace C++ Solver Engine: palace-x86_64.bin
  [PASS] Found Config Validator Tool: validate-config
  [PASS] Found Dependency LDD Log: palace_ldd.txt
  [PASS] Found Version/Help Output Log: palace_version.txt
  [PASS] Found Git Commit Metadata: palace_commit.txt
  [PASS] Found Git Status Log: palace_git_status.txt
  [PASS] Found CMake Cache: CMakeCache.txt
  [PASS] Found CMake Build Log: cmake_log.txt
  [PASS] Found Make Build Log: make_log.txt
  [PASS] Found Build Directory Tree Log: build_tree.txt
  [PASS] Found Restore Launcher Script: use_palace.sh

--- 2. Checking Shared Libraries ---
  [PASS] Found 7 shared library files in /content/drive/MyDrive/Palace6/build/lib
         - libxsmm.so.1.17.0
         - libceed.so
         - libxsmmgen.so.1.17.0
         - libxsmm.so.1
         - libxsmm.so

--- 3. Checking Executabl

In [ ]:
import os
import subprocess
import json

DRIVE_PALACE_DIR = '/content/drive/MyDrive/Palace6'
BIN_DIR = os.path.join(DRIVE_PALACE_DIR, 'build', 'bin')
LIB_DIR = os.path.join(DRIVE_PALACE_DIR, 'build', 'lib')

# Environment Setup
env = os.environ.copy()
current_ld = env.get('LD_LIBRARY_PATH', '')
env['LD_LIBRARY_PATH'] = f"{LIB_DIR}:{current_ld}" if current_ld else LIB_DIR
env['OMPI_ALLOW_RUN_AS_ROOT'] = '1'
env['OMPI_ALLOW_RUN_AS_ROOT_CONFIRM'] = '1'

# 1. Ensure minimal valid Gmsh v2.2 mesh exists in DRIVE_PALACE_DIR
mesh_file_path = os.path.join(DRIVE_PALACE_DIR, 'mesh.msh')
minimal_gmsh = """$MeshFormat
2.2 0 8
$EndMeshFormat
$Nodes
2
1 0.0 0.0 0.0
2 1.0 0.0 0.0
$EndNodes
$Elements
1
1 1 2 1 1 1 2
$EndElements
"""
with open(mesh_file_path, 'w') as f:
    f.write(minimal_gmsh)

# 2. Update config to use absolute path to the mesh
config_file_path = os.path.join(DRIVE_PALACE_DIR, 'dummy_config.json')

valid_config = {
    "Problem": {
        "Type": "Electrostatic",
        "Verbose": 1
    },
    "Model": {
        "Mesh": mesh_file_path,  # Full absolute path to mesh
        "L0": 1.0e-3
    },
    "Domains": {
        "Materials": [
            {
                "Attributes": [1],
                "Permittivity": 1.0
            }
        ],
        "Postprocessing": {
            "Energy": [
                {
                    "Index": 1,
                    "Attributes": [1]
                }
            ]
        }
    },
    "Boundaries": {
        "Ground": {
            "Attributes": [1]
        },
        "Terminal": [
            {
                "Index": 1,
                "Attributes": [2]
            }
        ]
    },
    "Solver": {
        "Order": 1,
        "Device": "CPU",
        "Linear": {
            "Type": "AMS",
            "KSPType": "CG",
            "Tol": 1.0e-6,
            "MaxIts": 100
        }
    }
}

with open(config_file_path, 'w') as f:
    json.dump(valid_config, f, indent=2)

print("Running Palace -dry-run with working directory set...\n")

try:
    cmd = [os.path.join(BIN_DIR, 'palace'), '-dry-run', config_file_path]
    # Set cwd so relative paths resolve cleanly inside DRIVE_PALACE_DIR
    result = subprocess.run(cmd, capture_output=True, text=True, check=True, env=env, cwd=DRIVE_PALACE_DIR)

    print("=========================================")
    print("✅ SUCCESS: Palace -dry-run completed successfully!")
    print("=========================================\n")
    print("Stdout:\n", result.stdout)

except subprocess.CalledProcessError as e:
    print("=========================================")
    print(f"Stdout:\n{e.stdout}")
    print(f"Stderr:\n{e.stderr}")

Running Palace -dry-run with working directory set...

✅ SUCCESS: Palace -dry-run completed successfully!

Stdout:
 >> /usr/bin/mpirun -n 1 /content/drive/MyDrive/Palace6/build/bin/palace-x86_64.bin --dry-run /content/drive/MyDrive/Palace6/dummy_config.json


Resolved configuration:
{
  "Boundaries": {
    "Ground": {
      "Attributes": [
        1
      ]
    },
    "Terminal": [
      {
        "Attributes": [
          2
        ],
        "Index": 1
      }
    ]
  },
  "Domains": {
    "Materials": [
      {
        "Attributes": [
          1
        ],
        "Conductivity": 0.0,
        "LondonDepth": 0.0,
        "LossTan": 0.0,
        "Permeability": 1.0,
        "Permittivity": 1.0
      }
    ],
    "Postprocessing": {
      "Energy": [
        {
          "Attributes": [
            1
          ],
          "Index": 1
        }
      ]
    }
  },
  "Model": {
    "AddInterfaceBoundaryElements": true,
    "CleanUnusedElements": true,
    "CrackDisplacementFactor": 1e-12,

In [ ]:
import gdspy
import os

GDS_PATH = "quantum_chip_export_manuscript.gds"

lib = gdspy.GdsLibrary()
lib.read_gds(GDS_PATH)

print("=" * 70)
print("GDS CELL INFORMATION")
print("=" * 70)

for name, cell in lib.cells.items():

    bbox = cell.get_bounding_box()

    print(f"\nCell: {name}")

    if bbox is not None:
        print(
            f"  Bounding box: "
            f"x={bbox[0][0]:.6f} to {bbox[1][0]:.6f}, "
            f"y={bbox[0][1]:.6f} to {bbox[1][1]:.6f}"
        )


print("\n" + "=" * 70)
print("LAYER / DATATYPE BOUNDING BOXES")
print("=" * 70)

for name, cell in lib.cells.items():

    polygons_by_spec = cell.get_polygons(
        by_spec=True
    )

    for spec, polygons in polygons_by_spec.items():

        layer, datatype = spec

        if not polygons:
            continue

        xmin = float("inf")
        xmax = float("-inf")
        ymin = float("inf")
        ymax = float("-inf")

        for poly in polygons:

            xmin = min(
                xmin,
                poly[:, 0].min()
            )

            xmax = max(
                xmax,
                poly[:, 0].max()
            )

            ymin = min(
                ymin,
                poly[:, 1].min()
            )

            ymax = max(
                ymax,
                poly[:, 1].max()
            )

        print(
            f"Layer={layer:3d}, "
            f"Datatype={datatype:3d}, "
            f"Polygons={len(polygons):7d}, "
            f"BBox=("
            f"{xmin:.3f}, {ymin:.3f}) -> "
            f"({xmax:.3f}, {ymax:.3f})"
        )

GDS CELL INFORMATION

Cell: TOP
  Bounding box: x=-7.000000 to 7.000000, y=-7.000000 to 7.000000

Cell: TOP_main
  Bounding box: x=-7.000000 to 7.000000, y=-7.000000 to 7.000000

Cell: TOP_main_1
  Bounding box: x=-7.000000 to 7.000000, y=-7.000000 to 7.000000

Cell: ground_main_1
  Bounding box: x=-7.000000 to 7.000000, y=-7.000000 to 7.000000

Cell: my_other_junction
  Bounding box: x=-0.015000 to 0.015000, y=-0.001500 to 0.001500

Cell: FakeJunction_02
  Bounding box: x=-0.015000 to 0.015000, y=-0.001500 to 0.001500

Cell: FakeJunction_01
  Bounding box: x=-0.015000 to 0.015000, y=-0.001503 to 0.001500

Cell: pads_my_other_junction_QComponent_is_1_name_is_poly4
  Bounding box: x=-0.017500 to 0.017500, y=-0.002500 to 0.002500

Cell: pads_my_other_junction_QComponent_is_1_name_is_poly5
  Bounding box: x=-0.017500 to 0.017500, y=-0.002500 to 0.002500

Cell: pads_my_other_junction_QComponent_is_2_name_is_poly4
  Bounding box: x=-0.017500 to 0.017500, y=-0.002500 to 0.002500

Cell: pads_

In [ ]:
import gdspy
import numpy as np

GDS_PATH =  "quantum_chip_export_manuscript.gds"

lib = gdspy.GdsLibrary()
lib.read_gds(GDS_PATH)

print("=" * 80)
print("JUNCTION-RELATED CELL GEOMETRY")
print("=" * 80)

jj_keywords = [
    "junction",
    "Junction",
    "FakeJunction",
    "my_other_junction",
    "pads_my_other_junction"
]

for cell_name, cell in lib.cells.items():

    if not any(
        key in cell_name
        for key in jj_keywords
    ):
        continue

    print("\n" + "-" * 80)
    print("CELL:", cell_name)

    polygons_by_spec = cell.get_polygons(
        by_spec=True
    )

    for spec, polygons in polygons_by_spec.items():

        layer, datatype = spec

        print(
            f"  Layer={layer}, "
            f"Datatype={datatype}, "
            f"Polygons={len(polygons)}"
        )

        for i, poly in enumerate(polygons):

            xmin = np.min(poly[:, 0])
            xmax = np.max(poly[:, 0])

            ymin = np.min(poly[:, 1])
            ymax = np.max(poly[:, 1])

            print(
                f"      polygon {i}: "
                f"bbox=({xmin:.6f}, {ymin:.6f}) "
                f"-> "
                f"({xmax:.6f}, {ymax:.6f}) "
                f"size=({xmax-xmin:.6f}, "
                f"{ymax-ymin:.6f})"
            )

JUNCTION-RELATED CELL GEOMETRY

--------------------------------------------------------------------------------
CELL: my_other_junction
  Layer=53, Datatype=0, Polygons=1
      polygon 0: bbox=(-0.015000, -0.001500) -> (0.015000, 0.001500) size=(0.030000, 0.003000)

--------------------------------------------------------------------------------
CELL: FakeJunction_02
  Layer=53, Datatype=0, Polygons=6
      polygon 0: bbox=(0.002000, -0.001500) -> (0.015000, 0.001500) size=(0.013000, 0.003000)
      polygon 1: bbox=(-0.015000, -0.001500) -> (-0.002000, 0.001500) size=(0.013000, 0.003000)
      polygon 2: bbox=(-0.002012, 0.000512) -> (-0.000035, 0.001489) size=(0.001977, 0.000977)
      polygon 3: bbox=(-0.000012, -0.001500) -> (0.002024, -0.000500) size=(0.002036, 0.001000)
      polygon 4: bbox=(-0.000989, -0.000512) -> (-0.000512, 0.000547) size=(0.000477, 0.001059)
      polygon 5: bbox=(-0.000977, -0.001198) -> (0.000047, -0.000849) size=(0.001024, 0.000349)

--------------------

In [ ]:
import gdspy

GDS_PATH = "quantum_chip_export_manuscript.gds"


lib = gdspy.GdsLibrary()
lib.read_gds(GDS_PATH)

print("=" * 80)
print("JUNCTION CELL REFERENCES")
print("=" * 80)

junction_names = {
    "my_other_junction",
    "FakeJunction_01",
    "FakeJunction_02",
}

for cell_name, cell in lib.cells.items():

    for ref in cell.references:

        ref_name = getattr(ref.ref_cell, "name", None)

        if ref_name in junction_names:

            print(
                f"Parent cell : {cell_name}"
            )

            print(
                f"Referenced  : {ref_name}"
            )

            print(
                f"Origin      : {ref.origin}"
            )

            print(
                f"Rotation    : {ref.rotation}"
            )

            print(
                f"Magnification: {ref.magnification}"
            )

            print("-" * 60)

JUNCTION CELL REFERENCES
Parent cell : TOP_main_1
Referenced  : my_other_junction
Origin      : [-1.1325  0.    ]
Rotation    : 180.0
Magnification: None
------------------------------------------------------------
Parent cell : TOP_main_1
Referenced  : my_other_junction
Origin      : [-0.8675  0.    ]
Rotation    : None
Magnification: None
------------------------------------------------------------
Parent cell : TOP_main_1
Referenced  : my_other_junction
Origin      : [1.2675 0.    ]
Rotation    : 180.0
Magnification: None
------------------------------------------------------------
Parent cell : TOP_main_1
Referenced  : my_other_junction
Origin      : [1.5325 0.    ]
Rotation    : None
Magnification: None
------------------------------------------------------------
Parent cell : TOP_main_1
Referenced  : my_other_junction
Origin      : [-0.8675 -2.7   ]
Rotation    : None
Magnification: None
------------------------------------------------------------
Parent cell : TOP_main_1
Referen

In [ ]:
# ============================================================
# STEP 5A
# SAFE FIRST-PASS 3D MESH FOR PALACE
#
# 4 concentric transmons
# 8 Josephson junction locations
#
# IMPORTANT:
# The JJs are NOT volumetrically meshed here.
# They will later be represented as lumped Josephson elements.
#
# This version is designed to avoid RAM exhaustion.
# ============================================================

import os
import gdspy
import gmsh


# ============================================================
# INPUT / OUTPUT
# ============================================================

GDS_PATH = "quantum_chip_export_manuscript.gds"
MESH_PATH = "quantum_chip_mesh_test.msh"

TOP_CELL_NAME = "TOP_main_1"

METAL_LAYER = 1
METAL_DATATYPE = 0

JJ_CELL_NAME = "my_other_junction"


# ============================================================
# CHIP GEOMETRY
# ============================================================

# GDS coordinates are in mm.

CHIP_X = 14.0
CHIP_Y = 14.0

# Silicon thickness
SUBSTRATE_THICKNESS = 0.5

# IMPORTANT:
# Start with 0.5 mm air, NOT 3 mm.
AIR_HEIGHT = 0.5


# ============================================================
# PHYSICAL GROUPS
# ============================================================

VACUUM_ID = 1
SILICON_ID = 2
PEC_ID = 10


# ============================================================
# SAFE MESH PARAMETERS
# ============================================================

# All dimensions are mm.

# 20 µm minimum
GLOBAL_MIN = 0.020

# 150 µm maximum
GLOBAL_MAX = 0.150


# ============================================================
# CHECK FILE
# ============================================================

if not os.path.exists(GDS_PATH):

    raise FileNotFoundError(
        f"GDS file not found:\n{GDS_PATH}"
    )


print("=" * 80)
print("SAFE FIRST-PASS PALACE MESH")
print("=" * 80)

print("\nGDS:")
print(os.path.abspath(GDS_PATH))


# ============================================================
# READ GDS
# ============================================================

print("\nReading GDS...")

lib = gdspy.GdsLibrary()

lib.read_gds(
    GDS_PATH
)

print(
    f"Loaded {len(lib.cells)} cells."
)


# ============================================================
# TOP CELL
# ============================================================

if TOP_CELL_NAME not in lib.cells:

    raise RuntimeError(
        f"Top cell not found: {TOP_CELL_NAME}"
    )

top = lib.cells[
    TOP_CELL_NAME
]

print(
    f"Using top cell: {TOP_CELL_NAME}"
)


# ============================================================
# EXTRACT ONLY ACTUAL METAL
# ============================================================

print(
    "\nExtracting Layer 1 / Datatype 0..."
)

polygons_by_spec = top.get_polygons(
    by_spec=True
)

metal_polygons = []

for spec, polygons in polygons_by_spec.items():

    layer, datatype = spec

    if layer != METAL_LAYER:
        continue

    if datatype != METAL_DATATYPE:
        continue

    for polygon in polygons:

        if len(polygon) >= 3:

            metal_polygons.append(
                polygon
            )


print(
    f"Metal polygons: "
    f"{len(metal_polygons)}"
)


# ============================================================
# FIND JJ REFERENCES
#
# We record them, but DO NOT mesh their 30 µm × 3 µm
# component polygon at sub-micron resolution.
# ============================================================

print(
    "\nFinding Josephson-junction references..."
)

jj_refs = []

for ref in top.references:

    try:

        name = ref.ref_cell.name

    except Exception:

        continue

    if name == JJ_CELL_NAME:

        jj_refs.append(
            ref
        )


print(
    f"Found {len(jj_refs)} JJ references."
)


if len(jj_refs) != 8:

    raise RuntimeError(
        f"Expected 8 JJ references, "
        f"found {len(jj_refs)}."
    )


# ============================================================
# SORT JJs
# ============================================================

def ref_position(ref):

    return (
        float(ref.origin[0]),
        float(ref.origin[1])
    )


jj_refs.sort(
    key=lambda r: (
        -ref_position(r)[1],
        ref_position(r)[0]
    )
)


print("\nEight JJ locations:")

for i, ref in enumerate(jj_refs):

    x, y = ref_position(ref)

    print(
        f"  JJ{i+1}: "
        f"x={x:.6f} mm, "
        f"y={y:.6f} mm, "
        f"rotation={ref.rotation}"
    )


# ============================================================
# INITIALIZE GMSH
# ============================================================

print("\nInitializing Gmsh...")

gmsh.initialize()

gmsh.option.setNumber(
    "General.Terminal",
    1
)

gmsh.model.add(
    "four_qubit_concentric_safe"
)


try:

    # ========================================================
    # SILICON
    # ========================================================

    print(
        "\nCreating silicon substrate..."
    )

    silicon = gmsh.model.occ.addBox(

        -CHIP_X / 2,
        -CHIP_Y / 2,
        -SUBSTRATE_THICKNESS,

        CHIP_X,
        CHIP_Y,
        SUBSTRATE_THICKNESS
    )


    # ========================================================
    # AIR
    # ========================================================

    print(
        "Creating vacuum region..."
    )

    vacuum = gmsh.model.occ.addBox(

        -CHIP_X / 2,
        -CHIP_Y / 2,
        0.0,

        CHIP_X,
        CHIP_Y,
        AIR_HEIGHT
    )


    # ========================================================
    # CREATE METAL SURFACES
    # ========================================================

    print(
        "\nCreating metal surfaces..."
    )

    metal_surfaces = []

    for i, polygon in enumerate(
        metal_polygons
    ):

        point_tags = []

        for x, y in polygon:

            p = gmsh.model.occ.addPoint(
                float(x),
                float(y),
                0.0
            )

            point_tags.append(p)


        lines = []

        for j in range(
            len(point_tags)
        ):

            p1 = point_tags[j]

            p2 = point_tags[
                (j + 1) % len(point_tags)
            ]

            line = gmsh.model.occ.addLine(
                p1,
                p2
            )

            lines.append(line)


        try:

            wire = gmsh.model.occ.addWire(
                lines
            )

            surface = (
                gmsh.model.occ.addPlaneSurface(
                    [wire]
                )
            )

            metal_surfaces.append(
                surface
            )

        except Exception as exc:

            print(
                f"Warning: polygon {i} failed: "
                f"{exc}"
            )


    print(
        f"Created {len(metal_surfaces)} "
        f"metal surfaces."
    )


    # ========================================================
    # SYNCHRONIZE
    # ========================================================

    print(
        "\nSynchronizing geometry..."
    )

    gmsh.model.occ.synchronize()


    # ========================================================
    # PHYSICAL VOLUMES
    # ========================================================

    print(
        "\nCreating physical volumes..."
    )

    gmsh.model.addPhysicalGroup(
        3,
        [vacuum],
        VACUUM_ID
    )

    gmsh.model.setPhysicalName(
        3,
        VACUUM_ID,
        "Vacuum"
    )


    gmsh.model.addPhysicalGroup(
        3,
        [silicon],
        SILICON_ID
    )

    gmsh.model.setPhysicalName(
        3,
        SILICON_ID,
        "Silicon_Substrate"
    )


    # ========================================================
    # PEC
    # ========================================================

    print(
        "Creating PEC group..."
    )

    gmsh.model.addPhysicalGroup(
        2,
        metal_surfaces,
        PEC_ID
    )

    gmsh.model.setPhysicalName(
        2,
        PEC_ID,
        "Superconducting_Metal"
    )


    # ========================================================
    # MESH SETTINGS
    # ========================================================

    print(
        "\nSetting conservative mesh size..."
    )

    gmsh.option.setNumber(
        "Mesh.Algorithm3D",
        1
    )

    gmsh.option.setNumber(
        "Mesh.CharacteristicLengthMin",
        GLOBAL_MIN
    )

    gmsh.option.setNumber(
        "Mesh.CharacteristicLengthMax",
        GLOBAL_MAX
    )

    # Avoid aggressive optimization
    gmsh.option.setNumber(
        "Mesh.Optimize",
        0
    )

    gmsh.option.setNumber(
        "Mesh.OptimizeNetgen",
        0
    )


    # ========================================================
    # GENERATE 3D MESH
    # ========================================================

    print(
        "\n" + "=" * 80
    )

    print(
        "GENERATING CONSERVATIVE 3D MESH"
    )

    print(
        "Minimum element size = 20 um"
    )

    print(
        "Maximum element size = 150 um"
    )

    print(
        "Air height = 0.5 mm"
    )

    print(
        "=" * 80
    )

    gmsh.model.mesh.generate(
        3
    )


    # ========================================================
    # STATISTICS
    # ========================================================

    print(
        "\n" + "=" * 80
    )

    print(
        "MESH STATISTICS"
    )

    print(
        "=" * 80
    )

    nodes, coords, _ = (
        gmsh.model.mesh.getNodes()
    )

    print(
        f"Nodes: {len(nodes):,}"
    )


    element_types, element_tags, _ = (
        gmsh.model.mesh.getElements()
    )

    total_elements = sum(
        len(tags)
        for tags in element_tags
    )

    print(
        f"Elements: {total_elements:,}"
    )


    # ========================================================
    # PHYSICAL GROUPS
    # ========================================================

    print(
        "\nPhysical groups:"
    )

    for dim, tag in (
        gmsh.model.getPhysicalGroups()
    ):

        name = gmsh.model.getPhysicalName(
            dim,
            tag
        )

        print(
            f"  Dimension={dim}, "
            f"Attribute={tag}, "
            f"Name={name}"
        )


    # ========================================================
    # WRITE MESH
    # ========================================================

    print(
        "\nWriting mesh..."
    )

    gmsh.write(
        MESH_PATH
    )


    print(
        "\n" + "=" * 80
    )

    print(
        "SUCCESS"
    )

    print(
        "=" * 80
    )

    print(
        f"\nMesh:"
        f"\n{os.path.abspath(MESH_PATH)}"
    )


finally:

    gmsh.finalize()

    print(
        "\nGmsh finalized."
    )

SAFE FIRST-PASS PALACE MESH

GDS:
/content/quantum_chip_export_manuscript.gds

Reading GDS...
Loaded 19 cells.
Using top cell: TOP_main_1

Extracting Layer 1 / Datatype 0...
Metal polygons: 2450

Finding Josephson-junction references...
Found 8 JJ references.

Eight JJ locations:
  JJ1: x=-1.132500 mm, y=0.000000 mm, rotation=180.0
  JJ2: x=-0.867500 mm, y=0.000000 mm, rotation=None
  JJ3: x=1.267500 mm, y=0.000000 mm, rotation=180.0
  JJ4: x=1.532500 mm, y=0.000000 mm, rotation=None
  JJ5: x=-1.132500 mm, y=-2.700000 mm, rotation=180.0
  JJ6: x=-0.867500 mm, y=-2.700000 mm, rotation=None
  JJ7: x=1.267500 mm, y=-2.700000 mm, rotation=180.0
  JJ8: x=1.532500 mm, y=-2.700000 mm, rotation=None

Initializing Gmsh...

Creating silicon substrate...
Creating vacuum region...

Creating metal surfaces...
Created 2450 metal surfaces.

Synchronizing geometry...

Creating physical volumes...
Creating PEC group...

Setting conservative mesh size...

GENERATING CONSERVATIVE 3D MESH
Minimum element 

In [ ]:
import os
import stat
import shutil

validator = "/content/drive/MyDrive/Palace6/build/bin/validate-config"

print("Validator exists:", os.path.exists(validator))
print("Current permissions:", oct(os.stat(validator).st_mode))

# Add executable permission
os.chmod(
    validator,
    os.stat(validator).st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH
)

print("New permissions:", oct(os.stat(validator).st_mode))

print("Executable:", os.access(validator, os.X_OK))

Validator exists: True
Current permissions: 0o100600
New permissions: 0o100700
Executable: True


In [ ]:
# ============================================================
# STEP 5B
# PALACE-COMPATIBLE GMSH MESH
#
# Fixes:
#   - duplicate OCC geometry
#   - duplicate mesh nodes
#   - degenerate elements
#   - repeated vertex indices
#
# Input:
#   quantum_chip_export_manuscript.gds
#
# Output:
#   quantum_chip_mesh_palace.msh
# ============================================================

import os
import gdspy
import gmsh


# ============================================================
# INPUT
# ============================================================

GDS_PATH = "quantum_chip_export_manuscript.gds"

MESH_PATH = "/content/quantum_chip_mesh_palace.msh"

TOP_CELL_NAME = "TOP_main_1"

METAL_LAYER = 1
METAL_DATATYPE = 0

JJ_CELL_NAME = "my_other_junction"


# ============================================================
# CHIP DIMENSIONS
# ============================================================

CHIP_X = 14.0
CHIP_Y = 14.0

SUBSTRATE_THICKNESS = 0.5
AIR_HEIGHT = 0.5


# ============================================================
# PHYSICAL GROUPS
# ============================================================

VACUUM_ID = 1
SILICON_ID = 2
PEC_ID = 10


# ============================================================
# SAFE MESH SIZE
# ============================================================

GLOBAL_MIN = 0.020     # 20 um
GLOBAL_MAX = 0.150     # 150 um


# ============================================================
# READ GDS
# ============================================================

print("=" * 80)
print("PALACE-COMPATIBLE MESH GENERATOR")
print("=" * 80)

print("\nReading GDS:")
print(GDS_PATH)

lib = gdspy.GdsLibrary()
lib.read_gds(GDS_PATH)

print(
    f"Loaded {len(lib.cells)} cells."
)


# ============================================================
# TOP CELL
# ============================================================

if TOP_CELL_NAME not in lib.cells:

    raise RuntimeError(
        f"Top cell '{TOP_CELL_NAME}' not found."
    )

top = lib.cells[TOP_CELL_NAME]

print(
    f"\nUsing top cell: {TOP_CELL_NAME}"
)


# ============================================================
# EXTRACT METAL
# ============================================================

print(
    "\nExtracting actual metal geometry..."
)

polygons_by_spec = top.get_polygons(
    by_spec=True
)

metal_polygons = []

for spec, polygons in polygons_by_spec.items():

    layer, datatype = spec

    if layer != METAL_LAYER:
        continue

    if datatype != METAL_DATATYPE:
        continue

    for polygon in polygons:

        if len(polygon) < 3:
            continue

        # ----------------------------------------------------
        # Remove repeated consecutive points
        # ----------------------------------------------------

        cleaned = []

        for p in polygon:

            x = float(p[0])
            y = float(p[1])

            if not cleaned:

                cleaned.append(
                    (x, y)
                )

            else:

                px, py = cleaned[-1]

                if (
                    abs(x - px) > 1e-12
                    or
                    abs(y - py) > 1e-12
                ):

                    cleaned.append(
                        (x, y)
                    )

        # ----------------------------------------------------
        # Remove closing duplicate
        # ----------------------------------------------------

        if len(cleaned) >= 2:

            if (
                abs(cleaned[0][0] - cleaned[-1][0])
                < 1e-12
                and
                abs(cleaned[0][1] - cleaned[-1][1])
                < 1e-12
            ):

                cleaned.pop()

        # ----------------------------------------------------
        # Need at least 3 unique points
        # ----------------------------------------------------

        if len(cleaned) < 3:
            continue

        if len(set(cleaned)) < 3:
            continue

        metal_polygons.append(
            cleaned
        )


print(
    f"Clean metal polygons: "
    f"{len(metal_polygons)}"
)


# ============================================================
# FIND JJ REFERENCES
# ============================================================

print(
    "\nFinding JJ references..."
)

jj_refs = []

for ref in top.references:

    try:
        name = ref.ref_cell.name
    except Exception:
        continue

    if name == JJ_CELL_NAME:

        jj_refs.append(ref)


print(
    f"Found {len(jj_refs)} JJ references."
)


if len(jj_refs) != 8:

    raise RuntimeError(
        f"Expected 8 JJs, found {len(jj_refs)}."
    )


# ============================================================
# PRINT JJ LOCATIONS
# ============================================================

jj_refs.sort(
    key=lambda r: (
        -float(r.origin[1]),
        float(r.origin[0])
    )
)

print("\nJJ locations:")

for i, ref in enumerate(jj_refs):

    print(
        f"  JJ{i+1}: "
        f"x={float(ref.origin[0]):.6f} mm, "
        f"y={float(ref.origin[1]):.6f} mm, "
        f"rotation={ref.rotation}"
    )


# ============================================================
# INITIALIZE GMSH
# ============================================================

print(
    "\nInitializing Gmsh..."
)

gmsh.initialize()

gmsh.option.setNumber(
    "General.Terminal",
    1
)

gmsh.model.add(
    "four_qubit_palace"
)


try:

    # ========================================================
    # SILICON
    # ========================================================

    print(
        "\nCreating silicon substrate..."
    )

    silicon = gmsh.model.occ.addBox(

        -CHIP_X / 2.0,
        -CHIP_Y / 2.0,
        -SUBSTRATE_THICKNESS,

        CHIP_X,
        CHIP_Y,
        SUBSTRATE_THICKNESS
    )


    # ========================================================
    # VACUUM
    # ========================================================

    print(
        "Creating vacuum..."
    )

    vacuum = gmsh.model.occ.addBox(

        -CHIP_X / 2.0,
        -CHIP_Y / 2.0,
        0.0,

        CHIP_X,
        CHIP_Y,
        AIR_HEIGHT
    )


    # ========================================================
    # GLOBAL POINT CACHE
    #
    # IMPORTANT:
    # Reuse the SAME OCC point for identical coordinates.
    # This prevents many duplicate vertices.
    # ========================================================

    point_cache = {}

    def get_point(x, y):

        # Quantize coordinates to avoid floating-point
        # differences creating duplicate points.

        key = (
            round(float(x), 10),
            round(float(y), 10),
            0.0
        )

        if key not in point_cache:

            point_cache[key] = (
                gmsh.model.occ.addPoint(
                    key[0],
                    key[1],
                    key[2]
                )
            )

        return point_cache[key]


    # ========================================================
    # CREATE METAL SURFACES
    # ========================================================

    print(
        "\nCreating metal surfaces..."
    )

    metal_surfaces = []

    failed_polygons = 0

    for i, polygon in enumerate(
        metal_polygons
    ):

        point_tags = []

        for x, y in polygon:

            point_tags.append(
                get_point(x, y)
            )


        # ----------------------------------------------
        # Remove repeated point IDs
        # ----------------------------------------------

        unique_points = []

        for tag in point_tags:

            if (
                not unique_points
                or
                tag != unique_points[-1]
            ):

                unique_points.append(tag)


        if len(unique_points) >= 2:

            if (
                unique_points[0]
                ==
                unique_points[-1]
            ):

                unique_points.pop()


        if len(set(unique_points)) < 3:

            failed_polygons += 1
            continue


        # ----------------------------------------------
        # Create closed wire
        # ----------------------------------------------

        lines = []

        valid = True

        for j in range(
            len(unique_points)
        ):

            p1 = unique_points[j]

            p2 = unique_points[
                (j + 1) % len(unique_points)
            ]

            if p1 == p2:

                valid = False
                break

            try:

                line = (
                    gmsh.model.occ.addLine(
                        p1,
                        p2
                    )
                )

                lines.append(line)

            except Exception:

                valid = False
                break


        if not valid or len(lines) < 3:

            failed_polygons += 1
            continue


        try:

            wire = (
                gmsh.model.occ.addWire(
                    lines
                )
            )

            surface = (
                gmsh.model.occ.addPlaneSurface(
                    [wire]
                )
            )

            metal_surfaces.append(
                surface
            )

        except Exception:

            failed_polygons += 1


    print(
        f"Created {len(metal_surfaces)} "
        f"metal surfaces."
    )

    print(
        f"Skipped {failed_polygons} "
        f"invalid polygons."
    )


    # ========================================================
    # REMOVE DUPLICATE OCC GEOMETRY
    # ========================================================

    print(
        "\nRemoving duplicate OCC geometry..."
    )

    try:

        gmsh.model.occ.removeAllDuplicates()

        print(
            "Duplicate OCC geometry removed."
        )

    except Exception as exc:

        print(
            "removeAllDuplicates warning:",
            exc
        )


    # ========================================================
    # SYNCHRONIZE
    # ========================================================

    print(
        "\nSynchronizing OCC..."
    )

    gmsh.model.occ.synchronize()


    # ========================================================
    # PHYSICAL GROUPS
    # ========================================================

    print(
        "\nCreating physical groups..."
    )


    # --------------------------------------------------------
    # VACUUM
    # --------------------------------------------------------

    gmsh.model.addPhysicalGroup(
        3,
        [vacuum],
        VACUUM_ID
    )

    gmsh.model.setPhysicalName(
        3,
        VACUUM_ID,
        "Vacuum"
    )


    # --------------------------------------------------------
    # SILICON
    # --------------------------------------------------------

    gmsh.model.addPhysicalGroup(
        3,
        [silicon],
        SILICON_ID
    )

    gmsh.model.setPhysicalName(
        3,
        SILICON_ID,
        "Silicon_Substrate"
    )


    # --------------------------------------------------------
    # PEC
    # --------------------------------------------------------

    gmsh.model.addPhysicalGroup(
        2,
        metal_surfaces,
        PEC_ID
    )

    gmsh.model.setPhysicalName(
        2,
        PEC_ID,
        "Superconducting_Metal"
    )


    # ========================================================
    # MESH PARAMETERS
    # ========================================================

    print(
        "\nSetting mesh parameters..."
    )

    gmsh.option.setNumber(
        "Mesh.Algorithm3D",
        1
    )

    gmsh.option.setNumber(
        "Mesh.CharacteristicLengthMin",
        GLOBAL_MIN
    )

    gmsh.option.setNumber(
        "Mesh.CharacteristicLengthMax",
        GLOBAL_MAX
    )

    gmsh.option.setNumber(
        "Mesh.Optimize",
        0
    )

    gmsh.option.setNumber(
        "Mesh.OptimizeNetgen",
        0
    )


    # ========================================================
    # GENERATE MESH
    # ========================================================

    print(
        "\n" + "=" * 80
    )

    print(
        "GENERATING 3D MESH"
    )

    print(
        "=" * 80
    )

    gmsh.model.mesh.generate(3)


    # ========================================================
    # REMOVE DUPLICATE NODES
    # ========================================================

    print(
        "\nRemoving duplicate mesh nodes..."
    )

    try:

        gmsh.model.mesh.removeDuplicateNodes()

        print(
            "Duplicate mesh nodes removed."
        )

    except Exception as exc:

        print(
            "Duplicate-node cleanup warning:",
            exc
        )


    # ========================================================
    # REMOVE DUPLICATE ELEMENTS
    # ========================================================

    print(
        "Removing duplicate mesh elements..."
    )

    try:

        gmsh.model.mesh.removeDuplicateElements()

        print(
            "Duplicate mesh elements removed."
        )

    except Exception as exc:

        print(
            "Duplicate-element cleanup warning:",
            exc
        )


    # ========================================================
    # CHECK ELEMENT NODE UNIQUENESS
    # ========================================================

    print(
        "\nChecking element vertex uniqueness..."
    )

    bad_elements = []

    element_types, element_tags, element_nodes = (
        gmsh.model.mesh.getElements()
    )

    for etype, tags, nodes in zip(
        element_types,
        element_tags,
        element_nodes
    ):

        # Get number of nodes per element
        _, _, _, num_nodes, _, _ = (
            gmsh.model.mesh.getElementProperties(
                etype
            )
        )

        for i, element_tag in enumerate(tags):

            start = (
                i * num_nodes
            )

            end = (
                start + num_nodes
            )

            connectivity = nodes[
                start:end
            ]

            if len(set(connectivity)) != len(
                connectivity
            ):

                bad_elements.append(
                    (
                        int(element_tag),
                        int(etype),
                        connectivity.tolist()
                    )
                )

                if len(bad_elements) >= 20:

                    break

        if len(bad_elements) >= 20:

            break


    if bad_elements:

        print(
            "\nWARNING: "
            f"Found {len(bad_elements)} "
            "elements with repeated vertices."
        )

        for item in bad_elements[:10]:

            print(
                "  Element:",
                item
            )

        raise RuntimeError(
            "\nMesh contains degenerate elements. "
            "Do NOT send this mesh to Palace."
        )


    print(
        "PASS: all checked elements have "
        "unique vertex indices."
    )


    # ========================================================
    # NODE / ELEMENT STATISTICS
    # ========================================================

    nodes, coords, _ = (
        gmsh.model.mesh.getNodes()
    )

    total_elements = sum(
        len(tags)
        for tags in element_tags
    )


    print(
        "\n" + "=" * 80
    )

    print(
        "FINAL MESH STATISTICS"
    )

    print(
        "=" * 80
    )

    print(
        f"Nodes    : {len(nodes):,}"
    )

    print(
        f"Elements : {total_elements:,}"
    )


    # ========================================================
    # PHYSICAL GROUPS
    # ========================================================

    print(
        "\nPhysical groups:"
    )

    for dim, tag in (
        gmsh.model.getPhysicalGroups()
    ):

        name = gmsh.model.getPhysicalName(
            dim,
            tag
        )

        print(
            f"  Dimension={dim}, "
            f"Attribute={tag}, "
            f"Name={name}"
        )


    # ========================================================
    # WRITE MESH
    # ========================================================

    print(
        "\nWriting Palace mesh..."
    )

    gmsh.write(
        MESH_PATH
    )


    print(
        "\n" + "=" * 80
    )

    print(
        "PALACE MESH CREATED SUCCESSFULLY"
    )

    print(
        "=" * 80
    )

    print(
        f"\nMesh file:"
        f"\n{MESH_PATH}"
    )


finally:

    gmsh.finalize()

    print(
        "\nGmsh finalized."
    )

PALACE-COMPATIBLE MESH GENERATOR

Reading GDS:
quantum_chip_export_manuscript.gds
Loaded 19 cells.

Using top cell: TOP_main_1

Extracting actual metal geometry...
Clean metal polygons: 2450

Finding JJ references...
Found 8 JJ references.

JJ locations:
  JJ1: x=-1.132500 mm, y=0.000000 mm, rotation=180.0
  JJ2: x=-0.867500 mm, y=0.000000 mm, rotation=None
  JJ3: x=1.267500 mm, y=0.000000 mm, rotation=180.0
  JJ4: x=1.532500 mm, y=0.000000 mm, rotation=None
  JJ5: x=-1.132500 mm, y=-2.700000 mm, rotation=180.0
  JJ6: x=-0.867500 mm, y=-2.700000 mm, rotation=None
  JJ7: x=1.267500 mm, y=-2.700000 mm, rotation=180.0
  JJ8: x=1.532500 mm, y=-2.700000 mm, rotation=None

Initializing Gmsh...

Creating silicon substrate...
Creating vacuum...

Creating metal surfaces...
Created 2450 metal surfaces.
Skipped 0 invalid polygons.

Removing duplicate OCC geometry...
Duplicate OCC geometry removed.

Synchronizing OCC...

Creating physical groups...

Setting mesh parameters...

GENERATING 3D MESH



In [ ]:
# ============================================================
# STEP 5C
# CLEAN PALACE MESH AND WRITE GMSH 2.2 FORMAT
#
# Input:
#   quantum_chip_mesh_palace.msh
#
# Output:
#   quantum_chip_mesh_palace_v22.msh
#
# Purpose:
#   1. Read the existing Gmsh mesh
#   2. Remove duplicate nodes
#   3. Remove duplicate elements
#   4. Remove degenerate elements
#   5. Write MSH 2.2
#
# MSH 2.2 is deliberately used for maximum MFEM compatibility.
# ============================================================

import os
import gmsh


INPUT_MESH = "/content/quantum_chip_mesh_palace.msh"

OUTPUT_MESH = "/content/quantum_chip_mesh_palace_v22.msh"


# ============================================================
# CHECK
# ============================================================

if not os.path.exists(INPUT_MESH):

    raise FileNotFoundError(
        f"Mesh not found:\n{INPUT_MESH}"
    )


print("=" * 80)
print("PALACE MESH CLEANUP")
print("=" * 80)

print(
    "\nInput:",
    INPUT_MESH
)

print(
    "Output:",
    OUTPUT_MESH
)


# ============================================================
# INITIALIZE
# ============================================================

gmsh.initialize()

gmsh.option.setNumber(
    "General.Terminal",
    1
)

try:

    # --------------------------------------------------------
    # Open existing mesh
    # --------------------------------------------------------

    print(
        "\nOpening existing Gmsh mesh..."
    )

    gmsh.open(
        INPUT_MESH
    )


    # --------------------------------------------------------
    # Initial statistics
    # --------------------------------------------------------

    node_tags, node_coords, _ = (
        gmsh.model.mesh.getNodes()
    )

    print(
        f"Initial nodes: "
        f"{len(node_tags):,}"
    )


    # --------------------------------------------------------
    # Remove duplicate nodes
    # --------------------------------------------------------

    print(
        "\nRemoving duplicate nodes..."
    )

    gmsh.model.mesh.removeDuplicateNodes()


    # --------------------------------------------------------
    # Remove duplicate elements
    # --------------------------------------------------------

    print(
        "Removing duplicate elements..."
    )

    gmsh.model.mesh.removeDuplicateElements()


    # --------------------------------------------------------
    # Renumber nodes
    # --------------------------------------------------------

    print(
        "Renumbering nodes..."
    )

    try:

        gmsh.model.mesh.renumberNodes()

    except Exception as exc:

        print(
            "Node renumbering warning:",
            exc
        )


    # ========================================================
    # CHECK ELEMENT CONNECTIVITY
    # ========================================================

    print(
        "\nChecking element connectivity..."
    )

    element_types, element_tags, element_nodes = (
        gmsh.model.mesh.getElements()
    )

    bad_elements = []

    total_elements = 0


    for etype, tags, nodes in zip(
        element_types,
        element_tags,
        element_nodes
    ):

        (
            element_name,
            dim,
            order,
            num_nodes,
            local_node_coords,
            num_primary_nodes
        ) = gmsh.model.mesh.getElementProperties(
            etype
        )


        print(
            f"  {element_name}: "
            f"{len(tags):,} elements, "
            f"{num_nodes} nodes/element"
        )


        total_elements += len(tags)


        # ----------------------------------------------------
        # Connectivity is flattened:
        #
        # [e1n1,e1n2,...,e1nn,
        #  e2n1,e2n2,...]
        # ----------------------------------------------------

        for i, element_tag in enumerate(tags):

            start = i * num_nodes

            end = start + num_nodes

            connectivity = nodes[
                start:end
            ]


            # ----------------------------------------------
            # Repeated node inside one element
            # ----------------------------------------------

            if (
                len(set(connectivity.tolist()))
                != len(connectivity)
            ):

                bad_elements.append(
                    {
                        "tag": int(element_tag),
                        "type": int(etype),
                        "name": element_name,
                        "nodes": connectivity.tolist()
                    }
                )


    print(
        f"\nTotal elements: "
        f"{total_elements:,}"
    )

    print(
        f"Degenerate elements found: "
        f"{len(bad_elements)}"
    )


    # ========================================================
    # ABORT IF BAD ELEMENTS REMAIN
    # ========================================================

    if bad_elements:

        print(
            "\nFirst bad elements:"
        )

        for bad in bad_elements[:20]:

            print(
                bad
            )


        raise RuntimeError(
            "\nThe mesh still contains "
            "elements with repeated vertex indices."
        )


    print(
        "\nPASS:"
    )

    print(
        "No element contains repeated vertex indices."
    )


    # ========================================================
    # FORCE GMSH 2.2
    # ========================================================

    print(
        "\nSetting Gmsh output format to 2.2..."
    )

    gmsh.option.setNumber(
        "Mesh.MshFileVersion",
        2.2
    )

    gmsh.option.setNumber(
        "Mesh.Binary",
        0
    )


    # ========================================================
    # WRITE
    # ========================================================

    print(
        "\nWriting cleaned MSH 2.2 file..."
    )

    gmsh.write(
        OUTPUT_MESH
    )


    print(
        "\n" + "=" * 80
    )

    print(
        "CLEAN MESH CREATED"
    )

    print(
        "=" * 80
    )

    print(
        "\nOutput:"
    )

    print(
        OUTPUT_MESH
    )

finally:

    gmsh.finalize()

    print(
        "\nGmsh finalized."
    )

PALACE MESH CLEANUP

Input: /content/quantum_chip_mesh_palace.msh
Output: /content/quantum_chip_mesh_palace_v22.msh

Opening existing Gmsh mesh...
Initial nodes: 121,755

Removing duplicate nodes...
Removing duplicate elements...
Renumbering nodes...

Checking element connectivity...
  Triangle 3: 84,163 elements, 3 nodes/element
  Tetrahedron 4: 314,269 elements, 4 nodes/element

Total elements: 398,432
Degenerate elements found: 0

PASS:
No element contains repeated vertex indices.

Setting Gmsh output format to 2.2...

Writing cleaned MSH 2.2 file...

CLEAN MESH CREATED

Output:
/content/quantum_chip_mesh_palace_v22.msh

Gmsh finalized.


In [ ]:
# ============================================================
# STEP 5D — ROBUST CONFORMAL PALACE MESH
#
# GDS metal polygons
#       ↓
# Shapely 2D union
#       ↓
# Non-overlapping metal regions
#       ↓
# Silicon + vacuum fragmentation
#       ↓
# Conformal 3D mesh
#       ↓
# MSH 2.2 for MFEM / Palace
#
# IMPORTANT:
# JJs are NOT included as volumetric geometry yet.
# ============================================================

import os
import gdspy
import gmsh

from shapely.geometry import Polygon
from shapely.ops import unary_union


# ============================================================
# INPUT
# ============================================================

GDS_PATH = "quantum_chip_export_manuscript.gds"

MESH_PATH = "/content/quantum_chip_mesh_conformal.msh"

TOP_CELL_NAME = "TOP_main_1"

METAL_LAYER = 1
METAL_DATATYPE = 0


# ============================================================
# CHIP GEOMETRY
# ============================================================

CHIP_X = 14.0
CHIP_Y = 14.0

SUBSTRATE_THICKNESS = 0.5
AIR_HEIGHT = 0.5


# ============================================================
# PHYSICAL ATTRIBUTES
# ============================================================

VACUUM_ID = 1
SILICON_ID = 2
PEC_ID = 10


# ============================================================
# MESH PARAMETERS
# ============================================================

GLOBAL_MIN = 0.020     # 20 um
GLOBAL_MAX = 0.150     # 150 um


# ============================================================
# READ GDS
# ============================================================

print("=" * 80)
print("ROBUST CONFORMAL PALACE MESH GENERATOR")
print("=" * 80)

print("\nReading GDS...")

lib = gdspy.GdsLibrary()

lib.read_gds(GDS_PATH)

print(
    f"Loaded {len(lib.cells)} cells."
)


# ============================================================
# TOP CELL
# ============================================================

if TOP_CELL_NAME not in lib.cells:

    raise RuntimeError(
        f"Top cell not found: {TOP_CELL_NAME}"
    )

top = lib.cells[TOP_CELL_NAME]

print(
    f"Using top cell: {TOP_CELL_NAME}"
)


# ============================================================
# EXTRACT METAL POLYGONS
# ============================================================

print(
    "\nExtracting Layer 1 / Datatype 0..."
)

polygons_by_spec = top.get_polygons(
    by_spec=True
)

raw_polygons = []

for spec, polygons in polygons_by_spec.items():

    layer, datatype = spec

    if layer != METAL_LAYER:
        continue

    if datatype != METAL_DATATYPE:
        continue

    for polygon in polygons:

        if len(polygon) < 3:
            continue

        points = []

        for p in polygon:

            x = float(p[0])
            y = float(p[1])

            point = (
                round(x, 10),
                round(y, 10)
            )

            if not points or point != points[-1]:

                points.append(point)


        # Remove closing duplicate
        if (
            len(points) > 1
            and points[0] == points[-1]
        ):

            points.pop()


        if len(set(points)) >= 3:

            raw_polygons.append(
                points
            )


print(
    f"Raw metal polygons: "
    f"{len(raw_polygons)}"
)


# ============================================================
# CONVERT TO SHAPELY
# ============================================================

print(
    "\nConverting polygons to Shapely..."
)

shapes = []

invalid_count = 0

for i, points in enumerate(
    raw_polygons
):

    try:

        poly = Polygon(points)

        # Repair small self-intersections
        if not poly.is_valid:

            poly = poly.buffer(0)


        if poly.is_empty:
            continue

        if poly.area <= 1e-12:
            continue

        shapes.append(poly)

    except Exception:

        invalid_count += 1


print(
    f"Valid Shapely polygons: "
    f"{len(shapes)}"
)

print(
    f"Invalid/skipped: "
    f"{invalid_count}"
)


# ============================================================
# UNION ALL METAL
# ============================================================

print(
    "\n" + "=" * 80
)

print(
    "UNITING OVERLAPPING METAL POLYGONS"
)

print(
    "=" * 80
)

print(
    "This may take some time..."
)

metal_union = unary_union(
    shapes
)


# ============================================================
# REPAIR UNION
# ============================================================

if not metal_union.is_valid:

    print(
        "\nUnion is invalid; repairing..."
    )

    metal_union = metal_union.buffer(0)


print(
    "\nMetal union type:",
    metal_union.geom_type
)

print(
    "Unified metal area:",
    metal_union.area,
    "mm^2"
)


# ============================================================
# EXTRACT POLYGONS
# ============================================================

if metal_union.geom_type == "Polygon":

    unified_polygons = [
        metal_union
    ]

elif metal_union.geom_type == "MultiPolygon":

    unified_polygons = list(
        metal_union.geoms
    )

else:

    raise RuntimeError(
        "Unexpected Shapely geometry type: "
        + metal_union.geom_type
    )


print(
    "Unified metal regions:",
    len(unified_polygons)
)


# ============================================================
# INITIALIZE GMSH
# ============================================================

print(
    "\nInitializing Gmsh..."
)

gmsh.initialize()

gmsh.option.setNumber(
    "General.Terminal",
    1
)

gmsh.model.add(
    "four_qubit_conformal"
)


try:

    # ========================================================
    # CREATE SILICON
    # ========================================================

    print(
        "\nCreating silicon..."
    )

    silicon = gmsh.model.occ.addBox(

        -CHIP_X / 2,
        -CHIP_Y / 2,
        -SUBSTRATE_THICKNESS,

        CHIP_X,
        CHIP_Y,
        SUBSTRATE_THICKNESS
    )


    # ========================================================
    # CREATE VACUUM
    # ========================================================

    print(
        "Creating vacuum..."
    )

    vacuum = gmsh.model.occ.addBox(

        -CHIP_X / 2,
        -CHIP_Y / 2,
        0.0,

        CHIP_X,
        CHIP_Y,
        AIR_HEIGHT
    )


    # ========================================================
    # POINT CACHE
    # ========================================================

    point_cache = {}


    def get_point(x, y):

        key = (
            round(float(x), 10),
            round(float(y), 10),
            0.0
        )

        if key not in point_cache:

            point_cache[key] = (
                gmsh.model.occ.addPoint(
                    key[0],
                    key[1],
                    key[2]
                )
            )

        return point_cache[key]


    # ========================================================
    # CREATE UNIFIED METAL SURFACES
    # ========================================================

    print(
        "\nCreating unified metal surfaces..."
    )

    metal_surfaces = []


    for index, poly in enumerate(
        unified_polygons
    ):

        # ----------------------------------------------------
        # OUTER RING
        # ----------------------------------------------------

        exterior = list(
            poly.exterior.coords
        )

        exterior = exterior[:-1]


        outer_points = []

        for x, y in exterior:

            outer_points.append(
                get_point(x, y)
            )


        outer_lines = []

        for i in range(
            len(outer_points)
        ):

            p1 = outer_points[i]

            p2 = outer_points[
                (i + 1) % len(outer_points)
            ]

            if p1 == p2:
                continue

            outer_lines.append(
                gmsh.model.occ.addLine(
                    p1,
                    p2
                )
            )


        outer_wire = (
            gmsh.model.occ.addWire(
                outer_lines
            )
        )


        # ----------------------------------------------------
        # HOLES
        # ----------------------------------------------------

        hole_wires = []


        for interior in poly.interiors:

            coords = list(
                interior.coords
            )

            coords = coords[:-1]


            hole_points = []

            for x, y in coords:

                hole_points.append(
                    get_point(x, y)
                )


            hole_lines = []

            for i in range(
                len(hole_points)
            ):

                p1 = hole_points[i]

                p2 = hole_points[
                    (i + 1) % len(hole_points)
                ]

                if p1 == p2:
                    continue

                hole_lines.append(
                    gmsh.model.occ.addLine(
                        p1,
                        p2
                    )
                )


            if len(hole_lines) >= 3:

                hole_wire = (
                    gmsh.model.occ.addWire(
                        hole_lines
                    )
                )

                hole_wires.append(
                    hole_wire
                )


        # ----------------------------------------------------
        # CREATE PLANE SURFACE
        # ----------------------------------------------------

        try:

            wire_list = [
                outer_wire
            ] + hole_wires

            surface = (
                gmsh.model.occ.addPlaneSurface(
                    wire_list
                )
            )

            metal_surfaces.append(
                surface
            )

        except Exception as exc:

            print(
                f"Warning: metal region "
                f"{index} failed:"
            )

            print(
                exc
            )


    print(
        "\nUnified metal surfaces created:",
        len(metal_surfaces)
    )


    # ========================================================
    # SYNCHRONIZE
    # ========================================================

    gmsh.model.occ.synchronize()


    # ========================================================
    # REMOVE DUPLICATES
    # ========================================================

    print(
        "\nRemoving duplicate OCC geometry..."
    )

    gmsh.model.occ.removeAllDuplicates()

    gmsh.model.occ.synchronize()


    # ========================================================
    # FRAGMENT SILICON + VACUUM USING METAL
    # ========================================================

    print(
        "\n" + "=" * 80
    )

    print(
        "FRAGMENTING VOLUMES WITH UNIFIED METAL"
    )

    print(
        "=" * 80
    )

    volume_objects = [

        (3, silicon),

        (3, vacuum)
    ]

    metal_objects = [

        (2, s)

        for s in metal_surfaces
    ]


    print(
        f"Silicon/vacuum volumes: "
        f"{len(volume_objects)}"
    )

    print(
        f"Unified metal surfaces: "
        f"{len(metal_objects)}"
    )


    fragmented, fragment_map = (
        gmsh.model.occ.fragment(
            volume_objects,
            metal_objects
        )
    )


    print(
        f"Fragmentation produced "
        f"{len(fragmented)} entities."
    )


    # ========================================================
    # SYNCHRONIZE
    # ========================================================

    print(
        "\nSynchronizing fragmented geometry..."
    )

    gmsh.model.occ.synchronize()


    # ========================================================
    # REMOVE DUPLICATES
    # ========================================================

    print(
        "Removing duplicate geometry..."
    )

    gmsh.model.occ.removeAllDuplicates()

    gmsh.model.occ.synchronize()


    # ========================================================
    # IDENTIFY VOLUMES
    # ========================================================

    volumes = gmsh.model.getEntities(3)

    print(
        "\n3D volumes:",
        len(volumes)
    )


    silicon_volumes = []

    vacuum_volumes = []


    for dim, tag in volumes:

        com = (
            gmsh.model.occ.getCenterOfMass(
                dim,
                tag
            )
        )

        if com[2] < -1e-8:

            silicon_volumes.append(tag)

        elif com[2] > 1e-8:

            vacuum_volumes.append(tag)


    print(
        "Silicon volumes:",
        len(silicon_volumes)
    )

    print(
        "Vacuum volumes:",
        len(vacuum_volumes)
    )


    # ========================================================
    # GET METAL SURFACES FROM FRAGMENT MAP
    # ========================================================

    print(
        "\nRecovering metal surfaces..."
    )

    mapped_metal_surfaces = []


    # fragment_map corresponds to input objects.
    #
    # First two input objects are the two volumes.
    # Remaining mappings correspond to metal surfaces.

    for mapping in fragment_map[2:]:

        for dim, tag in mapping:

            if dim == 2:

                mapped_metal_surfaces.append(
                    tag
                )


    # Remove duplicate tags

    mapped_metal_surfaces = sorted(
        set(mapped_metal_surfaces)
    )


    print(
        "Recovered metal surfaces:",
        len(mapped_metal_surfaces)
    )


    if not mapped_metal_surfaces:

        raise RuntimeError(
            "Could not recover metal surfaces "
            "after fragmentation."
        )


    # ========================================================
    # PHYSICAL GROUPS
    # ========================================================

    print(
        "\nCreating physical groups..."
    )


    # --------------------------------------------------------
    # VACUUM
    # --------------------------------------------------------

    gmsh.model.addPhysicalGroup(
        3,
        vacuum_volumes,
        VACUUM_ID
    )

    gmsh.model.setPhysicalName(
        3,
        VACUUM_ID,
        "Vacuum"
    )


    # --------------------------------------------------------
    # SILICON
    # --------------------------------------------------------

    gmsh.model.addPhysicalGroup(
        3,
        silicon_volumes,
        SILICON_ID
    )

    gmsh.model.setPhysicalName(
        3,
        SILICON_ID,
        "Silicon_Substrate"
    )


    # --------------------------------------------------------
    # PEC
    # --------------------------------------------------------

    gmsh.model.addPhysicalGroup(
        2,
        mapped_metal_surfaces,
        PEC_ID
    )

    gmsh.model.setPhysicalName(
        2,
        PEC_ID,
        "Superconducting_Metal"
    )


    # ========================================================
    # MESH SETTINGS
    # ========================================================

    print(
        "\nSetting mesh parameters..."
    )

    gmsh.option.setNumber(
        "Mesh.Algorithm3D",
        1
    )

    gmsh.option.setNumber(
        "Mesh.CharacteristicLengthMin",
        GLOBAL_MIN
    )

    gmsh.option.setNumber(
        "Mesh.CharacteristicLengthMax",
        GLOBAL_MAX
    )

    gmsh.option.setNumber(
        "Mesh.Optimize",
        0
    )

    gmsh.option.setNumber(
        "Mesh.OptimizeNetgen",
        0
    )


    # ========================================================
    # GENERATE 3D MESH
    # ========================================================

    print(
        "\n" + "=" * 80
    )

    print(
        "GENERATING CONFORMAL 3D MESH"
    )

    print(
        "=" * 80
    )

    gmsh.model.mesh.generate(3)


    # ========================================================
    # CLEAN MESH
    # ========================================================

    print(
        "\nRemoving duplicate mesh nodes..."
    )

    gmsh.model.mesh.removeDuplicateNodes()


    print(
        "Removing duplicate mesh elements..."
    )

    gmsh.model.mesh.removeDuplicateElements()


    # ========================================================
    # CHECK ELEMENTS
    # ========================================================

    print(
        "\nChecking element connectivity..."
    )

    element_types, element_tags, element_nodes = (
        gmsh.model.mesh.getElements()
    )

    bad_elements = []

    total_elements = 0


    for etype, tags, nodes in zip(
        element_types,
        element_tags,
        element_nodes
    ):

        properties = (
            gmsh.model.mesh.getElementProperties(
                etype
            )
        )

        element_name = properties[0]

        num_nodes = properties[3]

        total_elements += len(tags)


        for i, tag in enumerate(tags):

            connectivity = nodes[
                i * num_nodes:
                (i + 1) * num_nodes
            ]


            if (
                len(
                    set(
                        connectivity.tolist()
                    )
                )
                !=
                len(connectivity)
            ):

                bad_elements.append(
                    int(tag)
                )


                if len(bad_elements) >= 10:

                    break


    print(
        f"Total elements: "
        f"{total_elements:,}"
    )

    print(
        f"Degenerate elements: "
        f"{len(bad_elements)}"
    )


    if bad_elements:

        print(
            "Bad element IDs:",
            bad_elements[:10]
        )

        raise RuntimeError(
            "Degenerate elements detected."
        )


    # ========================================================
    # FINAL STATISTICS
    # ========================================================

    nodes, coords, _ = (
        gmsh.model.mesh.getNodes()
    )


    print(
        "\n" + "=" * 80
    )

    print(
        "FINAL MESH STATISTICS"
    )

    print(
        "=" * 80
    )

    print(
        f"Nodes    : {len(nodes):,}"
    )

    print(
        f"Elements : {total_elements:,}"
    )


    print(
        "\nPhysical groups:"
    )

    for dim, tag in (
        gmsh.model.getPhysicalGroups()
    ):

        name = gmsh.model.getPhysicalName(
            dim,
            tag
        )

        print(
            f"  Dimension={dim}, "
            f"Attribute={tag}, "
            f"Name={name}"
        )


    # ========================================================
    # WRITE MSH 2.2
    # ========================================================

    print(
        "\nWriting MSH 2.2..."
    )

    gmsh.option.setNumber(
        "Mesh.MshFileVersion",
        2.2
    )

    gmsh.option.setNumber(
        "Mesh.Binary",
        0
    )

    gmsh.write(
        MESH_PATH
    )


    print(
        "\n" + "=" * 80
    )

    print(
        "CONFORMAL PALACE MESH SUCCESS"
    )

    print(
        "=" * 80
    )

    print(
        "\nMesh:"
    )

    print(
        MESH_PATH
    )


finally:

    gmsh.finalize()

    print(
        "\nGmsh finalized."
    )

ROBUST CONFORMAL PALACE MESH GENERATOR

Reading GDS...
Loaded 19 cells.
Using top cell: TOP_main_1

Extracting Layer 1 / Datatype 0...
Raw metal polygons: 2450

Converting polygons to Shapely...
Valid Shapely polygons: 2450
Invalid/skipped: 0

UNITING OVERLAPPING METAL POLYGONS
This may take some time...

Metal union type: Polygon
Unified metal area: 185.75073679172058 mm^2
Unified metal regions: 1

Initializing Gmsh...

Creating silicon...
Creating vacuum...

Creating unified metal surfaces...

Unified metal surfaces created: 1

Removing duplicate OCC geometry...

FRAGMENTING VOLUMES WITH UNIFIED METAL
Silicon/vacuum volumes: 2
Unified metal surfaces: 1
Fragmentation produced 35 entities.

Synchronizing fragmented geometry...
Removing duplicate geometry...

3D volumes: 2
Silicon volumes: 1
Vacuum volumes: 1

Recovering metal surfaces...
Recovered metal surfaces: 1

Creating physical groups...

Setting mesh parameters...

GENERATING CONFORMAL 3D MESH

Removing duplicate mesh nodes...
R

In [ ]:
import os

# ==========================================
# STEP 19: Save Success Status to Google Drive permanently
# ==========================================
success_output = """================================================================================
CONFORMAL PALACE MESH SUCCESS
================================================================================

Mesh:
/content/quantum_chip_mesh_conformal.msh

Gmsh finalized."""

# Define permanent storage path in Google Drive
drive_dir = "/content/drive/MyDrive/Palace6"
os.makedirs(drive_dir, exist_ok=True)
log_file_path = os.path.join(drive_dir, "conformal_mesh_success.txt")

# Write output to Drive
with open(log_file_path, "w") as f:
    f.write(success_output)

print(f"Status successfully saved permanently to Google Drive at:\n{log_file_path}")

Status successfully saved permanently to Google Drive at:
/content/drive/MyDrive/Palace6/conformal_mesh_success.txt


In [ ]:
import shutil
import os

# Define the source and destination paths
source_file = '/content/quantum_chip_mesh_conformal.msh'
destination_dir = '/content/drive/MyDrive/Palace6/meshes'
destination_file = os.path.join(destination_dir, 'quantum_chip_mesh_conformal.msh')

# Create the destination directory if it doesn't exist
os.makedirs(destination_dir, exist_ok=True)

# Copy the file to Google Drive
if os.path.exists(source_file):
    shutil.copy2(source_file, destination_file)
    print(f"File successfully saved permanently to: {destination_file}")
else:
    print(f"Error: {source_file} not found. Please ensure the file was generated in the current session.")

File successfully saved permanently to: /content/drive/MyDrive/Palace6/meshes/quantum_chip_mesh_conformal.msh


In [ ]:
import shutil
import os

# ==========================================
# STEP 20: Permanent Google Drive Archival of Mesh & GDS Files
# ==========================================
files_to_save = [
    "/content/quantum_chip_export_manuscript.gds",
    "/content/quantum_chip_mesh.msh",
    "/content/quantum_chip_mesh_palace.msh",
    "/content/quantum_chip_mesh_palace_v22.msh",
    "/content/quantum_chip_mesh_test.msh",
    "/content/quantum_chip_mesh_conformal.msh"
]

destination_dir = "/content/drive/MyDrive/Palace6/meshes"
os.makedirs(destination_dir, exist_ok=True)

print(f"Beginning permanent archival to Google Drive: {destination_dir}\n")

saved_count = 0
for file_path in files_to_save:
    if os.path.exists(file_path):
        file_name = os.path.basename(file_path)
        dest_path = os.path.join(destination_dir, file_name)
        shutil.copy2(file_path, dest_path)
        print(f"[SAVED] {file_name} -> {dest_path}")
        saved_count += 1
    else:
        print(f"[SKIPPED] File not found locally: {file_path}")

print(f"\nSuccessfully archived {saved_count} file(s) permanently to Google Drive.")

Beginning permanent archival to Google Drive: /content/drive/MyDrive/Palace6/meshes

[SAVED] quantum_chip_export_manuscript.gds -> /content/drive/MyDrive/Palace6/meshes/quantum_chip_export_manuscript.gds
[SAVED] quantum_chip_mesh.msh -> /content/drive/MyDrive/Palace6/meshes/quantum_chip_mesh.msh
[SAVED] quantum_chip_mesh_palace.msh -> /content/drive/MyDrive/Palace6/meshes/quantum_chip_mesh_palace.msh
[SAVED] quantum_chip_mesh_palace_v22.msh -> /content/drive/MyDrive/Palace6/meshes/quantum_chip_mesh_palace_v22.msh
[SAVED] quantum_chip_mesh_test.msh -> /content/drive/MyDrive/Palace6/meshes/quantum_chip_mesh_test.msh
[SAVED] quantum_chip_mesh_conformal.msh -> /content/drive/MyDrive/Palace6/meshes/quantum_chip_mesh_conformal.msh

Successfully archived 6 file(s) permanently to Google Drive.


In [ ]:
import os

# ==========================================
# STEP 21: List files in Google Drive meshes directory
# ==========================================
meshes_dir = "/content/drive/MyDrive/Palace6/meshes"

if os.path.exists(meshes_dir):
    files = os.listdir(meshes_dir)
    print(f"Files found in '{meshes_dir}':")
    for file in files:
        print(f" - {file}")
else:
    print(f"Directory not found: {meshes_dir}")

Files found in '/content/drive/MyDrive/Palace6/meshes':
 - quantum_chip_mesh_conformal.msh
 - quantum_chip_export_manuscript.gds
 - quantum_chip_mesh.msh
 - quantum_chip_mesh_palace.msh
 - quantum_chip_mesh_palace_v22.msh
 - quantum_chip_mesh_test.msh


In [ ]:
# ============================================================
# PALACE MESH-READ TEST + 1 EIGENMODE
#
# Purpose:
#   Confirm that Palace/MFEM can successfully read:
#
#       quantum_chip_mesh_conformal.msh
#
#   and proceed into the eigenmode solver.
#
# NO Josephson junctions yet.
# NO EPR yet.
# ============================================================

import os
import json
import subprocess
import shutil


# ============================================================
# PATHS
# ============================================================

PALACE_DIR = "/content/drive/MyDrive/Palace6/build"

PALACE_BIN = os.path.join(
    PALACE_DIR,
    "bin",
    "palace-x86_64.bin"
)

LIB_DIR = os.path.join(
    PALACE_DIR,
    "lib"
)

MESH = "/content/drive/MyDrive/Palace6/meshes/quantum_chip_mesh_conformal.msh"

CONFIG = "/content/palace_mesh_test.json"

OUTPUT = "/content/palace_mesh_test_output"


# ============================================================
# CHECK FILES
# ============================================================

print("=" * 80)
print("PALACE MESH READ + 1 MODE TEST")
print("=" * 80)

if not os.path.exists(MESH):

    raise FileNotFoundError(
        f"\nMesh not found:\n{MESH}"
    )

if not os.path.exists(PALACE_BIN):

    raise FileNotFoundError(
        f"\nPalace executable not found:\n{PALACE_BIN}"
    )


mesh_size_gb = (
    os.path.getsize(MESH)
    / (1024 ** 3)
)

print("\nMesh:")
print(MESH)

print(
    f"\nMesh file size: "
    f"{mesh_size_gb:.2f} GB"
)

print(
    "\nExpected mesh:"
)

print(
    "  Nodes    : ~3.69 million"
)

print(
    "  Elements : ~25.97 million"
)


# ============================================================
# OUTPUT DIRECTORY
# ============================================================

if os.path.exists(OUTPUT):

    print(
        "\nRemoving previous Palace output..."
    )

    shutil.rmtree(
        OUTPUT
    )

os.makedirs(
    OUTPUT,
    exist_ok=True
)


# ============================================================
# ENVIRONMENT
# ============================================================

env = os.environ.copy()

env["LD_LIBRARY_PATH"] = (
    LIB_DIR
    + ":"
    + env.get(
        "LD_LIBRARY_PATH",
        ""
    )
)

env["OMPI_ALLOW_RUN_AS_ROOT"] = "1"

env[
    "OMPI_ALLOW_RUN_AS_ROOT_CONFIRM"
] = "1"


# ============================================================
# PALACE CONFIGURATION
# ============================================================

palace_config = {

    "Problem": {

        "Type": "Eigenmode",

        "Verbose": 2,

        "Output": OUTPUT
    },


    "Model": {

        "Mesh": MESH,

        # Gmsh coordinates are in mm.
        # Convert mm -> meters.
        "L0": 1.0e-3
    },


    "Domains": {

        "Materials": [

            {
                # Attribute 1 = Vacuum
                "Attributes": [1],

                "Permittivity": 1.0,

                "Permeability": 1.0
            },

            {
                # Attribute 2 = Silicon
                "Attributes": [2],

                "Permittivity": 11.7,

                "Permeability": 1.0
            }
        ]
    },


    "Boundaries": {

        # Attribute 10 = superconducting metal
        "PEC": {

            "Attributes": [10]
        }
    },


    "Solver": {

        # Lowest order elements to minimize RAM.
        "Order": 1,


        "Eigenmode": {

            # ONLY ONE MODE
            "N": 1,

            # Search target = 5 GHz
            "Target": 5.0,

            # Relaxed tolerance for this sanity test.
            "Tol": 1.0e-6,

            "MaxIts": 100
        },


        "Linear": {

            "Type": "BoomerAMG",

            "Tol": 1.0e-6,

            "MaxIts": 50
        }
    }
}


# ============================================================
# WRITE CONFIG
# ============================================================

with open(
    CONFIG,
    "w"
) as f:

    json.dump(
        palace_config,
        f,
        indent=4
    )


print(
    "\nConfiguration written:"
)

print(
    CONFIG
)


# ============================================================
# COMMAND
# ============================================================

cmd = [

    "mpirun",

    "-n",
    "1",

    "--allow-run-as-root",

    PALACE_BIN,

    CONFIG
]


print(
    "\n" + "=" * 80
)

print(
    "STARTING PALACE"
)

print(
    "=" * 80
)

print(
    "\nCommand:"
)

print(
    " ".join(cmd)
)


# ============================================================
# RUN
# ============================================================

try:

    result = subprocess.run(

        cmd,

        text=True,

        capture_output=True,

        env=env
    )

except Exception as exc:

    print(
        "\nCould not start Palace:"
    )

    print(exc)

    raise


# ============================================================
# PRINT OUTPUT
# ============================================================

print(
    "\n" + "=" * 80
)

print(
    "PALACE STDOUT"
)

print(
    "=" * 80
)

print(
    result.stdout
)


print(
    "\n" + "=" * 80
)

print(
    "PALACE STDERR"
)

print(
    "=" * 80
)

print(
    result.stderr
)


print(
    "\n" + "=" * 80
)

print(
    f"RETURN CODE: {result.returncode}"
)

print(
    "=" * 80
)


# ============================================================
# RESULT
# ============================================================

if result.returncode == 0:

    print(
        "\nSUCCESS!"
    )

    print(
        "Palace successfully read the conformal mesh "
        "and completed the 1-mode test."
    )

    print(
        "\nOutput directory:"
    )

    print(
        OUTPUT
    )

else:

    print(
        "\nPALACE TEST FAILED."
    )

    print(
        "\nThe important part of STDERR is:"
    )

    print(
        result.stderr[-5000:]
    )

PALACE MESH READ + 1 MODE TEST

Mesh:
/content/drive/MyDrive/Palace6/meshes/quantum_chip_mesh_conformal.msh

Mesh file size: 1.31 GB

Expected mesh:
  Nodes    : ~3.69 million
  Elements : ~25.97 million

Configuration written:
/content/palace_mesh_test.json

STARTING PALACE

Command:
mpirun -n 1 --allow-run-as-root /content/drive/MyDrive/Palace6/build/bin/palace-x86_64.bin /content/palace_mesh_test.json

PALACE STDOUT

_____________     _______
_____   __   \____ __   /____ ____________
____   /_/  /  __ ` /  /  __ ` /  ___/  _ \
___   _____/  /_/  /  /  /_/  /  /__/  ___/
  /__/     \___,__/__/\___,__/\_____\_____/

Git changeset ID: v0.17.0-194-g01f5d9bd1
Running with 1 MPI process, 1 OpenMP thread
Device configuration: omp,cpu
Memory configuration: host-std
libCEED backend: /cpu/self/xsmm/blocked


--> Warning!
51546 mesh faces with no associated boundary element exist on the domain boundary!

Added 560 elements in 2 iterations of local bisection for under-resolved interior boundar

In [ ]:
!pip install gdspy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.9/157.9 kB 4.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for gdspy: filename=gdspy-1.6.13-cp312-cp312-linux_x86_64.whl size=595854 sha256=3e1da8e840d15ad2cde6edf137efab5bd3917c3e8a355a108e328929a3598d75
  Stored in directory: /root/.cache/pip/wheels/02/05/a3/c4b581f8330cedff4d7f4aa7134ae298e88232ccce6a2d4859
Successfully built gdspy


In [ ]:
# ============================================================
# STEP 5E — MEMORY-EFFICIENT CONFORMAL PALACE MESH
#
# Goal:
#   Reduce the ~26 million element mesh to a much smaller
#   Palace-compatible conformal mesh.
#
# Geometry:
#   4 concentric transmons
#   8 JJ locations
#   readout structures
#   couplers
#   drive / flux structures
#
# Strategy:
#
#   Bulk region              ~200 um
#   Metal/interface region    ~75 um
#   JJ neighborhoods          ~20 um
#
# IMPORTANT:
#   This is still a PEC-only sanity mesh.
#   Josephson inductances are NOT added yet.
# ============================================================

import os
import gdspy
import gmsh

from shapely.geometry import Polygon
from shapely.ops import unary_union


# ============================================================
# INPUT / OUTPUT
# ============================================================

GDS_PATH = "/content/drive/MyDrive/Palace6/meshes/quantum_chip_export_manuscript.gds"

OUTPUT_MESH = (
    "/content/quantum_chip_mesh_reduced_conformal.msh"
)

TOP_CELL_NAME = "TOP_main_1"

METAL_LAYER = 1
METAL_DATATYPE = 0


# ============================================================
# CHIP GEOMETRY
# ============================================================

CHIP_X = 14.0
CHIP_Y = 14.0

SUBSTRATE_THICKNESS = 0.5
AIR_HEIGHT = 0.5


# ============================================================
# PHYSICAL GROUP IDS
# ============================================================

VACUUM_ID = 1
SILICON_ID = 2
PEC_ID = 10


# ============================================================
# MESH SIZES
#
# Units are mm because GDS coordinates are in mm.
# ============================================================

BULK_SIZE = 0.200       # 200 um

METAL_SIZE = 0.075      # 75 um

JJ_SIZE = 0.020         # 20 um


# ============================================================
# JJ LOCATIONS FROM YOUR GDS
#
# mm
# ============================================================

JJ_LOCATIONS = [

    (-1.1325,  0.0000),
    (-0.8675,  0.0000),

    ( 1.2675,  0.0000),
    ( 1.5325,  0.0000),

    (-1.1325, -2.7000),
    (-0.8675, -2.7000),

    ( 1.2675, -2.7000),
    ( 1.5325, -2.7000)
]


# ============================================================
# JJ REFINEMENT BOX
#
# 100 um × 100 um × full substrate/air height
# ============================================================

JJ_BOX_HALF = 0.075       # 75 um


# ============================================================
# READ GDS
# ============================================================

print("=" * 80)
print("MEMORY-EFFICIENT CONFORMAL PALACE MESH")
print("=" * 80)

print("\nReading GDS...")

lib = gdspy.GdsLibrary()

lib.read_gds(
    GDS_PATH
)

print(
    f"Loaded {len(lib.cells)} cells."
)


# ============================================================
# TOP CELL
# ============================================================

if TOP_CELL_NAME not in lib.cells:

    raise RuntimeError(
        f"Top cell not found: {TOP_CELL_NAME}"
    )

top = lib.cells[
    TOP_CELL_NAME
]

print(
    f"Using top cell: {TOP_CELL_NAME}"
)


# ============================================================
# EXTRACT METAL
# ============================================================

print(
    "\nExtracting Layer 1 / Datatype 0..."
)

polygons_by_spec = top.get_polygons(
    by_spec=True
)

raw_polygons = []


for spec, polygons in polygons_by_spec.items():

    layer, datatype = spec

    if layer != METAL_LAYER:
        continue

    if datatype != METAL_DATATYPE:
        continue

    for polygon in polygons:

        if len(polygon) < 3:
            continue

        points = []

        for p in polygon:

            point = (
                round(float(p[0]), 10),
                round(float(p[1]), 10)
            )

            if (
                not points
                or point != points[-1]
            ):

                points.append(point)


        if (
            len(points) > 1
            and points[0] == points[-1]
        ):

            points.pop()


        if len(set(points)) >= 3:

            raw_polygons.append(
                points
            )


print(
    f"Raw metal polygons: "
    f"{len(raw_polygons)}"
)


# ============================================================
# SHAPELY CLEANUP
# ============================================================

print(
    "\nCleaning metal geometry..."
)

shapes = []

for points in raw_polygons:

    try:

        poly = Polygon(points)

        if not poly.is_valid:

            poly = poly.buffer(0)

        if poly.is_empty:

            continue

        if poly.area <= 1e-12:

            continue

        shapes.append(poly)

    except Exception:

        continue


print(
    f"Valid polygons: "
    f"{len(shapes)}"
)


# ============================================================
# UNION
# ============================================================

print(
    "\nUniting overlapping metal polygons..."
)

metal_union = unary_union(
    shapes
)


if not metal_union.is_valid:

    metal_union = metal_union.buffer(0)


if metal_union.geom_type == "Polygon":

    unified_polygons = [
        metal_union
    ]

elif metal_union.geom_type == "MultiPolygon":

    unified_polygons = list(
        metal_union.geoms
    )

else:

    raise RuntimeError(
        "Unexpected metal geometry type: "
        + metal_union.geom_type
    )


print(
    f"Unified metal regions: "
    f"{len(unified_polygons)}"
)


# ============================================================
# INITIALIZE GMSH
# ============================================================

print(
    "\nInitializing Gmsh..."
)

gmsh.initialize()

gmsh.option.setNumber(
    "General.Terminal",
    1
)

gmsh.model.add(
    "four_qubit_reduced_conformal"
)


try:

    # ========================================================
    # SILICON
    # ========================================================

    print(
        "\nCreating silicon substrate..."
    )

    silicon = gmsh.model.occ.addBox(

        -CHIP_X / 2,
        -CHIP_Y / 2,
        -SUBSTRATE_THICKNESS,

        CHIP_X,
        CHIP_Y,
        SUBSTRATE_THICKNESS
    )


    # ========================================================
    # VACUUM
    # ========================================================

    print(
        "Creating vacuum..."
    )

    vacuum = gmsh.model.occ.addBox(

        -CHIP_X / 2,
        -CHIP_Y / 2,
        0.0,

        CHIP_X,
        CHIP_Y,
        AIR_HEIGHT
    )


    # ========================================================
    # POINT CACHE
    # ========================================================

    point_cache = {}


    def get_point(x, y):

        key = (
            round(float(x), 10),
            round(float(y), 10),
            0.0
        )

        if key not in point_cache:

            point_cache[key] = (
                gmsh.model.occ.addPoint(
                    key[0],
                    key[1],
                    key[2]
                )
            )

        return point_cache[key]


    # ========================================================
    # CREATE UNIFIED METAL SURFACES
    # ========================================================

    print(
        "\nCreating unified metal surfaces..."
    )

    metal_surfaces = []


    for index, poly in enumerate(
        unified_polygons
    ):

        # ----------------------------------------------------
        # EXTERIOR
        # ----------------------------------------------------

        exterior = list(
            poly.exterior.coords
        )

        exterior = exterior[:-1]


        outer_points = [

            get_point(x, y)

            for x, y in exterior
        ]


        outer_lines = []


        for i in range(
            len(outer_points)
        ):

            p1 = outer_points[i]

            p2 = outer_points[
                (i + 1)
                % len(outer_points)
            ]

            if p1 == p2:

                continue

            outer_lines.append(

                gmsh.model.occ.addLine(
                    p1,
                    p2
                )
            )


        outer_wire = (
            gmsh.model.occ.addWire(
                outer_lines
            )
        )


        # ----------------------------------------------------
        # HOLES
        # ----------------------------------------------------

        hole_wires = []


        for interior in poly.interiors:

            coords = list(
                interior.coords
            )

            coords = coords[:-1]


            hole_points = [

                get_point(x, y)

                for x, y in coords
            ]


            hole_lines = []


            for i in range(
                len(hole_points)
            ):

                p1 = hole_points[i]

                p2 = hole_points[
                    (i + 1)
                    % len(hole_points)
                ]

                if p1 == p2:

                    continue

                hole_lines.append(

                    gmsh.model.occ.addLine(
                        p1,
                        p2
                    )
                )


            if len(hole_lines) >= 3:

                hole_wires.append(

                    gmsh.model.occ.addWire(
                        hole_lines
                    )
                )


        # ----------------------------------------------------
        # SURFACE
        # ----------------------------------------------------

        wire_list = [
            outer_wire
        ] + hole_wires


        try:

            surface = (
                gmsh.model.occ.addPlaneSurface(
                    wire_list
                )
            )

            metal_surfaces.append(
                surface
            )

        except Exception as exc:

            print(
                f"Metal region {index} "
                f"failed: {exc}"
            )


    print(
        f"Unified metal surfaces created: "
        f"{len(metal_surfaces)}"
    )


    # ========================================================
    # SYNCHRONIZE
    # ========================================================

    gmsh.model.occ.synchronize()


    # ========================================================
    # REMOVE DUPLICATES
    # ========================================================

    print(
        "\nRemoving duplicate OCC geometry..."
    )

    gmsh.model.occ.removeAllDuplicates()

    gmsh.model.occ.synchronize()


    # ========================================================
    # FRAGMENT
    # ========================================================

    print(
        "\n" + "=" * 80
    )

    print(
        "FRAGMENTING VOLUMES WITH UNIFIED METAL"
    )

    print(
        "=" * 80
    )


    volume_objects = [

        (3, silicon),
        (3, vacuum)
    ]


    metal_objects = [

        (2, tag)

        for tag in metal_surfaces
    ]


    fragmented, fragment_map = (
        gmsh.model.occ.fragment(

            volume_objects,
            metal_objects
        )
    )


    print(
        f"Fragmentation produced "
        f"{len(fragmented)} entities."
    )


    # ========================================================
    # SYNCHRONIZE
    # ========================================================

    print(
        "\nSynchronizing fragmented geometry..."
    )

    gmsh.model.occ.synchronize()


    print(
        "Removing duplicate geometry..."
    )

    gmsh.model.occ.removeAllDuplicates()

    gmsh.model.occ.synchronize()


    # ========================================================
    # VOLUMES
    # ========================================================

    volumes = (
        gmsh.model.getEntities(3)
    )


    silicon_volumes = []

    vacuum_volumes = []


    for dim, tag in volumes:

        com = (
            gmsh.model.occ.getCenterOfMass(
                dim,
                tag
            )
        )


        if com[2] < -1e-8:

            silicon_volumes.append(
                tag
            )

        elif com[2] > 1e-8:

            vacuum_volumes.append(
                tag
            )


    print(
        "\n3D volumes:",
        len(volumes)
    )

    print(
        "Silicon volumes:",
        len(silicon_volumes)
    )

    print(
        "Vacuum volumes:",
        len(vacuum_volumes)
    )


    # ========================================================
    # RECOVER METAL SURFACE
    # ========================================================

    mapped_metal_surfaces = []


    for mapping in fragment_map[2:]:

        for dim, tag in mapping:

            if dim == 2:

                mapped_metal_surfaces.append(
                    tag
                )


    mapped_metal_surfaces = sorted(
        set(
            mapped_metal_surfaces
        )
    )


    print(
        "Recovered metal surfaces:",
        len(mapped_metal_surfaces)
    )


    # ========================================================
    # PHYSICAL GROUPS
    # ========================================================

    print(
        "\nCreating physical groups..."
    )


    gmsh.model.addPhysicalGroup(

        3,

        vacuum_volumes,

        VACUUM_ID
    )

    gmsh.model.setPhysicalName(

        3,

        VACUUM_ID,

        "Vacuum"
    )


    gmsh.model.addPhysicalGroup(

        3,

        silicon_volumes,

        SILICON_ID
    )

    gmsh.model.setPhysicalName(

        3,

        SILICON_ID,

        "Silicon_Substrate"
    )


    gmsh.model.addPhysicalGroup(

        2,

        mapped_metal_surfaces,

        PEC_ID
    )

    gmsh.model.setPhysicalName(

        2,

        PEC_ID,

        "Superconducting_Metal"
    )


    # ========================================================
    # MESH SIZES
    # ========================================================

    print(
        "\nConfiguring reduced mesh..."
    )

    # Global coarse mesh.
    gmsh.option.setNumber(
        "Mesh.CharacteristicLengthMin",
        BULK_SIZE
    )

    gmsh.option.setNumber(
        "Mesh.CharacteristicLengthMax",
        BULK_SIZE
    )


    # ========================================================
    # METAL SURFACE REFINEMENT
    # ========================================================
    #
    # Use a distance field from the metal boundary.
    #
    # This makes the mesh finer near the superconducting
    # structure without forcing the entire chip to 75 um.
    # ========================================================

    print(
        "Creating metal-distance refinement..."
    )


    metal_boundary_curves = []


    for dim, tag in (
        gmsh.model.getBoundary(

            [
                (2, s)

                for s in
                mapped_metal_surfaces
            ],

            combined=True,

            oriented=False,

            recursive=False
        )
    ):

        if dim == 1:

            metal_boundary_curves.append(
                tag
            )


    metal_boundary_curves = sorted(
        set(
            metal_boundary_curves
        )
    )


    if metal_boundary_curves:

        metal_distance = (
            gmsh.model.mesh.field.add(
                "Distance"
            )
        )

        gmsh.model.mesh.field.setNumbers(

            metal_distance,

            "CurvesList",

            metal_boundary_curves
        )


        metal_threshold = (
            gmsh.model.mesh.field.add(
                "Threshold"
            )
        )


        gmsh.model.mesh.field.setNumber(

            metal_threshold,

            "InField",

            metal_distance
        )


        # Fine size close to conductor
        gmsh.model.mesh.field.setNumber(

            metal_threshold,

            "SizeMin",

            METAL_SIZE
        )


        # Coarse size far away
        gmsh.model.mesh.field.setNumber(

            metal_threshold,

            "SizeMax",

            BULK_SIZE
        )


        # Start changing mesh size within 150 um
        gmsh.model.mesh.field.setNumber(

            metal_threshold,

            "DistMin",

            0.05
        )


        gmsh.model.mesh.field.setNumber(

            metal_threshold,

            "DistMax",

            0.20
        )


        print(
            f"Metal refinement curves: "
            f"{len(metal_boundary_curves)}"
        )

    else:

        metal_threshold = None

        print(
            "No metal boundary curves found."
        )


    # ========================================================
    # JJ LOCAL REFINEMENT
    # ========================================================

    print(
        "\nCreating local JJ refinement..."
    )


    jj_fields = []


    for index, (x, y) in enumerate(
        JJ_LOCATIONS
    ):

        field = (
            gmsh.model.mesh.field.add(
                "Box"
            )
        )


        gmsh.model.mesh.field.setNumber(

            field,

            "VIn",

            JJ_SIZE
        )


        gmsh.model.mesh.field.setNumber(

            field,

            "VOut",

            BULK_SIZE
        )


        gmsh.model.mesh.field.setNumber(

            field,

            "XMin",

            x - JJ_BOX_HALF
        )


        gmsh.model.mesh.field.setNumber(

            field,

            "XMax",

            x + JJ_BOX_HALF
        )


        gmsh.model.mesh.field.setNumber(

            field,

            "YMin",

            y - JJ_BOX_HALF
        )


        gmsh.model.mesh.field.setNumber(

            field,

            "YMax",

            y + JJ_BOX_HALF
        )


        gmsh.model.mesh.field.setNumber(

            field,

            "ZMin",

            -SUBSTRATE_THICKNESS
        )


        gmsh.model.mesh.field.setNumber(

            field,

            "ZMax",

            AIR_HEIGHT
        )


        jj_fields.append(
            field
        )


        print(
            f"  JJ {index + 1}: "
            f"({x:.4f}, {y:.4f}) mm"
        )


    # ========================================================
    # COMBINE FIELDS
    # ========================================================

    all_fields = []


    if metal_threshold is not None:

        all_fields.append(
            metal_threshold
        )


    all_fields.extend(
        jj_fields
    )


    if all_fields:

        minimum_field = (
            gmsh.model.mesh.field.add(
                "Min"
            )
        )


        gmsh.model.mesh.field.setNumbers(

            minimum_field,

            "FieldsList",

            all_fields
        )


        gmsh.model.mesh.field.setAsBackgroundMesh(

            minimum_field
        )


    # ========================================================
    # 3D MESH SETTINGS
    # ========================================================

    gmsh.option.setNumber(
        "Mesh.Algorithm3D",
        1
    )

    gmsh.option.setNumber(
        "Mesh.Optimize",
        0
    )

    gmsh.option.setNumber(
        "Mesh.OptimizeNetgen",
        0
    )


    # ========================================================
    # GENERATE
    # ========================================================

    print(
        "\n" + "=" * 80
    )

    print(
        "GENERATING REDUCED CONFORMAL 3D MESH"
    )

    print(
        "=" * 80
    )

    print(
        f"Bulk size  : {BULK_SIZE * 1000:.0f} um"
    )

    print(
        f"Metal size : {METAL_SIZE * 1000:.0f} um"
    )

    print(
        f"JJ size    : {JJ_SIZE * 1000:.0f} um"
    )


    gmsh.model.mesh.generate(
        3
    )


    # ========================================================
    # CLEAN
    # ========================================================

    print(
        "\nRemoving duplicate mesh nodes..."
    )

    gmsh.model.mesh.removeDuplicateNodes()


    print(
        "Removing duplicate mesh elements..."
    )

    gmsh.model.mesh.removeDuplicateElements()


    # ========================================================
    # CONNECTIVITY CHECK
    # ========================================================

    print(
        "\nChecking element connectivity..."
    )


    element_types, element_tags, element_nodes = (
        gmsh.model.mesh.getElements()
    )


    bad_elements = 0

    total_elements = 0


    for etype, tags, nodes in zip(

        element_types,

        element_tags,

        element_nodes

    ):

        props = (
            gmsh.model.mesh.getElementProperties(
                etype
            )
        )

        num_nodes = props[3]

        total_elements += len(tags)


        for i in range(
            len(tags)
        ):

            connectivity = nodes[

                i * num_nodes:

                (i + 1) * num_nodes

            ]


            if (

                len(
                    set(
                        connectivity.tolist()
                    )
                )

                !=

                len(connectivity)

            ):

                bad_elements += 1


                if bad_elements <= 5:

                    print(
                        "Bad element:",
                        int(tags[i])
                    )


    print(
        f"\nTotal elements: "
        f"{total_elements:,}"
    )

    print(
        f"Degenerate elements: "
        f"{bad_elements}"
    )


    if bad_elements:

        raise RuntimeError(
            "Degenerate elements detected."
        )


    # ========================================================
    # NODE COUNT
    # ========================================================

    nodes, coords, _ = (
        gmsh.model.mesh.getNodes()
    )


    print(
        "\n" + "=" * 80
    )

    print(
        "REDUCED MESH STATISTICS"
    )

    print(
        "=" * 80
    )

    print(
        f"Nodes    : {len(nodes):,}"
    )

    print(
        f"Elements : {total_elements:,}"
    )


    # ========================================================
    # PHYSICAL GROUPS
    # ========================================================

    print(
        "\nPhysical groups:"
    )


    for dim, tag in (
        gmsh.model.getPhysicalGroups()
    ):

        name = gmsh.model.getPhysicalName(
            dim,
            tag
        )

        print(
            f"  Dimension={dim}, "
            f"Attribute={tag}, "
            f"Name={name}"
        )


    # ========================================================
    # WRITE MSH 2.2
    # ========================================================

    print(
        "\nWriting MSH 2.2..."
    )


    gmsh.option.setNumber(
        "Mesh.MshFileVersion",
        2.2
    )

    gmsh.option.setNumber(
        "Mesh.Binary",
        0
    )


    gmsh.write(
        OUTPUT_MESH
    )


    print(
        "\n" + "=" * 80
    )

    print(
        "REDUCED CONFORMAL PALACE MESH CREATED"
    )

    print(
        "=" * 80
    )

    print(
        "\nMesh:"
    )

    print(
        OUTPUT_MESH
    )


finally:

    gmsh.finalize()

    print(
        "\nGmsh finalized."
    )

MEMORY-EFFICIENT CONFORMAL PALACE MESH

Reading GDS...
Loaded 19 cells.
Using top cell: TOP_main_1

Extracting Layer 1 / Datatype 0...
Raw metal polygons: 2450

Cleaning metal geometry...
Valid polygons: 2450

Uniting overlapping metal polygons...
Unified metal regions: 1

Initializing Gmsh...

Creating silicon substrate...
Creating vacuum...

Creating unified metal surfaces...
Unified metal surfaces created: 1

Removing duplicate OCC geometry...

FRAGMENTING VOLUMES WITH UNIFIED METAL
Fragmentation produced 35 entities.

Synchronizing fragmented geometry...
Removing duplicate geometry...

3D volumes: 2
Silicon volumes: 1
Vacuum volumes: 1
Recovered metal surfaces: 1

Creating physical groups...

Configuring reduced mesh...
Creating metal-distance refinement...
Metal refinement curves: 36280

Creating local JJ refinement...
  JJ 1: (-1.1325, 0.0000) mm
  JJ 2: (-0.8675, 0.0000) mm
  JJ 3: (1.2675, 0.0000) mm
  JJ 4: (1.5325, 0.0000) mm
  JJ 5: (-1.1325, -2.7000) mm
  JJ 6: (-0.8675, -2

In [ ]:
import shutil
import os

# ==========================================
# STEP 20: Permanent Google Drive Archival of Mesh & GDS Files
# ==========================================
files_to_save = [
    "/content/quantum_chip_mesh_reduced_conformal.msh"
]

destination_dir = "/content/drive/MyDrive/Palace6/meshes"
os.makedirs(destination_dir, exist_ok=True)

print(f"Beginning permanent archival to Google Drive: {destination_dir}\n")

saved_count = 0
for file_path in files_to_save:
    if os.path.exists(file_path):
        file_name = os.path.basename(file_path)
        dest_path = os.path.join(destination_dir, file_name)
        shutil.copy2(file_path, dest_path)
        print(f"[SAVED] {file_name} -> {dest_path}")
        saved_count += 1
    else:
        print(f"[SKIPPED] File not found locally: {file_path}")

print(f"\nSuccessfully archived {saved_count} file(s) permanently to Google Drive.")

Beginning permanent archival to Google Drive: /content/drive/MyDrive/Palace6/meshes

[SAVED] quantum_chip_mesh_reduced_conformal.msh -> /content/drive/MyDrive/Palace6/meshes/quantum_chip_mesh_reduced_conformal.msh

Successfully archived 1 file(s) permanently to Google Drive.


In [ ]:
# ============================================================
# STEP 5F — ULTRA-REDUCED CONFORMAL PALACE MESH
#
# Purpose:
#   Produce a mesh small enough for Palace/Colab memory.
#
# Changes from STEP 5E:
#
#   1. Simplify redundant GDS boundary vertices
#   2. NO global metal-distance refinement
#   3. Coarser bulk mesh = 300 um
#   4. Local JJ refinement = 30 um
#
# IMPORTANT:
#   This is a SANITY mesh.
#   Do not use its final frequencies for device publication.
# ============================================================

import os
import gdspy
import gmsh

from shapely.geometry import Polygon
from shapely.ops import unary_union


# ============================================================
# INPUT
# ============================================================

GDS_PATH =  "/content/drive/MyDrive/Palace6/meshes/quantum_chip_export_manuscript.gds"

OUTPUT_MESH = (
    "/content/drive/MyDrive/Palace6/meshes/quantum_chip_mesh_ultra_reduced.msh"
)

TOP_CELL_NAME = "TOP_main_1"

METAL_LAYER = 1
METAL_DATATYPE = 0


# ============================================================
# CHIP
# ============================================================

CHIP_X = 14.0
CHIP_Y = 14.0

SUBSTRATE_THICKNESS = 0.5
AIR_HEIGHT = 0.5


# ============================================================
# PHYSICAL GROUPS
# ============================================================

VACUUM_ID = 1
SILICON_ID = 2
PEC_ID = 10


# ============================================================
# MESH PARAMETERS
# ============================================================

# Global mesh
BULK_SIZE = 0.300       # 300 um

# JJ local mesh
JJ_SIZE = 0.030         # 30 um

# Size of JJ refinement box
JJ_BOX_HALF = 0.100     # 100 um

# Remove redundant boundary vertices
#
# 0.003 mm = 3 um
#
# This is ONLY for the sanity mesh.
SIMPLIFY_TOL = 0.003


# ============================================================
# JJ LOCATIONS
# ============================================================

JJ_LOCATIONS = [

    (-1.1325,  0.0000),
    (-0.8675,  0.0000),

    ( 1.2675,  0.0000),
    ( 1.5325,  0.0000),

    (-1.1325, -2.7000),
    (-0.8675, -2.7000),

    ( 1.2675, -2.7000),
    ( 1.5325, -2.7000)
]


# ============================================================
# HEADER
# ============================================================

print("=" * 80)
print("ULTRA-REDUCED CONFORMAL PALACE MESH")
print("=" * 80)

print(
    "\nTarget:"
)

print(
    f"  Bulk mesh     : {BULK_SIZE * 1000:.0f} um"
)

print(
    f"  JJ mesh       : {JJ_SIZE * 1000:.0f} um"
)

print(
    f"  Simplification: {SIMPLIFY_TOL * 1000:.1f} um"
)


# ============================================================
# READ GDS
# ============================================================

print(
    "\nReading GDS..."
)

lib = gdspy.GdsLibrary()

lib.read_gds(
    GDS_PATH
)

print(
    f"Loaded {len(lib.cells)} cells."
)


if TOP_CELL_NAME not in lib.cells:

    raise RuntimeError(
        f"Top cell not found: {TOP_CELL_NAME}"
    )


top = lib.cells[
    TOP_CELL_NAME
]

print(
    f"Using top cell: {TOP_CELL_NAME}"
)


# ============================================================
# EXTRACT METAL
# ============================================================

print(
    "\nExtracting Layer 1 / Datatype 0..."
)

polygons_by_spec = top.get_polygons(
    by_spec=True
)

raw_polygons = []


for spec, polygons in polygons_by_spec.items():

    layer, datatype = spec

    if layer != METAL_LAYER:
        continue

    if datatype != METAL_DATATYPE:
        continue

    for polygon in polygons:

        if len(polygon) < 3:
            continue

        points = []

        for p in polygon:

            point = (
                float(p[0]),
                float(p[1])
            )

            if (
                not points
                or point != points[-1]
            ):

                points.append(point)


        if (
            len(points) > 1
            and points[0] == points[-1]
        ):

            points.pop()


        if len(set(points)) >= 3:

            raw_polygons.append(
                points
            )


print(
    f"Raw polygons: "
    f"{len(raw_polygons)}"
)


# ============================================================
# SHAPELY CLEANUP
# ============================================================

print(
    "\nConverting to Shapely..."
)

shapes = []


for points in raw_polygons:

    try:

        poly = Polygon(points)

        if not poly.is_valid:

            poly = poly.buffer(0)

        if poly.is_empty:

            continue

        if poly.area <= 1e-12:

            continue

        shapes.append(poly)

    except Exception:

        pass


print(
    f"Valid polygons: "
    f"{len(shapes)}"
)


# ============================================================
# UNION
# ============================================================

print(
    "\nUniting metal geometry..."
)

metal_union = unary_union(
    shapes
)


if not metal_union.is_valid:

    metal_union = metal_union.buffer(0)


print(
    "Original union:",
    metal_union.geom_type
)


# ============================================================
# SIMPLIFY GEOMETRY
# ============================================================

print(
    "\nSimplifying redundant boundary vertices..."
)

original_area = metal_union.area

simplified_union = metal_union.simplify(
    SIMPLIFY_TOL,
    preserve_topology=True
)


if simplified_union.is_empty:

    raise RuntimeError(
        "Simplified metal geometry is empty."
    )


if not simplified_union.is_valid:

    print(
        "Simplified geometry invalid; repairing..."
    )

    simplified_union = (
        simplified_union.buffer(0)
    )


new_area = simplified_union.area


print(
    f"Original metal area : "
    f"{original_area:.6f} mm^2"
)

print(
    f"Simplified area     : "
    f"{new_area:.6f} mm^2"
)

print(
    f"Area change         : "
    f"{100.0 * abs(new_area-original_area) / original_area:.4f} %"
)


# ============================================================
# EXTRACT POLYGONS
# ============================================================

if simplified_union.geom_type == "Polygon":

    unified_polygons = [
        simplified_union
    ]

elif simplified_union.geom_type == "MultiPolygon":

    unified_polygons = list(
        simplified_union.geoms
    )

else:

    raise RuntimeError(
        "Unexpected geometry after simplification: "
        + simplified_union.geom_type
    )


print(
    f"Final metal regions: "
    f"{len(unified_polygons)}"
)


# ============================================================
# INITIALIZE GMSH
# ============================================================

print(
    "\nInitializing Gmsh..."
)

gmsh.initialize()

gmsh.option.setNumber(
    "General.Terminal",
    1
)

gmsh.model.add(
    "four_qubit_ultra_reduced"
)


try:

    # ========================================================
    # SUBSTRATE
    # ========================================================

    print(
        "\nCreating silicon..."
    )

    silicon = gmsh.model.occ.addBox(

        -CHIP_X / 2,
        -CHIP_Y / 2,
        -SUBSTRATE_THICKNESS,

        CHIP_X,
        CHIP_Y,
        SUBSTRATE_THICKNESS
    )


    # ========================================================
    # VACUUM
    # ========================================================

    print(
        "Creating vacuum..."
    )

    vacuum = gmsh.model.occ.addBox(

        -CHIP_X / 2,
        -CHIP_Y / 2,
        0.0,

        CHIP_X,
        CHIP_Y,
        AIR_HEIGHT
    )


    # ========================================================
    # POINT CACHE
    # ========================================================

    point_cache = {}


    def get_point(x, y):

        key = (
            round(float(x), 9),
            round(float(y), 9),
            0.0
        )

        if key not in point_cache:

            point_cache[key] = (
                gmsh.model.occ.addPoint(
                    key[0],
                    key[1],
                    key[2]
                )
            )

        return point_cache[key]


    # ========================================================
    # CREATE METAL SURFACE
    # ========================================================

    print(
        "\nCreating simplified metal surfaces..."
    )

    metal_surfaces = []


    for index, poly in enumerate(
        unified_polygons
    ):

        # ----------------------------------------------------
        # OUTER BOUNDARY
        # ----------------------------------------------------

        exterior = list(
            poly.exterior.coords
        )

        exterior = exterior[:-1]


        if len(exterior) < 3:

            continue


        outer_points = [

            get_point(x, y)

            for x, y in exterior
        ]


        outer_lines = []


        for i in range(
            len(outer_points)
        ):

            p1 = outer_points[i]

            p2 = outer_points[
                (i + 1)
                % len(outer_points)
            ]


            if p1 == p2:

                continue


            outer_lines.append(

                gmsh.model.occ.addLine(
                    p1,
                    p2
                )
            )


        if len(outer_lines) < 3:

            continue


        outer_wire = (
            gmsh.model.occ.addWire(
                outer_lines
            )
        )


        # ----------------------------------------------------
        # HOLES
        # ----------------------------------------------------

        hole_wires = []


        for interior in poly.interiors:

            coords = list(
                interior.coords
            )

            coords = coords[:-1]


            if len(coords) < 3:

                continue


            hole_points = [

                get_point(x, y)

                for x, y in coords
            ]


            hole_lines = []


            for i in range(
                len(hole_points)
            ):

                p1 = hole_points[i]

                p2 = hole_points[
                    (i + 1)
                    % len(hole_points)
                ]


                if p1 == p2:

                    continue


                hole_lines.append(

                    gmsh.model.occ.addLine(
                        p1,
                        p2
                    )
                )


            if len(hole_lines) >= 3:

                hole_wires.append(

                    gmsh.model.occ.addWire(
                        hole_lines
                    )
                )


        # ----------------------------------------------------
        # SURFACE
        # ----------------------------------------------------

        try:

            surface = (
                gmsh.model.occ.addPlaneSurface(

                    [outer_wire]
                    + hole_wires
                )
            )

            metal_surfaces.append(
                surface
            )

        except Exception as exc:

            print(
                f"Metal region {index} failed:"
            )

            print(exc)


    print(
        f"Metal surfaces created: "
        f"{len(metal_surfaces)}"
    )


    # ========================================================
    # OCC SYNC
    # ========================================================

    gmsh.model.occ.synchronize()


    print(
        "\nRemoving duplicate OCC geometry..."
    )

    gmsh.model.occ.removeAllDuplicates()

    gmsh.model.occ.synchronize()


    # ========================================================
    # FRAGMENT
    # ========================================================

    print(
        "\n" + "=" * 80
    )

    print(
        "FRAGMENTING SILICON / VACUUM"
    )

    print(
        "=" * 80
    )


    fragmented, fragment_map = (
        gmsh.model.occ.fragment(

            [
                (3, silicon),
                (3, vacuum)
            ],

            [
                (2, s)
                for s in metal_surfaces
            ]
        )
    )


    print(
        f"Fragmentation produced "
        f"{len(fragmented)} entities."
    )


    gmsh.model.occ.synchronize()


    # ========================================================
    # REMOVE DUPLICATES
    # ========================================================

    print(
        "\nRemoving duplicate geometry..."
    )

    gmsh.model.occ.removeAllDuplicates()

    gmsh.model.occ.synchronize()


    # ========================================================
    # FIND VOLUMES
    # ========================================================

    volumes = (
        gmsh.model.getEntities(3)
    )


    silicon_volumes = []
    vacuum_volumes = []


    for dim, tag in volumes:

        com = (
            gmsh.model.occ.getCenterOfMass(
                dim,
                tag
            )
        )


        if com[2] < -1e-8:

            silicon_volumes.append(tag)

        elif com[2] > 1e-8:

            vacuum_volumes.append(tag)


    print(
        "\n3D volumes:",
        len(volumes)
    )

    print(
        "Silicon volumes:",
        len(silicon_volumes)
    )

    print(
        "Vacuum volumes:",
        len(vacuum_volumes)
    )


    # ========================================================
    # RECOVER METAL
    # ========================================================

    mapped_metal_surfaces = []


    for mapping in fragment_map[2:]:

        for dim, tag in mapping:

            if dim == 2:

                mapped_metal_surfaces.append(
                    tag
                )


    mapped_metal_surfaces = sorted(
        set(mapped_metal_surfaces)
    )


    print(
        "Recovered metal surfaces:",
        len(mapped_metal_surfaces)
    )


    # ========================================================
    # PHYSICAL GROUPS
    # ========================================================

    print(
        "\nCreating physical groups..."
    )


    gmsh.model.addPhysicalGroup(
        3,
        vacuum_volumes,
        VACUUM_ID
    )

    gmsh.model.setPhysicalName(
        3,
        VACUUM_ID,
        "Vacuum"
    )


    gmsh.model.addPhysicalGroup(
        3,
        silicon_volumes,
        SILICON_ID
    )

    gmsh.model.setPhysicalName(
        3,
        SILICON_ID,
        "Silicon_Substrate"
    )


    gmsh.model.addPhysicalGroup(
        2,
        mapped_metal_surfaces,
        PEC_ID
    )

    gmsh.model.setPhysicalName(
        2,
        PEC_ID,
        "Superconducting_Metal"
    )


    # ========================================================
    # MESH SIZE
    # ========================================================

    print(
        "\nSetting global mesh size..."
    )


    gmsh.option.setNumber(
        "Mesh.CharacteristicLengthMin",
        BULK_SIZE
    )

    gmsh.option.setNumber(
        "Mesh.CharacteristicLengthMax",
        BULK_SIZE
    )


    # ========================================================
    # JJ LOCAL REFINEMENT ONLY
    # ========================================================

    print(
        "\nCreating JJ local refinement..."
    )


    jj_fields = []


    for i, (x, y) in enumerate(
        JJ_LOCATIONS
    ):

        field = (
            gmsh.model.mesh.field.add(
                "Box"
            )
        )


        gmsh.model.mesh.field.setNumber(
            field,
            "VIn",
            JJ_SIZE
        )


        gmsh.model.mesh.field.setNumber(
            field,
            "VOut",
            BULK_SIZE
        )


        gmsh.model.mesh.field.setNumber(
            field,
            "XMin",
            x - JJ_BOX_HALF
        )


        gmsh.model.mesh.field.setNumber(
            field,
            "XMax",
            x + JJ_BOX_HALF
        )


        gmsh.model.mesh.field.setNumber(
            field,
            "YMin",
            y - JJ_BOX_HALF
        )


        gmsh.model.mesh.field.setNumber(
            field,
            "YMax",
            y + JJ_BOX_HALF
        )


        gmsh.model.mesh.field.setNumber(
            field,
            "ZMin",
            -SUBSTRATE_THICKNESS
        )


        gmsh.model.mesh.field.setNumber(
            field,
            "ZMax",
            AIR_HEIGHT
        )


        jj_fields.append(
            field
        )


        print(
            f"  JJ {i+1}: "
            f"({x:.4f}, {y:.4f}) mm"
        )


    # ========================================================
    # COMBINE JJ FIELDS
    # ========================================================

    if jj_fields:

        jj_min_field = (
            gmsh.model.mesh.field.add(
                "Min"
            )
        )


        gmsh.model.mesh.field.setNumbers(
            jj_min_field,
            "FieldsList",
            jj_fields
        )


        gmsh.model.mesh.field.setAsBackgroundMesh(
            jj_min_field
        )


    # ========================================================
    # 3D MESH SETTINGS
    # ========================================================

    gmsh.option.setNumber(
        "Mesh.Algorithm3D",
        1
    )

    gmsh.option.setNumber(
        "Mesh.Optimize",
        0
    )

    gmsh.option.setNumber(
        "Mesh.OptimizeNetgen",
        0
    )


    # ========================================================
    # GENERATE
    # ========================================================

    print(
        "\n" + "=" * 80
    )

    print(
        "GENERATING ULTRA-REDUCED 3D MESH"
    )

    print(
        "=" * 80
    )


    gmsh.model.mesh.generate(
        3
    )


    # ========================================================
    # CLEAN
    # ========================================================

    print(
        "\nRemoving duplicate nodes..."
    )

    gmsh.model.mesh.removeDuplicateNodes()


    print(
        "Removing duplicate elements..."
    )

    gmsh.model.mesh.removeDuplicateElements()


    # ========================================================
    # CHECK CONNECTIVITY
    # ========================================================

    print(
        "\nChecking element connectivity..."
    )


    element_types, element_tags, element_nodes = (
        gmsh.model.mesh.getElements()
    )


    total_elements = 0
    bad_elements = 0


    for etype, tags, nodes in zip(
        element_types,
        element_tags,
        element_nodes
    ):

        props = (
            gmsh.model.mesh.getElementProperties(
                etype
            )
        )

        num_nodes = props[3]

        total_elements += len(tags)


        for i in range(
            len(tags)
        ):

            conn = nodes[
                i * num_nodes:
                (i + 1) * num_nodes
            ]


            if (
                len(
                    set(
                        conn.tolist()
                    )
                )
                != len(conn)
            ):

                bad_elements += 1


    print(
        f"Total elements: "
        f"{total_elements:,}"
    )

    print(
        f"Degenerate elements: "
        f"{bad_elements}"
    )


    if bad_elements:

        raise RuntimeError(
            "Degenerate elements remain."
        )


    # ========================================================
    # STATISTICS
    # ========================================================

    node_tags, node_coords, _ = (
        gmsh.model.mesh.getNodes()
    )


    print(
        "\n" + "=" * 80
    )

    print(
        "ULTRA-REDUCED MESH STATISTICS"
    )

    print(
        "=" * 80
    )

    print(
        f"Nodes    : "
        f"{len(node_tags):,}"
    )

    print(
        f"Elements : "
        f"{total_elements:,}"
    )


    print(
        "\nPhysical groups:"
    )


    for dim, tag in (
        gmsh.model.getPhysicalGroups()
    ):

        name = (
            gmsh.model.getPhysicalName(
                dim,
                tag
            )
        )

        print(
            f"  Dimension={dim}, "
            f"Attribute={tag}, "
            f"Name={name}"
        )


    # ========================================================
    # WRITE MSH 2.2
    # ========================================================

    print(
        "\nWriting MSH 2.2..."
    )


    gmsh.option.setNumber(
        "Mesh.MshFileVersion",
        2.2
    )

    gmsh.option.setNumber(
        "Mesh.Binary",
        0
    )


    gmsh.write(
        OUTPUT_MESH
    )


    print(
        "\n" + "=" * 80
    )

    print(
        "ULTRA-REDUCED PALACE MESH CREATED"
    )

    print(
        "=" * 80
    )

    print(
        "\nMesh:"
    )

    print(
        OUTPUT_MESH
    )


finally:

    gmsh.finalize()

    print(
        "\nGmsh finalized."
    )

ULTRA-REDUCED CONFORMAL PALACE MESH

Target:
  Bulk mesh     : 300 um
  JJ mesh       : 30 um
  Simplification: 3.0 um

Reading GDS...
Loaded 19 cells.
Using top cell: TOP_main_1

Extracting Layer 1 / Datatype 0...
Raw polygons: 2450

Converting to Shapely...
Valid polygons: 2450

Uniting metal geometry...
Original union: Polygon

Simplifying redundant boundary vertices...
Original metal area : 185.750737 mm^2
Simplified area     : 185.752476 mm^2
Area change         : 0.0009 %
Final metal regions: 1

Initializing Gmsh...

Creating silicon...
Creating vacuum...

Creating simplified metal surfaces...
Metal surfaces created: 1

Removing duplicate OCC geometry...

FRAGMENTING SILICON / VACUUM
Fragmentation produced 35 entities.

Removing duplicate geometry...

3D volumes: 2
Silicon volumes: 1
Vacuum volumes: 1
Recovered metal surfaces: 1

Creating physical groups...

Setting global mesh size...

Creating JJ local refinement...
  JJ 1: (-1.1325, 0.0000) mm
  JJ 2: (-0.8675, 0.0000) mm
  JJ

In [ ]:
# ============================================================
# PALACE — ULTRA-REDUCED MESH + 1 EIGENMODE TEST
# ============================================================

import os
import json
import subprocess
import shutil


# ============================================================
# PATHS
# ============================================================

PALACE_DIR = (
    "/content/drive/MyDrive/Palace6/build"
)

PALACE_BIN = os.path.join(
    PALACE_DIR,
    "bin",
    "palace-x86_64.bin"
)

LIB_DIR = os.path.join(
    PALACE_DIR,
    "lib"
)

MESH = (
    "/content/drive/MyDrive/Palace6/"
    "meshes/quantum_chip_mesh_ultra_reduced.msh"
)

CONFIG = (
    "/content/palace_ultra_reduced_test.json"
)

OUTPUT = (
    "/content/palace_ultra_reduced_output"
)


# ============================================================
# CHECK FILES
# ============================================================

print("=" * 80)
print("PALACE — ULTRA-REDUCED MESH + 1 MODE TEST")
print("=" * 80)

if not os.path.exists(MESH):

    raise FileNotFoundError(
        f"Mesh not found:\n{MESH}"
    )

if not os.path.exists(PALACE_BIN):

    raise FileNotFoundError(
        f"Palace executable not found:\n{PALACE_BIN}"
    )


mesh_size_mb = (
    os.path.getsize(MESH)
    / (1024 ** 2)
)

print(
    "\nMesh:"
)

print(
    MESH
)

print(
    f"\nMesh file size: "
    f"{mesh_size_mb:.1f} MB"
)

print(
    "\nMesh statistics:"
)

print(
    "  Nodes    : 58,471"
)

print(
    "  Elements : 414,344"
)


# ============================================================
# CLEAN PREVIOUS OUTPUT
# ============================================================

if os.path.exists(OUTPUT):

    print(
        "\nRemoving previous Palace output..."
    )

    shutil.rmtree(
        OUTPUT
    )

os.makedirs(
    OUTPUT,
    exist_ok=True
)


# ============================================================
# ENVIRONMENT
# ============================================================

env = os.environ.copy()

env["LD_LIBRARY_PATH"] = (
    LIB_DIR
    + ":"
    + env.get(
        "LD_LIBRARY_PATH",
        ""
    )
)

env["OMPI_ALLOW_RUN_AS_ROOT"] = "1"

env[
    "OMPI_ALLOW_RUN_AS_ROOT_CONFIRM"
] = "1"


# ============================================================
# PALACE CONFIGURATION
# ============================================================

palace_config = {

    "Problem": {

        "Type": "Eigenmode",

        "Verbose": 2,

        "Output": OUTPUT
    },


    "Model": {

        "Mesh": MESH,

        # GDS/Gmsh geometry is in mm.
        # Palace physical units are meters.
        "L0": 1.0e-3
    },


    "Domains": {

        "Materials": [

            {
                # Physical volume 1
                # Vacuum
                "Attributes": [1],

                "Permittivity": 1.0,

                "Permeability": 1.0
            },


            {
                # Physical volume 2
                # Silicon
                "Attributes": [2],

                "Permittivity": 11.7,

                "Permeability": 1.0
            }
        ]
    },


    "Boundaries": {

        # Physical surface 10
        # Superconducting metal
        "PEC": {

            "Attributes": [10]
        }
    },


    "Solver": {

        # Lowest-order FEM
        "Order": 1,


        "Eigenmode": {

            # =================================================
            # ONLY ONE MODE
            # =================================================

            "N": 1,

            # Search around 5 GHz
            "Target": 5.0,

            # Relaxed tolerance for sanity test
            "Tol": 1.0e-6,

            "MaxIts": 100
        },


        "Linear": {

            "Type": "BoomerAMG",

            "Tol": 1.0e-6,

            "MaxIts": 50
        }
    }
}


# ============================================================
# WRITE CONFIG
# ============================================================

with open(
    CONFIG,
    "w"
) as f:

    json.dump(
        palace_config,
        f,
        indent=4
    )


print(
    "\nConfiguration:"
)

print(
    CONFIG
)


# ============================================================
# COMMAND
# ============================================================

cmd = [

    "mpirun",

    "-n",
    "1",

    "--allow-run-as-root",

    PALACE_BIN,

    CONFIG
]


print(
    "\n" + "=" * 80
)

print(
    "EXECUTING PALACE"
)

print(
    "=" * 80
)

print(
    "\nCommand:"
)

print(
    " ".join(cmd)
)


# ============================================================
# RUN
# ============================================================

result = subprocess.run(

    cmd,

    text=True,

    capture_output=True,

    env=env
)


# ============================================================
# OUTPUT
# ============================================================

print(
    "\n" + "=" * 80
)

print(
    "PALACE STDOUT"
)

print(
    "=" * 80
)

print(
    result.stdout
)


print(
    "\n" + "=" * 80
)

print(
    "PALACE STDERR"
)

print(
    "=" * 80
)

print(
    result.stderr
)


print(
    "\n" + "=" * 80
)

print(
    "RETURN CODE"
)

print(
    result.returncode
)

print(
    "=" * 80
)


# ============================================================
# RESULT
# ============================================================

if result.returncode == 0:

    print(
        "\n" + "=" * 80
    )

    print(
        "SUCCESS"
    )

    print(
        "=" * 80
    )

    print(
        "\nPalace successfully:"
    )

    print(
        "  ✓ Read the Gmsh mesh"
    )

    print(
        "  ✓ Constructed the MFEM mesh"
    )

    print(
        "  ✓ Applied the PEC boundary"
    )

    print(
        "  ✓ Completed the 1-mode eigenmode test"
    )

    print(
        "\nOutput:"
    )

    print(
        OUTPUT
    )

else:

    print(
        "\n" + "=" * 80
    )

    print(
        "PALACE TEST FAILED"
    )

    print(
        "=" * 80
    )

    print(
        "\nLast part of Palace STDERR:"
    )

    print(
        result.stderr[-6000:]
    )

PALACE — ULTRA-REDUCED MESH + 1 MODE TEST

Mesh:
/content/drive/MyDrive/Palace6/meshes/quantum_chip_mesh_ultra_reduced.msh

Mesh file size: 16.8 MB

Mesh statistics:
  Nodes    : 58,471
  Elements : 414,344

Configuration:
/content/palace_ultra_reduced_test.json

EXECUTING PALACE

Command:
mpirun -n 1 --allow-run-as-root /content/drive/MyDrive/Palace6/build/bin/palace-x86_64.bin /content/palace_ultra_reduced_test.json

PALACE STDOUT

_____________     _______
_____   __   \____ __   /____ ____________
____   /_/  /  __ ` /  /  __ ` /  ___/  _ \
___   _____/  /_/  /  /  /_/  /  /__/  ___/
  /__/     \___,__/__/\___,__/\_____\_____/

Git changeset ID: v0.17.0-194-g01f5d9bd1
Running with 1 MPI process, 1 OpenMP thread
Device configuration: omp,cpu
Memory configuration: host-std
libCEED backend: /cpu/self/xsmm/blocked


--> Warning!
11764 mesh faces with no associated boundary element exist on the domain boundary!

Added 1221 elements in 2 iterations of local bisection for under-resolved i

In [ ]:
import psutil

ram = psutil.virtual_memory()

print(f"Total RAM     : {ram.total / 1024**3:.2f} GB")
print(f"Available RAM : {ram.available / 1024**3:.2f} GB")
print(f"Used RAM      : {ram.used / 1024**3:.2f} GB")

Total RAM     : 12.67 GB
Available RAM : 11.18 GB
Used RAM      : 1.18 GB


In [ ]:
import shutil
import os

# ==========================================
# STEP 20: Permanent Google Drive Archival (Directory)
# ==========================================
source_dir = "/content/palace_ultra_reduced_output"
destination_dir = "/content/drive/MyDrive/Palace6/palace_ultra_reduced_output"

print(f"Beginning permanent archival directory to Google Drive: {destination_dir}\n")

if os.path.exists(source_dir):
    # dirs_exist_ok=True prevents errors if the folder already exists in Drive
    shutil.copytree(source_dir, destination_dir, dirs_exist_ok=True)
    print(f"[SAVED] Directory successfully archived to {destination_dir}")
else:
    print(f"[SKIPPED] Directory not found locally: {source_dir}")

Beginning permanent archival directory to Google Drive: /content/drive/MyDrive/Palace6/palace_ultra_reduced_output

[SAVED] Directory successfully archived to /content/drive/MyDrive/Palace6/palace_ultra_reduced_output


In [ ]:
# ============================================================
# COMPLETE 4-QUBIT PALACE EIGENMODE RUN
#
# Current geometry:
#   - 4 concentric transmons
#   - 8 JJ locations geometrically present
#   - 4 qubit-qubit couplers
#   - 4 readout resonators/feedlines
#   - 4 readout finger capacitors
#   - 4 XY drive lines
#   - 4 flux-bias lines
#   - ground plane
#   - silicon substrate
#   - vacuum
#
# IMPORTANT:
#   JJs are currently PEC geometry only.
#   Josephson inductance is NOT included yet.
#
# Purpose:
#   Obtain the first 10 electromagnetic eigenmodes
#   of the complete chip geometry.
# ============================================================

import os
import json
import subprocess
import shutil
import psutil
import time


# ============================================================
# PATHS
# ============================================================

PALACE_DIR = (
    "/content/drive/MyDrive/Palace6/build"
)

PALACE_BIN = os.path.join(
    PALACE_DIR,
    "bin",
    "palace-x86_64.bin"
)

LIB_DIR = os.path.join(
    PALACE_DIR,
    "lib"
)

MESH = (
    "/content/drive/MyDrive/Palace6/"
    "meshes/quantum_chip_mesh_ultra_reduced.msh"
)


CONFIG = (
     "/content/drive/MyDrive/Palace6/mode10/palace_complete_4qubit.json"
)

OUTPUT = (
     "/content/drive/MyDrive/Palace6/mode10/palace_complete_4qubit_output"
)


# ============================================================
# SIMULATION PARAMETERS
# ============================================================

NUMBER_OF_MODES = 10

TARGET_GHZ = 5.0

FEM_ORDER = 1

LINEAR_TOL = 1.0e-8

LINEAR_MAX_ITS = 300

EIGEN_TOL = 1.0e-6

EIGEN_MAX_ITS = 200


# ============================================================
# HEADER
# ============================================================

print("=" * 80)
print("COMPLETE 4-QUBIT PALACE EIGENMODE RUN")
print("=" * 80)


# ============================================================
# RAM CHECK
# ============================================================

ram = psutil.virtual_memory()

print("\nCurrent Colab RAM:")

print(
    f"  Total     : "
    f"{ram.total / 1024**3:.2f} GB"
)

print(
    f"  Available : "
    f"{ram.available / 1024**3:.2f} GB"
)

print(
    f"  Used      : "
    f"{ram.used / 1024**3:.2f} GB"
)


# ============================================================
# FILE CHECKS
# ============================================================

print("\nChecking files...")


if not os.path.isfile(MESH):

    raise FileNotFoundError(
        f"\nMesh not found:\n{MESH}"
    )


if not os.path.isfile(PALACE_BIN):

    raise FileNotFoundError(
        f"\nPalace executable not found:\n{PALACE_BIN}"
    )


mesh_size_mb = (
    os.path.getsize(MESH)
    / (1024 ** 2)
)


print(
    f"Mesh size: "
    f"{mesh_size_mb:.2f} MB"
)

print(
    "Mesh elements: ~414,344"
)

print(
    "Mesh nodes: ~58,471"
)


# ============================================================
# CLEAN OUTPUT
# ============================================================

if os.path.exists(OUTPUT):

    print(
        "\nRemoving previous output directory..."
    )

    shutil.rmtree(
        OUTPUT
    )


os.makedirs(
    OUTPUT,
    exist_ok=True
)


# ============================================================
# ENVIRONMENT
# ============================================================

env = os.environ.copy()

env["LD_LIBRARY_PATH"] = (
    LIB_DIR
    + ":"
    + env.get(
        "LD_LIBRARY_PATH",
        ""
    )
)

env["OMPI_ALLOW_RUN_AS_ROOT"] = "1"

env[
    "OMPI_ALLOW_RUN_AS_ROOT_CONFIRM"
] = "1"


# ============================================================
# PALACE CONFIGURATION
# ============================================================

palace_config = {

    # --------------------------------------------------------
    # PROBLEM
    # --------------------------------------------------------

    "Problem": {

        "Type": "Eigenmode",

        "Verbose": 2,

        "Output": OUTPUT
    },


    # --------------------------------------------------------
    # MODEL
    # --------------------------------------------------------

    "Model": {

        "Mesh": MESH,

        # GDS/Gmsh coordinates are mm.
        # Convert to SI meters.
        "L0": 1.0e-3
    },


    # --------------------------------------------------------
    # MATERIALS
    # --------------------------------------------------------

    "Domains": {

        "Materials": [

            {
                # Attribute 1
                # Vacuum

                "Attributes": [1],

                "Permittivity": 1.0,

                "Permeability": 1.0
            },


            {
                # Attribute 2
                # Silicon

                "Attributes": [2],

                "Permittivity": 11.7,

                "Permeability": 1.0
            }
        ]
    },


    # --------------------------------------------------------
    # BOUNDARY CONDITIONS
    # --------------------------------------------------------

    "Boundaries": {

        # Attribute 10
        # Superconducting metal

        "PEC": {

            "Attributes": [10]
        }
    },


    # --------------------------------------------------------
    # SOLVER
    # --------------------------------------------------------

    "Solver": {

        # Keep Order 1 for Colab RAM.
        "Order": FEM_ORDER,


        # ----------------------------------------------------
        # EIGENMODE SOLVER
        # ----------------------------------------------------

        "Eigenmode": {

            # Request 10 modes.
            "N": NUMBER_OF_MODES,

            # Search around 5 GHz.
            "Target": TARGET_GHZ,

            # Eigenvalue tolerance.
            "Tol": EIGEN_TOL,

            "MaxIts": EIGEN_MAX_ITS
        },


        # ----------------------------------------------------
        # LINEAR SOLVER
        # ----------------------------------------------------

        "Linear": {

            "Type": "BoomerAMG",

            # Better than previous 1e-6.
            "Tol": LINEAR_TOL,

            # Previous 50 iterations was insufficient.
            "MaxIts": LINEAR_MAX_ITS
        }
    }
}


# ============================================================
# WRITE CONFIG
# ============================================================

with open(
    CONFIG,
    "w"
) as f:

    json.dump(
        palace_config,
        f,
        indent=4
    )


print(
    "\nConfiguration:"
)

print(
    CONFIG
)


# ============================================================
# PRINT SIMULATION SUMMARY
# ============================================================

print(
    "\n" + "=" * 80
)

print(
    "SIMULATION SETTINGS"
)

print(
    "=" * 80
)

print(
    f"Modes requested : {NUMBER_OF_MODES}"
)

print(
    f"Target          : {TARGET_GHZ} GHz"
)

print(
    f"FEM order       : {FEM_ORDER}"
)

print(
    f"Eigen tolerance  : {EIGEN_TOL}"
)

print(
    f"Linear tolerance : {LINEAR_TOL}"
)

print(
    f"Linear max its   : {LINEAR_MAX_ITS}"
)

print(
    "\nPhysics:"
)

print(
    "  Superconducting metal = PEC"
)

print(
    "  Josephson inductance  = NOT included"
)

print(
    "  Readout/drive/flux lines = passive geometry"
)

print(
    "  Qubit couplers = passive geometry"
)


# ============================================================
# COMMAND
# ============================================================

cmd = [

    "mpirun",

    "-n",
    "1",

    "--allow-run-as-root",

    PALACE_BIN,

    CONFIG
]


print(
    "\n" + "=" * 80
)

print(
    "EXECUTING PALACE"
)

print(
    "=" * 80
)

print(
    "\n"
    + " ".join(cmd)
)


# ============================================================
# START TIME
# ============================================================

start_time = time.time()


# ============================================================
# RUN
# ============================================================

try:

    result = subprocess.run(

        cmd,

        text=True,

        capture_output=True,

        env=env
    )

except Exception as exc:

    print(
        "\nFailed to start Palace:"
    )

    print(exc)

    raise


elapsed = (
    time.time()
    - start_time
)


# ============================================================
# OUTPUT
# ============================================================

print(
    "\n" + "=" * 80
)

print(
    "PALACE STDOUT"
)

print(
    "=" * 80
)

print(
    result.stdout
)


print(
    "\n" + "=" * 80
)

print(
    "PALACE STDERR"
)

print(
    "=" * 80
)

print(
    result.stderr
)


# ============================================================
# FINAL RAM
# ============================================================

ram = psutil.virtual_memory()

print(
    "\n" + "=" * 80
)

print(
    "RUN SUMMARY"
)

print(
    "=" * 80
)

print(
    f"Return code : {result.returncode}"
)

print(
    f"Runtime     : "
    f"{elapsed / 60:.2f} minutes"
)

print(
    f"RAM used    : "
    f"{ram.used / 1024**3:.2f} GB"
)

print(
    f"RAM free    : "
    f"{ram.available / 1024**3:.2f} GB"
)


# ============================================================
# SUCCESS / FAILURE
# ============================================================

if result.returncode == 0:

    print(
        "\n" + "=" * 80
    )

    print(
        "PALACE 10-MODE RUN COMPLETED"
    )

    print(
        "=" * 80
    )

    print(
        "\nOutput directory:"
    )

    print(
        OUTPUT
    )

    print(
        "\nNext:"
    )

    print(
        "Extract the 10 eigenfrequencies and identify"
        " qubit/readout/coupler-like modes."
    )

else:

    print(
        "\n" + "=" * 80
    )

    print(
        "PALACE RUN FAILED"
    )

    print(
        "=" * 80
    )

    print(
        "\nLast 8000 characters of STDERR:"
    )

    print(
        result.stderr[-8000:]
    )

    raise RuntimeError(
        "Palace eigenmode simulation failed."
    )

Streaming output truncated to the last 5000 lines.
  5 (restart 0) KSP residual norm 3.805993e-07
  6 (restart 0) KSP residual norm 3.634225e-07
  7 (restart 0) KSP residual norm 3.455362e-07
  8 (restart 0) KSP residual norm 3.302723e-07
  9 (restart 0) KSP residual norm 3.163363e-07
 10 (restart 0) KSP residual norm 3.075676e-07
 11 (restart 0) KSP residual norm 3.008833e-07
 12 (restart 0) KSP residual norm 2.966287e-07
 13 (restart 0) KSP residual norm 2.938911e-07
 14 (restart 0) KSP residual norm 2.934442e-07
 15 (restart 0) KSP residual norm 2.932655e-07
 16 (restart 0) KSP residual norm 2.912553e-07
 17 (restart 0) KSP residual norm 2.882348e-07
 18 (restart 0) KSP residual norm 2.844619e-07
 19 (restart 0) KSP residual norm 2.827634e-07
 20 (restart 0) KSP residual norm 2.824095e-07
 21 (restart 0) KSP residual norm 2.819543e-07
 22 (restart 0) KSP residual norm 2.802747e-07
 23 (restart 0) KSP residual norm 2.775972e-07
 24 (restart 0) KSP residual norm 2.758554e-07
 25 (rest

In [ ]:
!pip install gdspy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.9/157.9 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for gdspy: filename=gdspy-1.6.13-cp312-cp312-linux_x86_64.whl size=595857 sha256=bc69ab86dce8f34e5d21575fa2cb8206d26ea32dbd264ac81404861438d03141
  Stored in directory: /root/.cache/pip/wheels/02/05/a3/c4b581f8330cedff4d7f4aa7134ae298e88232ccce6a2d4859
Successfully built gdspy


In [ ]:
# ============================================================
# 8-JJ PALACE MESH
#
# Creates:
#
#   Attribute 10 : superconducting PEC metal
#   Attribute 20 : Q1_JJ1
#   Attribute 21 : Q1_JJ2
#   Attribute 22 : Q2_JJ1
#   Attribute 23 : Q2_JJ2
#   Attribute 24 : Q3_JJ1
#   Attribute 25 : Q3_JJ2
#   Attribute 26 : Q4_JJ1
#   Attribute 27 : Q4_JJ2
#
# The JJ attributes are subsequently used by Palace
# LumpedPort elements.
# ============================================================

import os
import gdspy
import gmsh

from shapely.geometry import Polygon
from shapely.ops import unary_union


# ============================================================
# PATHS
# ============================================================

GDS_PATH = "/content/drive/MyDrive/Palace6/meshes/quantum_chip_export_manuscript.gds"

OUTPUT_MESH = (
    "/content/drive/MyDrive/Palace6/meshes/"
    "quantum_chip_mesh_8JJ.msh"
)

TOP_CELL_NAME = "TOP_main_1"


# ============================================================
# GDS METAL
# ============================================================

METAL_LAYER = 1
METAL_DATATYPE = 0


# ============================================================
# CHIP
# ============================================================

CHIP_X = 14.0
CHIP_Y = 14.0

SUBSTRATE_THICKNESS = 0.5
AIR_HEIGHT = 0.5


# ============================================================
# PHYSICAL ATTRIBUTES
# ============================================================

VACUUM_ID = 1
SILICON_ID = 2

PEC_ID = 10

JJ_ATTRIBUTE_START = 20


# ============================================================
# MESH
# ============================================================

BULK_SIZE = 0.300       # 300 um
JJ_SIZE = 0.020         # 20 um

# JJ port dimensions.
#
# The actual JJ element should be small compared with
# the electromagnetic wavelength.
#
# These are NOT the physical oxide dimensions.
# They define the lumped-port integration surface.
JJ_W = 0.010            # 10 um
JJ_H = 0.006            # 6 um


# ============================================================
# EXACT JJ LOCATIONS FROM YOUR GDS
# ============================================================

JJ_LOCATIONS = [

    # Q1
    (-1.1325,  0.0000),
    (-0.8675,  0.0000),

    # Q2
    ( 1.2675,  0.0000),
    ( 1.5325,  0.0000),

    # Q3
    (-1.1325, -2.7000),
    (-0.8675, -2.7000),

    # Q4
    ( 1.2675, -2.7000),
    ( 1.5325, -2.7000)
]


# ============================================================
# HEADER
# ============================================================

print("=" * 80)
print("8-JJ PALACE MESH GENERATOR")
print("=" * 80)

print("\nJJ locations:")

for i, (x, y) in enumerate(JJ_LOCATIONS):

    print(
        f"  JJ{i+1}: "
        f"({x:.4f}, {y:.4f}) mm"
    )


# ============================================================
# READ GDS
# ============================================================

print("\nReading GDS...")

lib = gdspy.GdsLibrary()

lib.read_gds(
    GDS_PATH
)

print(
    f"Loaded {len(lib.cells)} cells."
)

top = lib.cells[
    TOP_CELL_NAME
]

print(
    f"Using top cell: {TOP_CELL_NAME}"
)


# ============================================================
# EXTRACT METAL
# ============================================================

print(
    "\nExtracting Layer 1 / Datatype 0..."
)

polygons_by_spec = (
    top.get_polygons(
        by_spec=True
    )
)

raw_polygons = []


for spec, polygons in polygons_by_spec.items():

    layer, datatype = spec

    if layer != METAL_LAYER:
        continue

    if datatype != METAL_DATATYPE:
        continue

    for polygon in polygons:

        if len(polygon) < 3:
            continue

        pts = []

        for p in polygon:

            point = (
                float(p[0]),
                float(p[1])
            )

            if (
                not pts
                or point != pts[-1]
            ):

                pts.append(point)


        if (
            len(pts) > 1
            and pts[0] == pts[-1]
        ):

            pts.pop()


        if len(set(pts)) >= 3:

            raw_polygons.append(
                pts
            )


print(
    f"Raw metal polygons: "
    f"{len(raw_polygons)}"
)


# ============================================================
# SHAPELY CLEANUP
# ============================================================

print(
    "\nCleaning metal geometry..."
)

shapes = []


for pts in raw_polygons:

    try:

        poly = Polygon(pts)

        if not poly.is_valid:

            poly = poly.buffer(0)

        if poly.is_empty:

            continue

        if poly.area <= 1e-12:

            continue

        shapes.append(poly)

    except Exception:

        pass


print(
    f"Valid polygons: "
    f"{len(shapes)}"
)


# ============================================================
# UNION
# ============================================================

print(
    "\nUniting metal polygons..."
)

metal_union = unary_union(
    shapes
)


if not metal_union.is_valid:

    metal_union = metal_union.buffer(0)


print(
    "Metal union:",
    metal_union.geom_type
)


# ============================================================
# SIMPLIFY
# ============================================================

print(
    "\nSimplifying metal boundary..."
)

metal_union = metal_union.simplify(
    0.003,
    preserve_topology=True
)

if not metal_union.is_valid:

    metal_union = metal_union.buffer(0)


# ============================================================
# GMSH
# ============================================================

print(
    "\nInitializing Gmsh..."
)

gmsh.initialize()

gmsh.option.setNumber(
    "General.Terminal",
    1
)

gmsh.model.add(
    "four_qubit_8JJ"
)


try:

    # ========================================================
    # SUBSTRATE
    # ========================================================

    print(
        "\nCreating silicon..."
    )

    silicon = gmsh.model.occ.addBox(

        -CHIP_X / 2,
        -CHIP_Y / 2,
        -SUBSTRATE_THICKNESS,

        CHIP_X,
        CHIP_Y,
        SUBSTRATE_THICKNESS
    )


    # ========================================================
    # VACUUM
    # ========================================================

    print(
        "Creating vacuum..."
    )

    vacuum = gmsh.model.occ.addBox(

        -CHIP_X / 2,
        -CHIP_Y / 2,
        0,

        CHIP_X,
        CHIP_Y,
        AIR_HEIGHT
    )


    # ========================================================
    # METAL POLYGON
    # ========================================================

    if metal_union.geom_type == "Polygon":

        metal_polys = [
            metal_union
        ]

    elif metal_union.geom_type == "MultiPolygon":

        metal_polys = list(
            metal_union.geoms
        )

    else:

        raise RuntimeError(
            "Unexpected metal geometry type."
        )


    print(
        f"\nCreating "
        f"{len(metal_polys)} metal surface(s)..."
    )


    # ========================================================
    # POINT CACHE
    # ========================================================

    point_cache = {}


    def point(x, y, z=0.0):

        key = (
            round(float(x), 9),
            round(float(y), 9),
            round(float(z), 9)
        )

        if key not in point_cache:

            point_cache[key] = (
                gmsh.model.occ.addPoint(
                    key[0],
                    key[1],
                    key[2]
                )
            )

        return point_cache[key]


    # ========================================================
    # CREATE METAL SURFACES
    # ========================================================

    metal_surfaces = []


    for poly in metal_polys:

        coords = list(
            poly.exterior.coords
        )[:-1]


        if len(coords) < 3:

            continue


        pts = [
            point(x, y)
            for x, y in coords
        ]


        lines = []


        for i in range(
            len(pts)
        ):

            p1 = pts[i]

            p2 = pts[
                (i + 1) % len(pts)
            ]

            if p1 != p2:

                lines.append(
                    gmsh.model.occ.addLine(
                        p1,
                        p2
                    )
                )


        if len(lines) < 3:

            continue


        wire = (
            gmsh.model.occ.addWire(
                lines
            )
        )


        holes = []


        for interior in poly.interiors:

            hcoords = list(
                interior.coords
            )[:-1]


            hpts = [
                point(x, y)
                for x, y in hcoords
            ]


            hlines = []


            for i in range(
                len(hpts)
            ):

                hlines.append(
                    gmsh.model.occ.addLine(
                        hpts[i],
                        hpts[
                            (i + 1)
                            % len(hpts)
                        ]
                    )
                )


            if len(hlines) >= 3:

                holes.append(
                    gmsh.model.occ.addWire(
                        hlines
                    )
                )


        surface = (
            gmsh.model.occ.addPlaneSurface(
                [wire] + holes
            )
        )


        metal_surfaces.append(
            surface
        )


    # ========================================================
    # JJ PORT SURFACES
    # ========================================================

    print(
        "\nCreating 8 JJ port surfaces..."
    )


    jj_surfaces = []


    for i, (x, y) in enumerate(
        JJ_LOCATIONS
    ):

        x1 = x - JJ_W / 2
        x2 = x + JJ_W / 2

        y1 = y - JJ_H / 2
        y2 = y + JJ_H / 2


        p1 = point(x1, y1)
        p2 = point(x2, y1)
        p3 = point(x2, y2)
        p4 = point(x1, y2)


        l1 = gmsh.model.occ.addLine(
            p1, p2
        )

        l2 = gmsh.model.occ.addLine(
            p2, p3
        )

        l3 = gmsh.model.occ.addLine(
            p3, p4
        )

        l4 = gmsh.model.occ.addLine(
            p4, p1
        )


        wire = gmsh.model.occ.addWire(
            [l1, l2, l3, l4]
        )


        surf = (
            gmsh.model.occ.addPlaneSurface(
                [wire]
            )
        )


        jj_surfaces.append(
            surf
        )


        print(
            f"  JJ{i+1}: "
            f"surface={surf}"
        )


    # ========================================================
    # SYNCHRONIZE
    # ========================================================

    gmsh.model.occ.synchronize()


    # ========================================================
    # REMOVE DUPLICATES
    # ========================================================

    print(
        "\nRemoving duplicate geometry..."
    )

    gmsh.model.occ.removeAllDuplicates()

    gmsh.model.occ.synchronize()


    # ========================================================
    # FRAGMENT VOLUMES
    # ========================================================

    print(
        "\nFragmenting silicon/vacuum..."
    )


    fragmented, mapping = (
        gmsh.model.occ.fragment(

            [
                (3, silicon),
                (3, vacuum)
            ],

            [
                (2, s)
                for s in metal_surfaces
            ]
        )
    )


    gmsh.model.occ.synchronize()

    gmsh.model.occ.removeAllDuplicates()

    gmsh.model.occ.synchronize()


    # ========================================================
    # FIND VOLUMES
    # ========================================================

    volumes = (
        gmsh.model.getEntities(3)
    )


    silicon_volumes = []
    vacuum_volumes = []


    for dim, tag in volumes:

        com = (
            gmsh.model.occ.getCenterOfMass(
                dim,
                tag
            )
        )


        if com[2] < 0:

            silicon_volumes.append(tag)

        elif com[2] > 0:

            vacuum_volumes.append(tag)


    print(
        "\nVolumes:",
        len(volumes)
    )

    print(
        "Silicon:",
        silicon_volumes
    )

    print(
        "Vacuum:",
        vacuum_volumes
    )


    # ========================================================
    # RECOVER METAL
    # ========================================================

    metal_faces = []


    for dim, tag in (
        gmsh.model.getEntities(2)
    ):

        bbox = (
            gmsh.model.getBoundingBox(
                dim,
                tag
            )
        )


        # Metal lies at z = 0.
        zmin = bbox[2]
        zmax = bbox[5]


        if (
            abs(zmin) < 1e-8
            and abs(zmax) < 1e-8
        ):

            metal_faces.append(
                tag
            )


    # ========================================================
    # PHYSICAL GROUPS
    # ========================================================

    print(
        "\nCreating physical groups..."
    )


    gmsh.model.addPhysicalGroup(
        3,
        vacuum_volumes,
        VACUUM_ID
    )

    gmsh.model.setPhysicalName(
        3,
        VACUUM_ID,
        "Vacuum"
    )


    gmsh.model.addPhysicalGroup(
        3,
        silicon_volumes,
        SILICON_ID
    )

    gmsh.model.setPhysicalName(
        3,
        SILICON_ID,
        "Silicon_Substrate"
    )


    # --------------------------------------------------------
    # PEC
    # --------------------------------------------------------

    gmsh.model.addPhysicalGroup(
        2,
        metal_faces,
        PEC_ID
    )

    gmsh.model.setPhysicalName(
        2,
        PEC_ID,
        "Superconducting_Metal"
    )


    # --------------------------------------------------------
    # JJ ATTRIBUTES
    # --------------------------------------------------------

    for i, surf in enumerate(
        jj_surfaces
    ):

        attr = (
            JJ_ATTRIBUTE_START
            + i
        )


        gmsh.model.addPhysicalGroup(
            2,
            [surf],
            attr
        )


        gmsh.model.setPhysicalName(
            2,
            attr,
            f"JJ{i+1}"
        )


        print(
            f"  Attribute {attr}: "
            f"JJ{i+1}"
        )


    # ========================================================
    # MESH
    # ========================================================

    print(
        "\nConfiguring mesh..."
    )


    gmsh.option.setNumber(
        "Mesh.CharacteristicLengthMin",
        BULK_SIZE
    )

    gmsh.option.setNumber(
        "Mesh.CharacteristicLengthMax",
        BULK_SIZE
    )


    # ========================================================
    # JJ REFINEMENT
    # ========================================================

    print(
        "\nApplying local JJ refinement..."
    )


    fields = []


    for i, (x, y) in enumerate(
        JJ_LOCATIONS
    ):

        f = (
            gmsh.model.mesh.field.add(
                "Box"
            )
        )


        gmsh.model.mesh.field.setNumber(
            f,
            "VIn",
            JJ_SIZE
        )

        gmsh.model.mesh.field.setNumber(
            f,
            "VOut",
            BULK_SIZE
        )


        gmsh.model.mesh.field.setNumber(
            f,
            "XMin",
            x - 0.100
        )

        gmsh.model.mesh.field.setNumber(
            f,
            "XMax",
            x + 0.100
        )

        gmsh.model.mesh.field.setNumber(
            f,
            "YMin",
            y - 0.100
        )

        gmsh.model.mesh.field.setNumber(
            f,
            "YMax",
            y + 0.100
        )

        gmsh.model.mesh.field.setNumber(
            f,
            "ZMin",
            -SUBSTRATE_THICKNESS
        )

        gmsh.model.mesh.field.setNumber(
            f,
            "ZMax",
            AIR_HEIGHT
        )


        fields.append(f)


    if fields:

        fmin = (
            gmsh.model.mesh.field.add(
                "Min"
            )
        )


        gmsh.model.mesh.field.setNumbers(
            fmin,
            "FieldsList",
            fields
        )


        gmsh.model.mesh.field.setAsBackgroundMesh(
            fmin
        )


    # ========================================================
    # 3D MESH
    # ========================================================

    print(
        "\nGenerating 3D mesh..."
    )


    gmsh.option.setNumber(
        "Mesh.Algorithm3D",
        1
    )

    gmsh.option.setNumber(
        "Mesh.Optimize",
        0
    )

    gmsh.option.setNumber(
        "Mesh.OptimizeNetgen",
        0
    )


    gmsh.model.mesh.generate(
        3
    )


    # ========================================================
    # CLEAN
    # ========================================================

    gmsh.model.mesh.removeDuplicateNodes()

    gmsh.model.mesh.removeDuplicateElements()


    # ========================================================
    # STATISTICS
    # ========================================================

    nodes, coords, _ = (
        gmsh.model.mesh.getNodes()
    )


    types, tags, conn = (
        gmsh.model.mesh.getElements()
    )


    total = sum(
        len(t)
        for t in tags
    )


    print(
        "\n" + "=" * 80
    )

    print(
        "8-JJ MESH STATISTICS"
    )

    print(
        "=" * 80
    )

    print(
        f"Nodes    : "
        f"{len(nodes):,}"
    )

    print(
        f"Elements : "
        f"{total:,}"
    )


    print(
        "\nPhysical groups:"
    )


    for dim, tag in (
        gmsh.model.getPhysicalGroups()
    ):

        name = (
            gmsh.model.getPhysicalName(
                dim,
                tag
            )
        )

        print(
            f"  Dimension={dim}, "
            f"Attribute={tag}, "
            f"Name={name}"
        )


    # ========================================================
    # WRITE
    # ========================================================

    print(
        "\nWriting MSH 2.2..."
    )


    gmsh.option.setNumber(
        "Mesh.MshFileVersion",
        2.2
    )

    gmsh.option.setNumber(
        "Mesh.Binary",
        0
    )


    gmsh.write(
        OUTPUT_MESH
    )


    print(
        "\n" + "=" * 80
    )

    print(
        "8-JJ PALACE MESH CREATED"
    )

    print(
        "=" * 80
    )

    print(
        "\nMesh:"
    )

    print(
        OUTPUT_MESH
    )


finally:

    gmsh.finalize()

    print(
        "\nGmsh finalized."
    )

8-JJ PALACE MESH GENERATOR

JJ locations:
  JJ1: (-1.1325, 0.0000) mm
  JJ2: (-0.8675, 0.0000) mm
  JJ3: (1.2675, 0.0000) mm
  JJ4: (1.5325, 0.0000) mm
  JJ5: (-1.1325, -2.7000) mm
  JJ6: (-0.8675, -2.7000) mm
  JJ7: (1.2675, -2.7000) mm
  JJ8: (1.5325, -2.7000) mm

Reading GDS...
Loaded 19 cells.
Using top cell: TOP_main_1

Extracting Layer 1 / Datatype 0...
Raw metal polygons: 2450

Cleaning metal geometry...
Valid polygons: 2450

Uniting metal polygons...
Metal union: Polygon

Simplifying metal boundary...

Initializing Gmsh...

Creating silicon...
Creating vacuum...

Creating 1 metal surface(s)...

Creating 8 JJ port surfaces...
  JJ1: surface=14
  JJ2: surface=15
  JJ3: surface=16
  JJ4: surface=17
  JJ5: surface=18
  JJ6: surface=19
  JJ7: surface=20
  JJ8: surface=21

Removing duplicate geometry...

Fragmenting silicon/vacuum...

Volumes: 2
Silicon: [1]
Vacuum: [2]

Creating physical groups...
  Attribute 20: JJ1
  Attribute 21: JJ2
  Attribute 22: JJ3
  Attribute 23: JJ4
  Attr

In [ ]:
# ============================================================
# CHECK 8-JJ MESH PHYSICAL GROUPS
# ============================================================

import gmsh
import os

MESH = (
    "/content/drive/MyDrive/Palace6/meshes/"
    "quantum_chip_mesh_8JJ.msh"
)

print("=" * 80)
print("CHECKING 8-JJ PALACE MESH")
print("=" * 80)

print("\nMesh:")
print(MESH)

if not os.path.exists(MESH):
    raise FileNotFoundError(MESH)

gmsh.initialize()

try:

    gmsh.option.setNumber(
        "General.Terminal",
        1
    )

    gmsh.open(MESH)

    print("\nPhysical groups found:")

    groups = gmsh.model.getPhysicalGroups()

    if not groups:
        print("WARNING: No physical groups found!")

    for dim, tag in groups:

        name = gmsh.model.getPhysicalName(
            dim,
            tag
        )

        entities = gmsh.model.getEntitiesForPhysicalGroup(
            dim,
            tag
        )

        print(
            f"  Dimension={dim:1d}, "
            f"Attribute={tag:2d}, "
            f"Name={name:25s}, "
            f"Entities={len(entities)}"
        )

    print("\nRequired groups:")

    required = {
        1: "Vacuum",
        2: "Silicon_Substrate",
        10: "Superconducting_Metal",
        20: "JJ1",
        21: "JJ2",
        22: "JJ3",
        23: "JJ4",
        24: "JJ5",
        25: "JJ6",
        26: "JJ7",
        27: "JJ8",
    }

    found = {
        tag: name
        for dim, tag in groups
    }

    for tag, expected in required.items():

        if tag in found:

            print(
                f"  PASS  {tag:2d} -> "
                f"{found[tag]}"
            )

        else:

            print(
                f"  FAIL  {tag:2d} -> "
                f"{expected} MISSING"
            )

    # --------------------------------------------------------
    # Check element counts by physical group
    # --------------------------------------------------------

    print(
        "\nPhysical-group element counts:"
    )

    for dim, tag in groups:

        name = gmsh.model.getPhysicalName(
            dim,
            tag
        )

        entities = (
            gmsh.model.getEntitiesForPhysicalGroup(
                dim,
                tag
            )
        )

        total = 0

        for entity_tag in entities:

            types, elem_tags, _ = (
                gmsh.model.mesh.getElements(
                    dim,
                    entity_tag
                )
            )

            for etags in elem_tags:
                total += len(etags)

        print(
            f"  {tag:2d} "
            f"{name:25s} "
            f"{total:,} elements"
        )

finally:

    gmsh.finalize()

print("\nDone.")

CHECKING 8-JJ PALACE MESH

Mesh:
/content/drive/MyDrive/Palace6/meshes/quantum_chip_mesh_8JJ.msh

Physical groups found:
  Dimension=2, Attribute=20, Name=JJ1                      , Entities=1
  Dimension=2, Attribute=21, Name=JJ2                      , Entities=1
  Dimension=2, Attribute=22, Name=JJ3                      , Entities=1
  Dimension=2, Attribute=23, Name=JJ4                      , Entities=1
  Dimension=2, Attribute=24, Name=JJ5                      , Entities=1
  Dimension=2, Attribute=25, Name=JJ6                      , Entities=1
  Dimension=2, Attribute=26, Name=JJ7                      , Entities=1
  Dimension=2, Attribute=27, Name=JJ8                      , Entities=1
  Dimension=3, Attribute= 1, Name=Vacuum                   , Entities=1
  Dimension=3, Attribute= 2, Name=Silicon_Substrate        , Entities=1

Required groups:
  PASS   1 -> Silicon_Substrate
  PASS   2 -> Silicon_Substrate
  FAIL  10 -> Superconducting_Metal MISSING
  PASS  20 -> Silicon_Substrate
 

In [ ]:
# function KeepAlive() {
#     console.log("Colab Keep-Alive Triggered");
#     document.querySelector("colab-connect-button")?.shadowRoot.querySelector("#connect")?.click();
# }
# setInterval(KeepAlive, 60000);

In [ ]:
# ============================================================
# ROBUST 8-JJ PALACE MESH
#
# Uses:
#   Layer 1 / Datatype 0  -> superconducting metal
#   Layer 53 / Datatype 0 -> JJ geometry reference
#
# Physical groups:
#
#   1  = Vacuum
#   2  = Silicon_Substrate
#   10 = Superconducting_Metal
#   20 = JJ1
#   21 = JJ2
#   22 = JJ3
#   23 = JJ4
#   24 = JJ5
#   25 = JJ6
#   26 = JJ7
#   27 = JJ8
#
# ============================================================

import os
import gdspy
import gmsh

from shapely.geometry import Polygon
from shapely.ops import unary_union


# ============================================================
# PATHS
# ============================================================

GDS_PATH = "/content/drive/MyDrive/Palace6/meshes/quantum_chip_export_manuscript.gds"

OUTPUT_MESH = (
    "/content/drive/MyDrive/Palace6/meshes/"
    "quantum_chip_mesh_8JJ_v2.msh"
)

TOP_CELL_NAME = "TOP_main_1"


# ============================================================
# GDS LAYERS
# ============================================================

METAL_LAYER = 1
METAL_DATATYPE = 0

JJ_LAYER = 53
JJ_DATATYPE = 0


# ============================================================
# CHIP
# ============================================================

CHIP_X = 14.0
CHIP_Y = 14.0

SUBSTRATE_THICKNESS = 0.5
AIR_HEIGHT = 0.5


# ============================================================
# PHYSICAL ATTRIBUTES
# ============================================================

VACUUM_ID = 1
SILICON_ID = 2
PEC_ID = 10

JJ_START_ID = 20


# ============================================================
# MESH
# ============================================================

BULK_SIZE = 0.300
JJ_SIZE = 0.025

JJ_REFINE_HALF = 0.100


# ============================================================
# JJ LOCATIONS
# ============================================================

JJ_LOCATIONS = [

    (-1.1325,  0.0000),
    (-0.8675,  0.0000),

    ( 1.2675,  0.0000),
    ( 1.5325,  0.0000),

    (-1.1325, -2.7000),
    (-0.8675, -2.7000),

    ( 1.2675, -2.7000),
    ( 1.5325, -2.7000)
]


# ============================================================
# HEADER
# ============================================================

print("=" * 80)
print("ROBUST 8-JJ PALACE MESH V2")
print("=" * 80)


# ============================================================
# READ GDS
# ============================================================

print("\nReading GDS...")

lib = gdspy.GdsLibrary()

lib.read_gds(
    GDS_PATH
)

print(
    f"Loaded {len(lib.cells)} cells."
)


if TOP_CELL_NAME not in lib.cells:

    raise RuntimeError(
        f"Missing top cell: {TOP_CELL_NAME}"
    )


top = lib.cells[
    TOP_CELL_NAME
]

print(
    f"Using top cell: {TOP_CELL_NAME}"
)


# ============================================================
# EXTRACT METAL
# ============================================================

print(
    "\nExtracting superconducting metal..."
)

polygons_by_spec = (
    top.get_polygons(
        by_spec=True
    )
)

metal_polygons = []


for spec, polygons in polygons_by_spec.items():

    layer, datatype = spec

    if layer != METAL_LAYER:
        continue

    if datatype != METAL_DATATYPE:
        continue

    for p in polygons:

        if len(p) < 3:
            continue

        pts = [
            (
                float(x),
                float(y)
            )
            for x, y in p
        ]

        if (
            len(pts) > 1
            and pts[0] == pts[-1]
        ):

            pts.pop()

        if len(set(pts)) >= 3:

            metal_polygons.append(
                Polygon(pts)
            )


print(
    f"Metal polygons: "
    f"{len(metal_polygons)}"
)


# ============================================================
# CLEAN METAL
# ============================================================

clean_metal = []


for poly in metal_polygons:

    try:

        if not poly.is_valid:

            poly = poly.buffer(0)

        if (
            not poly.is_empty
            and poly.area > 1e-12
        ):

            clean_metal.append(
                poly
            )

    except Exception:
        pass


print(
    f"Valid metal polygons: "
    f"{len(clean_metal)}"
)


# ============================================================
# UNION METAL
# ============================================================

print(
    "\nUniting metal..."
)

metal_union = unary_union(
    clean_metal
)

if not metal_union.is_valid:

    metal_union = metal_union.buffer(0)


print(
    "Metal type:",
    metal_union.geom_type
)

print(
    f"Metal area: "
    f"{metal_union.area:.6f} mm^2"
)


# ============================================================
# SIMPLIFY
# ============================================================

metal_union = metal_union.simplify(
    0.003,
    preserve_topology=True
)

if not metal_union.is_valid:

    metal_union = metal_union.buffer(0)


# ============================================================
# EXTRACT ACTUAL JJ GEOMETRY
# ============================================================

print(
    "\nExtracting JJ geometry from "
    f"Layer {JJ_LAYER} / Datatype {JJ_DATATYPE}..."
)

jj_polygons = []


for spec, polygons in polygons_by_spec.items():

    layer, datatype = spec

    if layer != JJ_LAYER:
        continue

    if datatype != JJ_DATATYPE:
        continue

    for p in polygons:

        if len(p) < 3:
            continue

        pts = [
            (
                float(x),
                float(y)
            )
            for x, y in p
        ]

        if (
            len(pts) > 1
            and pts[0] == pts[-1]
        ):

            pts.pop()

        if len(set(pts)) >= 3:

            try:

                poly = Polygon(pts)

                if not poly.is_valid:

                    poly = poly.buffer(0)

                if (
                    not poly.is_empty
                    and poly.area > 1e-12
                ):

                    jj_polygons.append(
                        poly
                    )

            except Exception:
                pass


print(
    f"Layer-53 JJ polygons found: "
    f"{len(jj_polygons)}"
)


# ============================================================
# FIND THE JJ POLYGON NEAREST EACH EXPECTED LOCATION
# ============================================================

print(
    "\nMatching actual JJ polygons "
    "to the eight JJ locations..."
)

matched_jj = []


for i, (x, y) in enumerate(
    JJ_LOCATIONS
):

    target_x = x
    target_y = y


    best = None
    best_distance = float("inf")


    for poly in jj_polygons:

        centroid = poly.centroid

        dx = (
            centroid.x
            - target_x
        )

        dy = (
            centroid.y
            - target_y
        )

        distance = (
            dx * dx
            + dy * dy
        ) ** 0.5


        if distance < best_distance:

            best_distance = distance
            best = poly


    if best is None:

        raise RuntimeError(
            f"Could not find JJ{i+1}"
        )


    # 100 um matching tolerance
    if best_distance > 0.100:

        raise RuntimeError(
            f"JJ{i+1} matched too far away: "
            f"{best_distance:.6f} mm"
        )


    matched_jj.append(
        best
    )


    print(
        f"  JJ{i+1}: "
        f"centroid=("
        f"{best.centroid.x:.6f}, "
        f"{best.centroid.y:.6f}) mm, "
        f"distance={best_distance*1000:.3f} um"
    )


# ============================================================
# INITIALIZE GMSH
# ============================================================

print(
    "\nInitializing Gmsh..."
)

gmsh.initialize()

gmsh.option.setNumber(
    "General.Terminal",
    1
)

gmsh.model.add(
    "four_qubit_8JJ_v2"
)


try:

    # ========================================================
    # SUBSTRATE
    # ========================================================

    print(
        "\nCreating silicon..."
    )

    silicon = gmsh.model.occ.addBox(

        -CHIP_X / 2,
        -CHIP_Y / 2,
        -SUBSTRATE_THICKNESS,

        CHIP_X,
        CHIP_Y,
        SUBSTRATE_THICKNESS
    )


    # ========================================================
    # VACUUM
    # ========================================================

    print(
        "Creating vacuum..."
    )

    vacuum = gmsh.model.occ.addBox(

        -CHIP_X / 2,
        -CHIP_Y / 2,
        0.0,

        CHIP_X,
        CHIP_Y,
        AIR_HEIGHT
    )


    # ========================================================
    # POINT CACHE
    # ========================================================

    point_cache = {}


    def get_point(
        x,
        y,
        z=0.0
    ):

        key = (
            round(float(x), 9),
            round(float(y), 9),
            round(float(z), 9)
        )

        if key not in point_cache:

            point_cache[key] = (
                gmsh.model.occ.addPoint(
                    key[0],
                    key[1],
                    key[2]
                )
            )

        return point_cache[key]


    # ========================================================
    # POLYGON → SURFACE
    # ========================================================

    def make_surface(poly):

        exterior = list(
            poly.exterior.coords
        )[:-1]


        if len(exterior) < 3:

            return None


        pts = [
            get_point(x, y)
            for x, y in exterior
        ]


        lines = []


        for i in range(
            len(pts)
        ):

            p1 = pts[i]

            p2 = pts[
                (i + 1) % len(pts)
            ]


            if p1 != p2:

                lines.append(
                    gmsh.model.occ.addLine(
                        p1,
                        p2
                    )
                )


        if len(lines) < 3:

            return None


        wire = (
            gmsh.model.occ.addWire(
                lines
            )
        )


        # Holes
        holes = []


        for interior in poly.interiors:

            coords = list(
                interior.coords
            )[:-1]


            if len(coords) < 3:

                continue


            hp = [
                get_point(x, y)
                for x, y in coords
            ]


            hl = []


            for i in range(
                len(hp)
            ):

                hl.append(
                    gmsh.model.occ.addLine(
                        hp[i],
                        hp[
                            (i + 1)
                            % len(hp)
                        ]
                    )
                )


            if len(hl) >= 3:

                holes.append(
                    gmsh.model.occ.addWire(
                        hl
                    )
                )


        return (
            gmsh.model.occ.addPlaneSurface(
                [wire] + holes
            )
        )


    # ========================================================
    # METAL SURFACE
    # ========================================================

    print(
        "\nCreating metal surface..."
    )


    if metal_union.geom_type == "Polygon":

        metal_regions = [
            metal_union
        ]

    else:

        metal_regions = list(
            metal_union.geoms
        )


    metal_surfaces = []


    for poly in metal_regions:

        s = make_surface(
            poly
        )

        if s is not None:

            metal_surfaces.append(
                s
            )


    print(
        f"Metal surfaces: "
        f"{len(metal_surfaces)}"
    )


    # ========================================================
    # JJ SURFACES
    # ========================================================

    print(
        "\nCreating JJ surfaces..."
    )


    jj_surfaces = []


    for i, poly in enumerate(
        matched_jj
    ):

        s = make_surface(
            poly
        )

        if s is None:

            raise RuntimeError(
                f"Failed JJ{i+1}"
            )


        jj_surfaces.append(
            s
        )


        print(
            f"  JJ{i+1}: "
            f"surface={s}"
        )


    # ========================================================
    # SYNCHRONIZE
    # ========================================================

    gmsh.model.occ.synchronize()


    # ========================================================
    # DO NOT FRAGMENT
    #
    # The metal/JJ surfaces are deliberately retained.
    # ========================================================

    print(
        "\nKeeping metal/JJ surfaces "
        "without OCC volume fragmentation..."
    )


    # ========================================================
    # PHYSICAL GROUPS
    # ========================================================

    print(
        "\nCreating physical groups..."
    )


    # Vacuum
    gmsh.model.addPhysicalGroup(
        3,
        [vacuum],
        VACUUM_ID
    )

    gmsh.model.setPhysicalName(
        3,
        VACUUM_ID,
        "Vacuum"
    )


    # Silicon
    gmsh.model.addPhysicalGroup(
        3,
        [silicon],
        SILICON_ID
    )

    gmsh.model.setPhysicalName(
        3,
        SILICON_ID,
        "Silicon_Substrate"
    )


    # PEC
    gmsh.model.addPhysicalGroup(
        2,
        metal_surfaces,
        PEC_ID
    )

    gmsh.model.setPhysicalName(
        2,
        PEC_ID,
        "Superconducting_Metal"
    )


    # JJ1 ... JJ8
    for i, surface in enumerate(
        jj_surfaces
    ):

        attr = (
            JJ_START_ID
            + i
        )


        gmsh.model.addPhysicalGroup(
            2,
            [surface],
            attr
        )

        gmsh.model.setPhysicalName(
            2,
            attr,
            f"JJ{i+1}"
        )


    # ========================================================
    # MESH SIZE
    # ========================================================

    gmsh.option.setNumber(
        "Mesh.CharacteristicLengthMin",
        BULK_SIZE
    )

    gmsh.option.setNumber(
        "Mesh.CharacteristicLengthMax",
        BULK_SIZE
    )


    # ========================================================
    # JJ REFINEMENT
    # ========================================================

    print(
        "\nApplying JJ refinement..."
    )


    fields = []


    for x, y in JJ_LOCATIONS:

        f = (
            gmsh.model.mesh.field.add(
                "Box"
            )
        )


        gmsh.model.mesh.field.setNumber(
            f,
            "VIn",
            JJ_SIZE
        )

        gmsh.model.mesh.field.setNumber(
            f,
            "VOut",
            BULK_SIZE
        )


        gmsh.model.mesh.field.setNumber(
            f,
            "XMin",
            x - JJ_REFINE_HALF
        )

        gmsh.model.mesh.field.setNumber(
            f,
            "XMax",
            x + JJ_REFINE_HALF
        )

        gmsh.model.mesh.field.setNumber(
            f,
            "YMin",
            y - JJ_REFINE_HALF
        )

        gmsh.model.mesh.field.setNumber(
            f,
            "YMax",
            y + JJ_REFINE_HALF
        )

        gmsh.model.mesh.field.setNumber(
            f,
            "ZMin",
            -SUBSTRATE_THICKNESS
        )

        gmsh.model.mesh.field.setNumber(
            f,
            "ZMax",
            AIR_HEIGHT
        )


        fields.append(
            f
        )


    if fields:

        fmin = (
            gmsh.model.mesh.field.add(
                "Min"
            )
        )

        gmsh.model.mesh.field.setNumbers(
            fmin,
            "FieldsList",
            fields
        )

        gmsh.model.mesh.field.setAsBackgroundMesh(
            fmin
        )


    # ========================================================
    # 3D MESH
    # ========================================================

    print(
        "\n" + "=" * 80
    )

    print(
        "GENERATING 3D MESH"
    )

    print(
        "=" * 80
    )


    gmsh.option.setNumber(
        "Mesh.Algorithm3D",
        1
    )


    gmsh.option.setNumber(
        "Mesh.Optimize",
        0
    )


    gmsh.option.setNumber(
        "Mesh.OptimizeNetgen",
        0
    )


    gmsh.model.mesh.generate(
        3
    )


    # ========================================================
    # CLEAN
    # ========================================================

    gmsh.model.mesh.removeDuplicateNodes()

    gmsh.model.mesh.removeDuplicateElements()


    # ========================================================
    # STATISTICS
    # ========================================================

    node_tags, _, _ = (
        gmsh.model.mesh.getNodes()
    )

    types, tags, _ = (
        gmsh.model.mesh.getElements()
    )


    total_elements = sum(
        len(t)
        for t in tags
    )


    print(
        "\n" + "=" * 80
    )

    print(
        "MESH STATISTICS"
    )

    print(
        "=" * 80
    )

    print(
        f"Nodes    : "
        f"{len(node_tags):,}"
    )

    print(
        f"Elements : "
        f"{total_elements:,}"
    )


    # ========================================================
    # PHYSICAL GROUPS
    # ========================================================

    print(
        "\nPhysical groups:"
    )


    for dim, tag in (
        gmsh.model.getPhysicalGroups()
    ):

        name = (
            gmsh.model.getPhysicalName(
                dim,
                tag
            )
        )


        entities = (
            gmsh.model.getEntitiesForPhysicalGroup(
                dim,
                tag
            )
        )


        print(
            f"  Dimension={dim}, "
            f"Attribute={tag}, "
            f"Name={name}, "
            f"Entities={len(entities)}"
        )


    # ========================================================
    # WRITE
    # ========================================================

    print(
        "\nWriting MSH 2.2..."
    )


    gmsh.option.setNumber(
        "Mesh.MshFileVersion",
        2.2
    )

    gmsh.option.setNumber(
        "Mesh.Binary",
        0
    )


    gmsh.write(
        OUTPUT_MESH
    )


    print(
        "\n" + "=" * 80
    )

    print(
        "8-JJ V2 MESH CREATED"
    )

    print(
        "=" * 80
    )

    print(
        "\nMesh:"
    )

    print(
        OUTPUT_MESH
    )


finally:

    gmsh.finalize()

    print(
        "\nGmsh finalized."
    )

ROBUST 8-JJ PALACE MESH V2

Reading GDS...
Loaded 19 cells.
Using top cell: TOP_main_1

Extracting superconducting metal...
Metal polygons: 2450
Valid metal polygons: 2450

Uniting metal...
Metal type: Polygon
Metal area: 185.750737 mm^2

Extracting JJ geometry from Layer 53 / Datatype 0...
Layer-53 JJ polygons found: 8

Matching actual JJ polygons to the eight JJ locations...
  JJ1: centroid=(-1.132500, 0.000000) mm, distance=0.000 um
  JJ2: centroid=(-0.867500, -0.000000) mm, distance=0.000 um
  JJ3: centroid=(1.267500, 0.000000) mm, distance=0.000 um
  JJ4: centroid=(1.532500, -0.000000) mm, distance=0.000 um
  JJ5: centroid=(-1.132500, -2.700000) mm, distance=0.000 um
  JJ6: centroid=(-0.867500, -2.700000) mm, distance=0.000 um
  JJ7: centroid=(1.267500, -2.700000) mm, distance=0.000 um
  JJ8: centroid=(1.532500, -2.700000) mm, distance=0.000 um

Initializing Gmsh...

Creating silicon...
Creating vacuum...

Creating metal surface...
Metal surfaces: 1

Creating JJ surfaces...
  JJ1:

In [ ]:
# ============================================================
# ROBUST 8-JJ PALACE MESH GENERATOR - V11
#
# Fixes V10:
#   - robust polygon vertex cleanup
#   - removes duplicate/near-duplicate vertices
#   - removes zero-length edges
#   - handles Polygon / MultiPolygon safely
#   - handles interior holes
#   - globally reuses OCC points
#   - does NOT automatically run Palace
#
# Target:
#   4 qubits
#   8 Josephson junctions
# ============================================================

import os
import math
import gdspy
import gmsh

from shapely.geometry import Polygon, MultiPolygon, box
from shapely.ops import unary_union


# ============================================================
# PATHS
# ============================================================

GDS_PATH = (
    "/content/drive/MyDrive/Palace6/meshes/"
    "quantum_chip_export_manuscript.gds"
)

OUTPUT_MESH = (
    "/content/drive/MyDrive/Palace6/meshes/"
    "quantum_chip_mesh_8JJ_v11.msh"
)


# ============================================================
# GDS LAYERS
# ============================================================

METAL_LAYER = 1
METAL_DATATYPE = 0

PAD_LAYER = 1
PAD_DATATYPE = 10

JJ_LAYER = 53
JJ_DATATYPE = 0


# ============================================================
# JJ LOCATIONS
# ============================================================

JJ_LOCATIONS = [
    (-1.1325,  0.0000),
    (-0.8675,  0.0000),

    ( 1.2675,  0.0000),
    ( 1.5325,  0.0000),

    (-1.1325, -2.7000),
    (-0.8675, -2.7000),

    ( 1.2675, -2.7000),
    ( 1.5325, -2.7000),
]


# ============================================================
# PHYSICAL JJ DIMENSIONS
# ============================================================

JJ_WIDTH = 0.020       # mm = 20 um
JJ_HEIGHT = 0.005      # mm = 5 um

CUT_TOL = 0.0005       # mm = 0.5 um


# ============================================================
# CHIP
# ============================================================

CHIP_HALF = 7.0

Z_SI_BOTTOM = -0.5
Z_INTERFACE = 0.0
Z_VAC_TOP = 0.5


# ============================================================
# MESH
# ============================================================

BULK_SIZE = 0.30       # 300 um
JJ_SIZE = 0.030        # 30 um
JJ_RADIUS = 0.10       # 100 um


# ============================================================
# PHYSICAL IDS
# ============================================================

VACUUM_ID = 1
SILICON_ID = 2
PEC_ID = 10
JJ_ID_START = 20


# ============================================================
# GEOMETRY TOLERANCES
# ============================================================

POINT_TOL = 1e-9
AREA_TOL = 1e-12


print("=" * 80)
print("ROBUST 8-JJ PALACE MESH GENERATOR - V11")
print("=" * 80)

print("\nGDS:")
print(GDS_PATH)

print("\nOutput:")
print(OUTPUT_MESH)


# ============================================================
# CHECK INPUT
# ============================================================

if not os.path.isfile(GDS_PATH):
    raise FileNotFoundError(GDS_PATH)


# ============================================================
# READ GDS
# ============================================================

print("\n" + "=" * 80)
print("READING GDS")
print("=" * 80)

lib = gdspy.GdsLibrary()
lib.read_gds(GDS_PATH)

print(f"Loaded {len(lib.cells)} cells.")

TOP_CELL = "TOP_main_1"

if TOP_CELL not in lib.cells:
    raise RuntimeError(
        f"Top cell {TOP_CELL} not found."
    )

top = lib.cells[TOP_CELL]

print(
    f"Using top cell: {TOP_CELL}"
)


# ============================================================
# GET POLYGONS
# ============================================================

polygons_by_spec = top.get_polygons(
    by_spec=True
)


# ============================================================
# SHAPELY CLEANER
# ============================================================

def clean_polygon(points):

    if points is None:
        return None

    if len(points) < 3:
        return None

    try:

        p = Polygon(points)

        if p.is_empty:
            return None

        if not p.is_valid:
            p = p.buffer(0)

        if p.is_empty:
            return None

        if p.geom_type == "MultiPolygon":

            parts = [
                g
                for g in p.geoms
                if g.area > AREA_TOL
            ]

            if not parts:
                return None

            p = max(
                parts,
                key=lambda g: g.area
            )

        if p.area <= AREA_TOL:
            return None

        return p

    except Exception:
        return None


# ============================================================
# MAIN METAL
# ============================================================

print("\n" + "=" * 80)
print("EXTRACTING MAIN SUPERCONDUCTING METAL")
print("=" * 80)

main_polygons = []

for spec, polys in polygons_by_spec.items():

    layer, datatype = spec

    if (
        layer == METAL_LAYER
        and datatype == METAL_DATATYPE
    ):

        for pts in polys:

            p = clean_polygon(pts)

            if p is not None:
                main_polygons.append(p)


print(
    f"Main metal polygons: "
    f"{len(main_polygons)}"
)

if not main_polygons:
    raise RuntimeError(
        "No main metal polygons found."
    )


print("Uniting main metal...")

main_metal = unary_union(
    main_polygons
)

if not main_metal.is_valid:
    main_metal = main_metal.buffer(0)

print(
    "Metal geometry:",
    main_metal.geom_type
)

print(
    f"Metal area: "
    f"{main_metal.area:.12f} mm²"
)


# ============================================================
# PAD POLYGONS
# ============================================================

print("\n" + "=" * 80)
print("EXTRACTING JJ ELECTRODE PADS")
print("=" * 80)

pad_polygons = []

for spec, polys in polygons_by_spec.items():

    layer, datatype = spec

    if (
        layer == PAD_LAYER
        and datatype == PAD_DATATYPE
    ):

        for pts in polys:

            p = clean_polygon(pts)

            if p is not None:
                pad_polygons.append(p)


print(
    f"Pad polygons: "
    f"{len(pad_polygons)}"
)


# ============================================================
# JJ REFERENCES
# ============================================================

print("\n" + "=" * 80)
print("EXTRACTING JJ REFERENCES")
print("=" * 80)

jj_refs = []

for spec, polys in polygons_by_spec.items():

    layer, datatype = spec

    if (
        layer == JJ_LAYER
        and datatype == JJ_DATATYPE
    ):

        for pts in polys:

            p = clean_polygon(pts)

            if p is not None:
                jj_refs.append(p)


print(
    f"JJ polygons: "
    f"{len(jj_refs)}"
)

if len(jj_refs) != 8:

    raise RuntimeError(
        f"Expected 8 JJ polygons, "
        f"found {len(jj_refs)}."
    )


# ============================================================
# MATCH JJ REFERENCES
# ============================================================

print("\n" + "=" * 80)
print("MATCHING JJ REFERENCES")
print("=" * 80)

unused = list(jj_refs)
matched_jjs = []

for i, (jx, jy) in enumerate(
    JJ_LOCATIONS,
    1
):

    best = None
    best_idx = None
    best_d = float("inf")

    for k, p in enumerate(unused):

        c = p.centroid

        d = math.hypot(
            c.x - jx,
            c.y - jy
        )

        if d < best_d:

            best = p
            best_idx = k
            best_d = d

    if best is None:

        raise RuntimeError(
            f"Cannot match JJ{i}."
        )

    if best_d > 0.050:

        raise RuntimeError(
            f"JJ{i} mismatch: "
            f"{best_d * 1000:.3f} um"
        )

    matched_jjs.append(best)

    unused.pop(best_idx)

    print(
        f"JJ{i}: "
        f"centroid=("
        f"{best.centroid.x:.6f}, "
        f"{best.centroid.y:.6f}) mm, "
        f"error={best_d*1000:.3f} um"
    )


# ============================================================
# FIND PAD PAIRS
# ============================================================

print("\n" + "=" * 80)
print("IDENTIFYING JJ ELECTRODE PAIRS")
print("=" * 80)

jj_pad_pairs = []
used_pad_indices = set()

for i, (jx, jy) in enumerate(
    JJ_LOCATIONS,
    1
):

    candidates = []

    for k, p in enumerate(
        pad_polygons
    ):

        if k in used_pad_indices:
            continue

        c = p.centroid

        d = math.hypot(
            c.x - jx,
            c.y - jy
        )

        if d < 0.080:

            candidates.append(
                (d, k, p)
            )

    left = [
        x for x in candidates
        if x[2].centroid.x < jx
    ]

    right = [
        x for x in candidates
        if x[2].centroid.x > jx
    ]

    if not left or not right:

        raise RuntimeError(
            f"JJ{i}: could not identify "
            "left/right electrodes."
        )

    L = min(
        left,
        key=lambda x: x[0]
    )

    R = min(
        right,
        key=lambda x: x[0]
    )

    _, li, lp = L
    _, ri, rp = R

    used_pad_indices.add(li)
    used_pad_indices.add(ri)

    jj_pad_pairs.append(
        (lp, rp)
    )

    print(
        f"\nJJ{i}"
    )

    print(
        "  LEFT :",
        lp.bounds
    )

    print(
        "  RIGHT:",
        rp.bounds
    )


# ============================================================
# BUILD PHYSICAL JJ GAPS
# ============================================================

print("\n" + "=" * 80)
print("BUILDING JJ GAPS")
print("=" * 80)

jj_physical_gaps = []
jj_cut_gaps = []

for i, (lp, rp) in enumerate(
    jj_pad_pairs,
    1
):

    lx1, ly1, lx2, ly2 = lp.bounds
    rx1, ry1, rx2, ry2 = rp.bounds

    minx = lx2
    maxx = rx1

    miny = max(
        ly1,
        ry1
    )

    maxy = min(
        ly2,
        ry2
    )

    width = maxx - minx
    height = maxy - miny

    print(
        f"JJ{i}: "
        f"{width*1000:.3f} x "
        f"{height*1000:.3f} um"
    )

    if width <= 0 or height <= 0:

        raise RuntimeError(
            f"JJ{i}: invalid gap."
        )

    if abs(width - JJ_WIDTH) > 0.002:

        raise RuntimeError(
            f"JJ{i}: unexpected width."
        )

    if abs(height - JJ_HEIGHT) > 0.002:

        raise RuntimeError(
            f"JJ{i}: unexpected height."
        )

    # Exact physical JJ gap

    physical_gap = box(
        minx,
        miny,
        maxx,
        maxy
    )

    # Slightly extended cut region.
    # This guarantees the PEC geometry is actually
    # separated from the JJ surface.

    cut_gap = box(
        minx - CUT_TOL,
        miny - CUT_TOL,
        maxx + CUT_TOL,
        maxy + CUT_TOL
    )

    jj_physical_gaps.append(
        physical_gap
    )

    jj_cut_gaps.append(
        cut_gap
    )


# ============================================================
# COMBINE METAL + PADS
# ============================================================

print("\n" + "=" * 80)
print("COMBINING METAL AND JJ ELECTRODES")
print("=" * 80)

metal_components = [
    main_metal
]

for lp, rp in jj_pad_pairs:

    metal_components.append(lp)
    metal_components.append(rp)


combined_metal = unary_union(
    metal_components
)

if not combined_metal.is_valid:
    combined_metal = combined_metal.buffer(0)


print(
    "Combined metal:",
    combined_metal.geom_type
)

print(
    f"Combined metal area: "
    f"{combined_metal.area:.12f} mm²"
)


# ============================================================
# CUT GAPS
# ============================================================

print("\n" + "=" * 80)
print("CUTTING JJ GAPS FROM METAL")
print("=" * 80)

final_metal = combined_metal

for i, gap in enumerate(
    jj_cut_gaps,
    1
):

    before = final_metal.area

    overlap = (
        final_metal.intersection(
            gap
        )
    )

    overlap_area = overlap.area

    print(
        f"JJ{i}: "
        f"metal overlap="
        f"{overlap_area:.12e} mm²"
    )

    if overlap_area <= 1e-12:

        raise RuntimeError(
            f"JJ{i}: gap does not "
            "intersect metal."
        )

    candidate = (
        final_metal.difference(
            gap
        )
    )

    if not candidate.is_valid:
        candidate = candidate.buffer(0)

    removed = (
        before -
        candidate.area
    )

    print(
        f"     removed="
        f"{removed:.12e} mm²"
    )

    if removed <= 1e-12:

        raise RuntimeError(
            f"JJ{i}: Boolean cut failed."
        )

    final_metal = candidate


# ============================================================
# NORMALIZE METAL INTO POLYGONS
# ============================================================

if final_metal.geom_type == "Polygon":

    metal_regions = [
        final_metal
    ]

elif final_metal.geom_type == "MultiPolygon":

    metal_regions = list(
        final_metal.geoms
    )

else:

    raise RuntimeError(
        f"Unexpected final metal type: "
        f"{final_metal.geom_type}"
    )


metal_regions = [
    p
    for p in metal_regions
    if p.area > AREA_TOL
]


print(
    f"\nFinal metal regions: "
    f"{len(metal_regions)}"
)


# ============================================================
# ROBUST RING CLEANER
# ============================================================

def clean_ring(coords):

    cleaned = []

    for x, y in coords:

        if not (
            math.isfinite(x)
            and math.isfinite(y)
        ):
            continue

        if not cleaned:

            cleaned.append(
                (x, y)
            )

            continue

        px, py = cleaned[-1]

        d = math.hypot(
            x - px,
            y - py
        )

        if d > POINT_TOL:

            cleaned.append(
                (x, y)
            )

    # Remove closing duplicate

    if len(cleaned) >= 2:

        if math.hypot(
            cleaned[0][0] - cleaned[-1][0],
            cleaned[0][1] - cleaned[-1][1]
        ) <= POINT_TOL:

            cleaned.pop()

    return cleaned


# ============================================================
# INITIALIZE GMSH
# ============================================================

print("\n" + "=" * 80)
print("INITIALIZING GMSH")
print("=" * 80)

gmsh.initialize()

gmsh.option.setNumber(
    "General.Terminal",
    1
)

gmsh.model.add(
    "4Q_8JJ_PALACE_V11"
)


# ============================================================
# GLOBAL OCC POINT CACHE
#
# THIS IS THE IMPORTANT FIX FOR V10.
# ============================================================

point_cache = {}


def get_point(x, y, z):

    key = (
        round(x, 9),
        round(y, 9),
        round(z, 9)
    )

    if key in point_cache:

        return point_cache[key]

    tag = gmsh.model.occ.addPoint(
        x,
        y,
        z
    )

    point_cache[key] = tag

    return tag


# ============================================================
# ROBUST OCC RING
# ============================================================

def make_ring(coords):

    pts = clean_ring(
        coords
    )

    if len(pts) < 3:

        raise RuntimeError(
            "Polygon ring has fewer "
            "than 3 unique vertices."
        )

    line_tags = []

    for k in range(
        len(pts)
    ):

        x1, y1 = pts[k]

        x2, y2 = pts[
            (k + 1) % len(pts)
        ]

        distance = math.hypot(
            x2 - x1,
            y2 - y1
        )

        if distance <= POINT_TOL:

            continue

        p1 = get_point(
            x1,
            y1,
            Z_INTERFACE
        )

        p2 = get_point(
            x2,
            y2,
            Z_INTERFACE
        )

        if p1 == p2:

            continue

        try:

            line = (
                gmsh.model.occ.addLine(
                    p1,
                    p2
                )
            )

        except Exception as exc:

            raise RuntimeError(
                "Could not create OCC line.\n"
                f"p1={p1}, p2={p2}\n"
                f"({x1},{y1}) -> "
                f"({x2},{y2})\n"
                f"distance={distance}\n"
                f"Original error: {exc}"
            )

        line_tags.append(
            line
        )

    if len(line_tags) < 3:

        raise RuntimeError(
            "Ring produced fewer than "
            "3 valid lines."
        )

    try:

        return gmsh.model.occ.addWire(
            line_tags
        )

    except Exception as exc:

        raise RuntimeError(
            f"Could not create wire "
            f"from {len(line_tags)} lines: "
            f"{exc}"
        )


# ============================================================
# POLYGON -> OCC SURFACE
# ============================================================

def polygon_to_surface(poly):

    if poly.is_empty:
        return None

    if poly.geom_type != "Polygon":

        raise RuntimeError(
            "polygon_to_surface received "
            f"{poly.geom_type}"
        )

    exterior = make_ring(
        list(
            poly.exterior.coords
        )
    )

    holes = []

    for interior in poly.interiors:

        ring = make_ring(
            list(
                interior.coords
            )
        )

        holes.append(
            ring
        )

    loops = [
        exterior
    ] + holes

    try:

        return (
            gmsh.model.occ.addPlaneSurface(
                loops
            )
        )

    except Exception as exc:

        raise RuntimeError(
            "Could not create plane surface.\n"
            f"Exterior vertices="
            f"{len(list(poly.exterior.coords))}\n"
            f"Holes={len(holes)}\n"
            f"Area={poly.area}\n"
            f"Error={exc}"
        )


# ============================================================
# CREATE SILICON
# ============================================================

print(
    "\nCreating silicon volume..."
)

silicon = gmsh.model.occ.addBox(
    -CHIP_HALF,
    -CHIP_HALF,
    Z_SI_BOTTOM,
    2 * CHIP_HALF,
    2 * CHIP_HALF,
    -Z_SI_BOTTOM
)


# ============================================================
# CREATE VACUUM
# ============================================================

print(
    "Creating vacuum volume..."
)

vacuum = gmsh.model.occ.addBox(
    -CHIP_HALF,
    -CHIP_HALF,
    Z_INTERFACE,
    2 * CHIP_HALF,
    2 * CHIP_HALF,
    Z_VAC_TOP
)


# ============================================================
# CREATE PEC SURFACES
# ============================================================

print(
    "\nCreating superconducting metal surfaces..."
)

pec_surfaces = []

for i, poly in enumerate(
    metal_regions,
    1
):

    print(
        f"  Metal region {i}/"
        f"{len(metal_regions)} "
        f"area={poly.area:.6e} mm²"
    )

    s = polygon_to_surface(
        poly
    )

    if s is not None:

        pec_surfaces.append(
            s
        )


print(
    f"PEC surfaces created: "
    f"{len(pec_surfaces)}"
)


# ============================================================
# CREATE JJ SURFACES
# ============================================================

print(
    "\nCreating physical JJ surfaces..."
)

jj_surfaces = []

for i, gap in enumerate(
    jj_physical_gaps,
    1
):

    minx, miny, maxx, maxy = (
        gap.bounds
    )

    coords = [
        (minx, miny),
        (maxx, miny),
        (maxx, maxy),
        (minx, maxy)
    ]

    wire = make_ring(
        coords
    )

    surface = (
        gmsh.model.occ.addPlaneSurface(
            [wire]
        )
    )

    jj_surfaces.append(
        surface
    )

    print(
        f"JJ{i}: "
        f"surface={surface}, "
        f"size="
        f"{(maxx-minx)*1000:.3f} x "
        f"{(maxy-miny)*1000:.3f} um"
    )


# ============================================================
# SYNCHRONIZE
# ============================================================

print(
    "\nSynchronizing OCC..."
)

gmsh.model.occ.synchronize()


# ============================================================
# FRAGMENT VOLUMES WITH INTERFACE
# ============================================================

print("\n" + "=" * 80)
print("CONFORMAL SILICON / VACUUM FRAGMENTATION")
print("=" * 80)

interface_surfaces = (
    pec_surfaces +
    jj_surfaces
)

objects = [
    (3, silicon),
    (3, vacuum)
]

tools = [
    (2, s)
    for s in interface_surfaces
]


print(
    f"Volumes: {len(objects)}"
)

print(
    f"Interface surfaces: "
    f"{len(tools)}"
)


fragmented, mapping = (
    gmsh.model.occ.fragment(
        objects,
        tools
    )
)

gmsh.model.occ.synchronize()


print(
    f"Fragmented entities: "
    f"{len(fragmented)}"
)


# ============================================================
# DO NOT CALL removeAllDuplicates()
#
# It can destroy the surface identity that we need
# for PEC/JJ classification.
# ============================================================

print(
    "Skipping aggressive OCC duplicate removal."
)


# ============================================================
# IDENTIFY VOLUMES
# ============================================================

volumes = (
    gmsh.model.getEntities(
        3
    )
)

print(
    f"\n3D volumes: "
    f"{len(volumes)}"
)

silicon_volumes = []
vacuum_volumes = []

for dim, tag in volumes:

    cx, cy, cz = (
        gmsh.model.occ.getCenterOfMass(
            3,
            tag
        )
    )

    vol = (
        gmsh.model.occ.getMass(
            3,
            tag
        )
    )

    print(
        f"Volume {tag}: "
        f"center=({cx:.6f}, "
        f"{cy:.6f}, "
        f"{cz:.6f}), "
        f"V={vol:.6e} mm³"
    )

    if cz < 0:

        silicon_volumes.append(
            tag
        )

    elif cz > 0:

        vacuum_volumes.append(
            tag
        )


if not silicon_volumes:

    raise RuntimeError(
        "No silicon volume found."
    )

if not vacuum_volumes:

    raise RuntimeError(
        "No vacuum volume found."
    )


print(
    "\nPASS: silicon and vacuum "
    "volumes identified."
)


# ============================================================
# GET INTERFACE FACES
# ============================================================

all_faces = (
    gmsh.model.getEntities(
        2
    )
)

interface_faces = []

for dim, tag in all_faces:

    try:

        cx, cy, cz = (
            gmsh.model.occ.getCenterOfMass(
                2,
                tag
            )
        )

    except Exception:

        continue

    if abs(cz) < 1e-7:

        interface_faces.append(
            tag
        )


print(
    f"\nInterface faces at z=0: "
    f"{len(interface_faces)}"
)


# ============================================================
# IDENTIFY JJ FACES
# ============================================================

print(
    "\nIdentifying JJ faces..."
)

final_jj_faces = []

for i, (jx, jy) in enumerate(
    JJ_LOCATIONS,
    1
):

    candidates = []

    for tag in interface_faces:

        try:

            cx, cy, cz = (
                gmsh.model.occ.getCenterOfMass(
                    2,
                    tag
                )
            )

            area = (
                gmsh.model.occ.getMass(
                    2,
                    tag
                )
            )

        except Exception:

            continue

        d = math.hypot(
            cx - jx,
            cy - jy
        )

        # Expected JJ area:
        # 20 um * 5 um = 100 um²
        # = 1e-4 mm²

        if (
            d < 0.002
            and
            2e-5 < area < 5e-4
        ):

            candidates.append(
                (d, tag, area)
            )


    if not candidates:

        raise RuntimeError(
            f"Could not identify JJ{i}."
        )


    candidates.sort(
        key=lambda x: x[0]
    )

    d, tag, area = candidates[0]

    final_jj_faces.append(
        tag
    )

    print(
        f"JJ{i}: "
        f"face={tag}, "
        f"distance={d*1000:.3f} um, "
        f"area={area:.6e} mm²"
    )


if len(
    set(final_jj_faces)
) != 8:

    raise RuntimeError(
        "JJ faces are not unique."
    )


print(
    "PASS: 8 unique JJ faces found."
)


# ============================================================
# PEC FACES
# ============================================================

jj_set = set(
    final_jj_faces
)

pec_faces = []

for tag in interface_faces:

    if tag in jj_set:
        continue

    try:

        area = (
            gmsh.model.occ.getMass(
                2,
                tag
            )
        )

    except Exception:

        continue

    if area > 1e-9:

        pec_faces.append(
            tag
        )


print(
    f"PEC faces: "
    f"{len(pec_faces)}"
)

if not pec_faces:

    raise RuntimeError(
        "No PEC faces found."
    )


print(
    "PASS: PEC faces identified."
)


# ============================================================
# PHYSICAL GROUPS
# ============================================================

print(
    "\nCreating physical groups..."
)

gmsh.model.addPhysicalGroup(
    3,
    silicon_volumes,
    SILICON_ID
)

gmsh.model.setPhysicalName(
    3,
    SILICON_ID,
    "Silicon_Substrate"
)


gmsh.model.addPhysicalGroup(
    3,
    vacuum_volumes,
    VACUUM_ID
)

gmsh.model.setPhysicalName(
    3,
    VACUUM_ID,
    "Vacuum"
)


gmsh.model.addPhysicalGroup(
    2,
    pec_faces,
    PEC_ID
)

gmsh.model.setPhysicalName(
    2,
    PEC_ID,
    "Superconducting_Metal"
)


for i, face in enumerate(
    final_jj_faces,
    1
):

    gid = (
        JJ_ID_START +
        i -
        1
    )

    gmsh.model.addPhysicalGroup(
        2,
        [face],
        gid
    )

    gmsh.model.setPhysicalName(
        2,
        gid,
        f"JJ{i}"
    )


# ============================================================
# MESH SETTINGS
# ============================================================

print(
    "\nConfiguring mesh..."
)

gmsh.option.setNumber(
    "Mesh.MshFileVersion",
    2.2
)

gmsh.option.setNumber(
    "Mesh.Binary",
    0
)

gmsh.option.setNumber(
    "Mesh.CharacteristicLengthMin",
    JJ_SIZE
)

gmsh.option.setNumber(
    "Mesh.CharacteristicLengthMax",
    BULK_SIZE
)

gmsh.option.setNumber(
    "Mesh.Optimize",
    0
)

gmsh.option.setNumber(
    "Mesh.OptimizeNetgen",
    0
)


# ============================================================
# JJ REFINEMENT
# ============================================================

print(
    "Applying JJ refinement..."
)

fields = []

for x, y in JJ_LOCATIONS:

    f = (
        gmsh.model.mesh.field.add(
            "Box"
        )
    )

    gmsh.model.mesh.field.setNumber(
        f,
        "VIn",
        JJ_SIZE
    )

    gmsh.model.mesh.field.setNumber(
        f,
        "VOut",
        BULK_SIZE
    )

    gmsh.model.mesh.field.setNumber(
        f,
        "XMin",
        x - JJ_RADIUS
    )

    gmsh.model.mesh.field.setNumber(
        f,
        "XMax",
        x + JJ_RADIUS
    )

    gmsh.model.mesh.field.setNumber(
        f,
        "YMin",
        y - JJ_RADIUS
    )

    gmsh.model.mesh.field.setNumber(
        f,
        "YMax",
        y + JJ_RADIUS
    )

    gmsh.model.mesh.field.setNumber(
        f,
        "ZMin",
        Z_SI_BOTTOM
    )

    gmsh.model.mesh.field.setNumber(
        f,
        "ZMax",
        Z_VAC_TOP
    )

    fields.append(f)


if fields:

    fmin = (
        gmsh.model.mesh.field.add(
            "Min"
        )
    )

    gmsh.model.mesh.field.setNumbers(
        fmin,
        "FieldsList",
        fields
    )

    gmsh.model.mesh.field.setAsBackgroundMesh(
        fmin
    )


# ============================================================
# GENERATE MESH
# ============================================================

print("\n" + "=" * 80)
print("GENERATING 3D TETRAHEDRAL MESH")
print("=" * 80)

gmsh.model.mesh.generate(
    3
)


# ============================================================
# CLEAN MESH
# ============================================================

print(
    "\nRemoving duplicate nodes..."
)

gmsh.model.mesh.removeDuplicateNodes()

print(
    "Removing duplicate elements..."
)

gmsh.model.mesh.removeDuplicateElements()


# ============================================================
# BASIC MESH STATISTICS
# ============================================================

nodes, _, _ = (
    gmsh.model.mesh.getNodes()
)

types, element_tags, element_nodes = (
    gmsh.model.mesh.getElements()
)

total_elements = 0
degenerate = 0
tetrahedra = 0

for etype, tags, conn in zip(
    types,
    element_tags,
    element_nodes
):

    total_elements += len(tags)

    if etype == 4:

        tetrahedra += len(tags)

        for k in range(
            0,
            len(conn),
            4
        ):

            verts = conn[
                k:k+4
            ]

            if len(
                set(verts)
            ) != 4:

                degenerate += 1


print(
    "\n" + "=" * 80
)

print(
    "MESH STATISTICS"
)

print(
    "=" * 80
)

print(
    f"Nodes        : "
    f"{len(nodes):,}"
)

print(
    f"Total elements: "
    f"{total_elements:,}"
)

print(
    f"Tetrahedra   : "
    f"{tetrahedra:,}"
)

print(
    f"Degenerate   : "
    f"{degenerate:,}"
)


if degenerate != 0:

    raise RuntimeError(
        "Degenerate tetrahedral elements found."
    )


# ============================================================
# PHYSICAL GROUP CHECK
# ============================================================

print(
    "\n" + "=" * 80
)

print(
    "PHYSICAL GROUP VALIDATION"
)

print(
    "=" * 80
)

required = {
    1: "Vacuum",
    2: "Silicon_Substrate",
    10: "Superconducting_Metal",
    20: "JJ1",
    21: "JJ2",
    22: "JJ3",
    23: "JJ4",
    24: "JJ5",
    25: "JJ6",
    26: "JJ7",
    27: "JJ8",
}

groups = (
    gmsh.model.getPhysicalGroups()
)

group_dict = {}

for dim, tag in groups:

    group_dict[tag] = (
        dim,
        gmsh.model.getPhysicalName(
            dim,
            tag
        )
    )


for gid, expected in required.items():

    if gid not in group_dict:

        raise RuntimeError(
            f"Missing group {gid}: "
            f"{expected}"
        )

    dim, name = group_dict[gid]

    print(
        f"PASS  "
        f"dim={dim}, "
        f"id={gid}: "
        f"{name}"
    )


# ============================================================
# WRITE MESH
# ============================================================

print(
    "\n" + "=" * 80
)

print(
    "WRITING MSH 2.2"
)

print(
    "=" * 80
)

gmsh.write(
    OUTPUT_MESH
)


# ============================================================
# FINAL FILE CHECK
# ============================================================

if not os.path.isfile(
    OUTPUT_MESH
):

    raise RuntimeError(
        "Mesh file was not written."
    )


size_gb = (
    os.path.getsize(
        OUTPUT_MESH
    )
    /
    (1024**3)
)


# ============================================================
# SUCCESS
# ============================================================

print(
    "\n" + "=" * 80
)

print(
    "SUCCESS"
)

print(
    "=" * 80
)

print(
    f"Mesh:"
)

print(
    OUTPUT_MESH
)

print(
    f"\nFile size: "
    f"{size_gb:.3f} GB"
)

print(
    f"Nodes: "
    f"{len(nodes):,}"
)

print(
    f"Elements: "
    f"{total_elements:,}"
)

print(
    "\nPhysical groups:"
)

print(
    "  1  -> Vacuum"
)

print(
    "  2  -> Silicon_Substrate"
)

print(
    "  10 -> Superconducting_Metal"
)

for i in range(1, 9):

    print(
        f"  {19+i:2d} -> JJ{i}"
    )


print(
    "\nDO NOT RUN PALACE YET."
)

print(
    "First test Palace mesh reading only."
)


# ============================================================
# FINALIZE
# ============================================================

gmsh.finalize()

print(
    "\nGmsh finalized."
)

ROBUST 8-JJ PALACE MESH GENERATOR - V11

GDS:
/content/drive/MyDrive/Palace6/meshes/quantum_chip_export_manuscript.gds

Output:
/content/drive/MyDrive/Palace6/meshes/quantum_chip_mesh_8JJ_v11.msh

READING GDS
Loaded 19 cells.
Using top cell: TOP_main_1

EXTRACTING MAIN SUPERCONDUCTING METAL
Main metal polygons: 2450
Uniting main metal...
Metal geometry: MultiPolygon
Metal area: 185.737968344919 mm²

EXTRACTING JJ ELECTRODE PADS
Pad polygons: 90

EXTRACTING JJ REFERENCES
JJ polygons: 8

MATCHING JJ REFERENCES
JJ1: centroid=(-1.132500, 0.000000) mm, error=0.000 um
JJ2: centroid=(-0.867500, -0.000000) mm, error=0.000 um
JJ3: centroid=(1.267500, 0.000000) mm, error=0.000 um
JJ4: centroid=(1.532500, -0.000000) mm, error=0.000 um
JJ5: centroid=(-1.132500, -2.700000) mm, error=0.000 um
JJ6: centroid=(-0.867500, -2.700000) mm, error=0.000 um
JJ7: centroid=(1.267500, -2.700000) mm, error=0.000 um
JJ8: centroid=(1.532500, -2.700000) mm, error=0.000 um

IDENTIFYING JJ ELECTRODE PAIRS

JJ1
  LEFT 

In [ ]:
# config_file_path = 'palace_config.json'
# with open(config_file_path, 'w') as f:
#     json.dump(palace_config, f, indent=4)


# # ==========================================
# # STEP 5: Robust 3D Meshing Handler for GDS
# # ==========================================
# def generate_palace_mesh(gds_path, msh_output_path):
#     import gmsh
#     print(f"Initializing Gmsh for 3D volumetric extrusion pipeline...")
#     gmsh.initialize()
#     gmsh.option.setNumber("General.Terminal", 1)
#     gmsh.model.add("quantum_chip_3d_model")

#     if not os.path.exists(gds_path):
#         raise FileNotFoundError(f"GDS file missing: {gds_path}")

#     # Note: To parse GDS layouts directly into Gmsh solid volumes,
#     # pipelines like `meshwell` or `gds2palace` convert polygons to STEP/OpenCASCADE shapes.
#     # Below is the direct structural handler pattern once shapes are loaded into Gmsh OpenCASCADE:

#     try:
#         # Creating standard substrate bounding box representation fallback for automated script flow
#         # (Substrate Box: 14mm x 14mm x 500um thickness)
#         substrate = gmsh.model.occ.addBox(-7.0, -7.0, -0.5, 14.0, 14.0, 0.5, tag=1)
#         # Vacuum Box overlay (Air boundary for 3D full-wave containment)
#         airbox = gmsh.model.occ.addBox(-7.0, -7.0, 0.0, 14.0, 14.0, 3.0, tag=2)

#         gmsh.model.occ.synchronize()

#         # Set mesh sizing constraints near fine superconducting features
#         gmsh.option.setNumber("Mesh.CharacteristicLengthMin", 2.0)
#         gmsh.option.setNumber("Mesh.CharacteristicLengthMax", 40.0)

#         print("Executing 3D Tetrahedral Volumetric Generation...")
#         gmsh.model.mesh.generate(3)
#         gmsh.write(msh_output_path)
#         print(f"Successfully generated mesh: {msh_output_path}")

#     except Exception as e:
#         print(f"Meshing routine warning/error handled: {e}")
#     finally:
#         gmsh.finalize()

# generate_palace_mesh(output_gds_path, 'quantum_chip_mesh.msh')




In [ ]:
import os
import stat
import shutil

validator = "/content/drive/MyDrive/Palace6/build/bin/validate-config"

print("Validator exists:", os.path.exists(validator))
print("Current permissions:", oct(os.stat(validator).st_mode))

# Add executable permission
os.chmod(
    validator,
    os.stat(validator).st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH
)

print("New permissions:", oct(os.stat(validator).st_mode))


In [ ]:
# ============================================================
# PALACE V11 — MINIMAL CONFIGURATION VALIDATION
# ============================================================
#
# IMPORTANT:
#   This ONLY validates the JSON configuration.
#   It does NOT run the 8-JJ simulation.
#
# ============================================================

import os
import json
import subprocess

PALACE = (
    "/content/drive/MyDrive/Palace6/"
    "build/bin/palace-x86_64.bin"
)

VALIDATOR = (
    "/content/drive/MyDrive/Palace6/"
    "build/bin/validate-config"
)

MESH = (
    "/content/drive/MyDrive/Palace6/meshes/"
    "quantum_chip_mesh_8JJ_v11.msh"
)

CONFIG = "/content/palace_8jj_validation_v3.json"


print("=" * 80)
print("PALACE V11 — MINIMAL CONFIGURATION VALIDATION")
print("=" * 80)

# ------------------------------------------------------------
# Check files
# ------------------------------------------------------------

if not os.path.isfile(MESH):
    raise FileNotFoundError(
        f"Mesh not found:\n{MESH}"
    )

if not os.path.isfile(VALIDATOR):
    raise FileNotFoundError(
        f"Palace validator not found:\n{VALIDATOR}"
    )

mesh_size = os.path.getsize(MESH) / (1024**3)

print("\nMesh:")
print(MESH)

print(f"\nMesh size: {mesh_size:.3f} GB")


# ============================================================
# MINIMAL PALACE CONFIGURATION
# ============================================================

config = {

    # --------------------------------------------------------
    # PROBLEM
    # --------------------------------------------------------

    "Problem": {
        "Type": "Eigenmode",
        "Output": "/content/palace_8jj_validation_output",
        "Verbose": 2
    },


    # --------------------------------------------------------
    # MODEL
    # --------------------------------------------------------
    #
    # Your Gmsh mesh coordinates are in mm.
    #
    # Therefore:
    #
    #       1 mesh unit = 1 mm = 1e-3 m
    #
    # Palace uses L0 to convert mesh units to SI.
    #
    # --------------------------------------------------------

    "Model": {
        "Mesh": MESH,
        "L0": 1.0e-3
    },


    # --------------------------------------------------------
    # DOMAINS
    # --------------------------------------------------------
    #
    # Attribute 1 = Vacuum
    # Attribute 2 = Silicon_Substrate
    #
    # --------------------------------------------------------

    "Domains": {
        "Materials": [

            {
                "Attributes": [1],
                "Permittivity": 1.0,
                "Permeability": 1.0
            },

            {
                "Attributes": [2],
                "Permittivity": 11.45,
                "Permeability": 1.0
            }
        ]
    },


    # --------------------------------------------------------
    # BOUNDARIES
    # --------------------------------------------------------
    #
    # Attribute 10 = superconducting metal
    #
    # For this validation we only need to confirm that
    # Palace accepts the PEC boundary definition.
    #
    # --------------------------------------------------------

    "Boundaries": {
        "PEC": {
            "Attributes": [10]
        }
    },


    # --------------------------------------------------------
    # SOLVER
    # --------------------------------------------------------
    #
    # Keep this minimal.
    #
    # Do NOT add guessed fields such as:
    #   Linear
    #   NodeOrder
    #   MaxIts
    #
    # until the configuration itself is validated.
    #
    # --------------------------------------------------------

    "Solver": {

        "Order": 1,

        "Eigenmode": {
            "N": 1,
            "Target": 5.0,
            "Tol": 1.0e-8
        }
    }
}


# ============================================================
# WRITE CONFIG
# ============================================================

with open(CONFIG, "w") as f:
    json.dump(
        config,
        f,
        indent=4
    )


print("\nConfiguration written:")
print(CONFIG)


# ============================================================
# DISPLAY CONFIGURATION
# ============================================================

print("\n" + "=" * 80)
print("CONFIGURATION")
print("=" * 80)

print(
    json.dumps(
        config,
        indent=4
    )
)


# ============================================================
# VALIDATE ONLY
# ============================================================

print("\n" + "=" * 80)
print("RUNNING PALACE CONFIGURATION VALIDATOR")
print("=" * 80)

validation = subprocess.run(
    [
        VALIDATOR,
        CONFIG
    ],
    capture_output=True,
    text=True
)


print("\nValidator stdout:")
print(validation.stdout)

print("\nValidator stderr:")
print(validation.stderr)

print(
    f"\nValidator return code: {validation.returncode}"
)


# ============================================================
# STOP ON ERROR
# ============================================================

if validation.returncode != 0:

    raise RuntimeError(
        "\n"
        "PALACE CONFIGURATION VALIDATION FAILED.\n"
        "\n"
        "DO NOT RUN PALACE.\n"
        "\n"
        "The V11 mesh has NOT been modified."
    )


# ============================================================
# SUCCESS
# ============================================================

print("\n" + "=" * 80)
print("PASS")
print("=" * 80)

print("""
Palace configuration validation succeeded.

The V11 mesh itself has NOT been simulated.

Next step:
    Palace mesh-read-only test
    ↓
    1 eigenmode sanity test
    ↓
    only then full 8-JJ simulation
""")

PALACE V11 — MINIMAL CONFIGURATION VALIDATION

Mesh:
/content/drive/MyDrive/Palace6/meshes/quantum_chip_mesh_8JJ_v11.msh

Mesh size: 0.819 GB

Configuration written:
/content/palace_8jj_validation_v3.json

CONFIGURATION
{
    "Problem": {
        "Type": "Eigenmode",
        "Output": "/content/palace_8jj_validation_output",
        "Verbose": 2
    },
    "Model": {
        "Mesh": "/content/drive/MyDrive/Palace6/meshes/quantum_chip_mesh_8JJ_v11.msh",
        "L0": 0.001
    },
    "Domains": {
        "Materials": [
            {
                "Attributes": [
                    1
                ],
                "Permittivity": 1.0,
                "Permeability": 1.0
            },
            {
                "Attributes": [
                    2
                ],
                "Permittivity": 11.45,
                "Permeability": 1.0
            }
        ]
    },
    "Boundaries": {
        "PEC": {
            "Attributes": [
                10
            ]
        }


In [ ]:
import os
import subprocess

PALACE = "/content/drive/MyDrive/Palace6/build/bin/palace-x86_64.bin"
LIBDIR = "/content/drive/MyDrive/Palace6/build/lib"

print("=" * 80)
print("PALACE SHARED-LIBRARY VERIFICATION")
print("=" * 80)

# ------------------------------------------------------------
# 1. Check libraries
# ------------------------------------------------------------

required = [
    "libceed.so",
    "libxsmm.so",
    "libxsmmgen.so",
]

print("\nChecking required libraries:")

for name in required:
    path = os.path.join(LIBDIR, name)

    if os.path.exists(path):
        print(f"  [PASS] {path}")
    else:
        print(f"  [FAIL] {path}")

# ------------------------------------------------------------
# 2. Check Palace executable
# ------------------------------------------------------------

print("\nChecking Palace executable:")

if os.path.isfile(PALACE):
    print("  [PASS] Palace executable exists")
else:
    raise RuntimeError("Palace executable not found.")

if os.access(PALACE, os.X_OK):
    print("  [PASS] Palace executable permission")
else:
    raise RuntimeError(
        "Palace executable is not executable."
    )

# ------------------------------------------------------------
# 3. Set LD_LIBRARY_PATH
# ------------------------------------------------------------

env = os.environ.copy()

old = env.get("LD_LIBRARY_PATH", "")

if old:
    env["LD_LIBRARY_PATH"] = (
        LIBDIR + ":" + old
    )
else:
    env["LD_LIBRARY_PATH"] = LIBDIR

print("\nLD_LIBRARY_PATH:")
print(env["LD_LIBRARY_PATH"])

# ------------------------------------------------------------
# 4. Run ldd
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("LDD CHECK")
print("=" * 80)

ldd = subprocess.run(
    ["ldd", PALACE],
    capture_output=True,
    text=True,
    env=env
)

print(ldd.stdout)

missing = []

for line in ldd.stdout.splitlines():

    if "not found" in line:
        missing.append(line)

if missing:

    print("\n" + "=" * 80)
    print("MISSING LIBRARIES")
    print("=" * 80)

    for line in missing:
        print(line)

    raise RuntimeError(
        "\nPalace still has unresolved shared libraries."
    )

print("\nPASS: No missing shared libraries detected.")

# ------------------------------------------------------------
# 5. Test direct Palace startup
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PALACE STARTUP TEST")
print("=" * 80)

test = subprocess.run(
    [PALACE, "--help"],
    capture_output=True,
    text=True,
    env=env
)

print("\nSTDOUT:")
print(test.stdout[:5000])

print("\nSTDERR:")
print(test.stderr[:5000])

print("\nReturn code:", test.returncode)

if test.returncode != 0:

    raise RuntimeError(
        "\nPalace executable could not start correctly."
    )

print("\n" + "=" * 80)
print("PALACE INSTALLATION: PASS")
print("=" * 80)

print("""
libCEED is visible.
Palace shared libraries are resolved.
Palace executable starts successfully.

NEXT:
Run the V11 1-mode mesh test.
Do NOT run the full 8-JJ simulation yet.
""")

PALACE SHARED-LIBRARY VERIFICATION

Checking required libraries:
  [PASS] /content/drive/MyDrive/Palace6/build/lib/libceed.so
  [PASS] /content/drive/MyDrive/Palace6/build/lib/libxsmm.so
  [PASS] /content/drive/MyDrive/Palace6/build/lib/libxsmmgen.so

Checking Palace executable:
  [PASS] Palace executable exists
  [PASS] Palace executable permission

LD_LIBRARY_PATH:
/content/drive/MyDrive/Palace6/build/lib

LDD CHECK
	linux-vdso.so.1 (0x00007ffed0d48000)
	libmpi_cxx.so.40 => /lib/x86_64-linux-gnu/libmpi_cxx.so.40 (0x00007fc7704e8000)
	libmpi.so.40 => /lib/x86_64-linux-gnu/libmpi.so.40 (0x00007fc76cac9000)
	libstdc++.so.6 => /lib/x86_64-linux-gnu/libstdc++.so.6 (0x00007fc76c89d000)
	libceed.so => /content/drive/MyDrive/Palace6/build/lib/libceed.so (0x00007fc76c400000)
	libz.so.1 => /lib/x86_64-linux-gnu/libz.so.1 (0x00007fc7704cc000)
	libopenblas.so.0 => /lib/x86_64-linux-gnu/libopenblas.so.0 (0x00007fc769fb0000)
	libgomp.so.1 => /lib/x86_64-linux-gnu/libgomp.so.1 (0x00007fc770480000)


In [ ]:

# =============================================================================
# 8-JJ GEOMETRY-ONLY VALIDATION
# =============================================================================
# SAFE TEST:
#   - Reads GDS only
#   - Does NOT initialize Gmsh
#   - Does NOT generate a 3D mesh
#   - Does NOT run Palace
#   - Verifies the actual electrode gaps around all 8 JJs
#
# Expected:
#   Each JJ:
#       gap width  ~= 20 um
#       gap height ~= 5 um
#       left/right electrodes on opposite sides of JJ
#       JJ reference centered inside the gap
# =============================================================================

import gdspy
import numpy as np
from shapely.geometry import Polygon, box
from shapely.ops import unary_union

# -----------------------------------------------------------------------------
# PATH
# -----------------------------------------------------------------------------

GDS_FILE = (
    "/content/drive/MyDrive/Palace6/meshes/"
    "quantum_chip_export_manuscript.gds"
)

# -----------------------------------------------------------------------------
# JJ LOCATIONS (mm)
# -----------------------------------------------------------------------------

JJ_TARGETS = np.array([
    [-1.132500,  0.000000],   # JJ1
    [-0.867500,  0.000000],   # JJ2
    [ 1.267500,  0.000000],   # JJ3
    [ 1.532500,  0.000000],   # JJ4
    [-1.132500, -2.700000],   # JJ5
    [-0.867500, -2.700000],   # JJ6
    [ 1.267500, -2.700000],   # JJ7
    [ 1.532500, -2.700000],   # JJ8
])

# -----------------------------------------------------------------------------
# PARAMETERS
# -----------------------------------------------------------------------------

# GDS coordinates are converted to mm.
# Do not use the JJ rectangle as a metal subtraction.
#
# We search the actual Layer-1/datatype-10 electrode polygons.

PAD_LAYER = 1
PAD_DATATYPE = 10

JJ_LAYER = 53
JJ_DATATYPE = 0

SEARCH_X = 0.080       # mm
SEARCH_Y = 0.020       # mm

EXPECTED_GAP_X = 0.020 # 20 um
EXPECTED_GAP_Y = 0.005 # 5 um

TOL = 2e-6             # 2 um geometry tolerance

print("=" * 80)
print("8-JJ GDS ELECTRODE/GAP GEOMETRY VALIDATION")
print("=" * 80)

print("\nGDS:")
print(GDS_FILE)

# -----------------------------------------------------------------------------
# READ GDS
# -----------------------------------------------------------------------------

print("\nReading GDS...")

gds = gdspy.GdsLibrary(infile=GDS_FILE)

print(f"Loaded {len(gds.cells)} cells.")

# Find top-level cell
top_cells = gds.top_level()

if len(top_cells) == 0:
    raise RuntimeError("No top-level GDS cell found.")

top = top_cells[0]

print(f"Using top cell: {top.name}")

# -----------------------------------------------------------------------------
# EXTRACT POLYGONS
# -----------------------------------------------------------------------------

print("\nExtracting Layer 1 / datatype 10 electrode polygons...")

raw_pad_polys = []

for polyset in top.get_polygons(
    by_spec=True
).get((PAD_LAYER, PAD_DATATYPE), []):
    pass



8-JJ GDS ELECTRODE/GAP GEOMETRY VALIDATION

GDS:
/content/drive/MyDrive/Palace6/meshes/quantum_chip_export_manuscript.gds

Reading GDS...
Loaded 19 cells.
Using top cell: FakeJunction_01

Extracting Layer 1 / datatype 10 electrode polygons...


In [ ]:

# =============================================================================
# 8-JJ GDS ELECTRODE/GAP VALIDATION - CORRECT TOP CELL
# =============================================================================
# SAFE:
#   - GDS geometry inspection only
#   - NO Gmsh
#   - NO 3D mesh
#   - NO Palace
#
# Purpose:
#   Verify the REAL 20 um x 5 um JJ gaps using the actual electrode geometry.
# =============================================================================

import gdspy
import numpy as np
from shapely.geometry import Polygon, box
from shapely.ops import unary_union

# -----------------------------------------------------------------------------
# FILE
# -----------------------------------------------------------------------------

GDS_FILE = (
    "/content/drive/MyDrive/Palace6/meshes/"
    "quantum_chip_export_manuscript.gds"
)

# -----------------------------------------------------------------------------
# IMPORTANT: ACTUAL CHIP TOP CELL
# -----------------------------------------------------------------------------

TOP_CELL_NAME = "TOP_main_1"

# -----------------------------------------------------------------------------
# LAYER INFORMATION
# -----------------------------------------------------------------------------

MAIN_LAYER = 1
MAIN_DATATYPE = 0

PAD_LAYER = 1
PAD_DATATYPE = 10

JJ_LAYER = 53
JJ_DATATYPE = 0

# -----------------------------------------------------------------------------
# EXPECTED JJ LOCATIONS (mm)
# -----------------------------------------------------------------------------

JJ_TARGETS = np.array([
    [-1.132500,  0.000000],   # JJ1
    [-0.867500,  0.000000],   # JJ2
    [ 1.267500,  0.000000],   # JJ3
    [ 1.532500,  0.000000],   # JJ4
    [-1.132500, -2.700000],   # JJ5
    [-0.867500, -2.700000],   # JJ6
    [ 1.267500, -2.700000],   # JJ7
    [ 1.532500, -2.700000],   # JJ8
])

EXPECTED_GAP_X = 0.020   # mm = 20 um
EXPECTED_GAP_Y = 0.005   # mm = 5 um

JJ_TOL = 0.005           # mm = 5 um
PAD_SEARCH_X = 0.060     # mm
PAD_SEARCH_Y = 0.020     # mm

print("=" * 80)
print("8-JJ GDS ELECTRODE/GAP GEOMETRY VALIDATION")
print("=" * 80)

print("\nGDS:")
print(GDS_FILE)

# =============================================================================
# READ GDS
# =============================================================================

print("\nReading GDS...")

gds = gdspy.GdsLibrary(infile=GDS_FILE)

print(f"Loaded {len(gds.cells)} cells.")

# =============================================================================
# SHOW TOP-LEVEL CELLS
# =============================================================================

print("\nAvailable top-level cells:")

top_cells = gds.top_level()

for c in top_cells:
    print("  ", c.name)

# =============================================================================
# FORCE CORRECT CHIP CELL
# =============================================================================

if TOP_CELL_NAME not in gds.cells:
    raise RuntimeError(
        f"Required chip cell '{TOP_CELL_NAME}' was not found.\n"
        f"Available cells include:\n"
        + "\n".join(sorted(gds.cells.keys()))
    )

top = gds.cells[TOP_CELL_NAME]

print("\nUsing chip cell:")
print(" ", top.name)

if top.name != TOP_CELL_NAME:
    raise RuntimeError("Wrong GDS cell selected.")

# =============================================================================
# EXTRACT POLYGONS INCLUDING REFERENCES
# =============================================================================
#
# get_polygons(by_spec=True) resolves cell references when called on the
# selected top cell.
# =============================================================================

print("\nExtracting polygons from TOP_main_1...")

polys_by_spec = top.get_polygons(
    by_spec=True,
    depth=None
)

print(f"Number of layer/datatype specifications: {len(polys_by_spec)}")

# =============================================================================
# HELPER
# =============================================================================

def get_polygons(spec):
    """Return polygons for a given (layer, datatype)."""
    arr = polys_by_spec.get(spec, [])

    result = []

    for p in arr:
        p = np.asarray(p, dtype=float)

        if len(p) < 3:
            continue

        try:
            shp = Polygon(p)

            if not shp.is_valid:
                shp = shp.buffer(0)

            if not shp.is_empty and shp.area > 0:
                result.append(shp)

        except Exception:
            continue

    return result


# =============================================================================
# MAIN METAL
# =============================================================================

main_polys = get_polygons(
    (MAIN_LAYER, MAIN_DATATYPE)
)

print("\nMain metal polygons:")
print(" ", len(main_polys))

if len(main_polys) == 0:
    raise RuntimeError(
        "No Layer 1 / datatype 0 polygons found in TOP_main_1."
    )

main_union = unary_union(main_polys)

print("Main metal geometry:", main_union.geom_type)
print(
    "Main metal area:",
    f"{main_union.area:.12f}",
    "mm²"
)

# =============================================================================
# PAD POLYGONS
# =============================================================================

pad_polys = get_polygons(
    (PAD_LAYER, PAD_DATATYPE)
)

print("\nLayer 1 / datatype 10 polygons:")
print(" ", len(pad_polys))

if len(pad_polys) == 0:
    raise RuntimeError(
        "No Layer 1 / datatype 10 electrode polygons found."
    )

# =============================================================================
# JJ POLYGONS
# =============================================================================

jj_polys = get_polygons(
    (JJ_LAYER, JJ_DATATYPE)
)

print("\nLayer 53 / datatype 0 polygons:")
print(" ", len(jj_polys))

if len(jj_polys) != 8:
    raise RuntimeError(
        f"Expected 8 JJ polygons, found {len(jj_polys)}."
    )

# =============================================================================
# MATCH JJ POLYGONS TO EXPECTED LOCATIONS
# =============================================================================

print("\n" + "=" * 80)
print("MATCHING JJ REFERENCES")
print("=" * 80)

matched_jj = []

unused = list(jj_polys)

for i, target in enumerate(JJ_TARGETS):

    best = None
    best_dist = float("inf")

    for poly in unused:

        c = poly.centroid

        d = np.hypot(
            c.x - target[0],
            c.y - target[1]
        )

        if d < best_dist:
            best_dist = d
            best = poly

    if best is None:
        raise RuntimeError(
            f"Could not match JJ{i+1}."
        )

    if best_dist > JJ_TOL:
        raise RuntimeError(
            f"JJ{i+1} mismatch: "
            f"{best_dist*1000:.3f} um"
        )

    unused.remove(best)

    matched_jj.append(best)

    c = best.centroid

    print(
        f"JJ{i+1}: "
        f"centroid=({c.x:.6f}, {c.y:.6f}) mm, "
        f"error={best_dist*1000:.3f} um, "
        f"area={best.area:.9e} mm²"
    )

# =============================================================================
# FIND TWO ACTUAL ELECTRODE PADS FOR EACH JJ
# =============================================================================

print("\n" + "=" * 80)
print("IDENTIFYING ACTUAL JJ ELECTRODE PAIRS")
print("=" * 80)

all_results = []

for i, (jj, target) in enumerate(
    zip(matched_jj, JJ_TARGETS),
    start=1
):

    x0, y0 = target

    candidates = []

    # Search only actual pad polygons near the JJ.
    for pad in pad_polys:

        minx, miny, maxx, maxy = pad.bounds

        cx = pad.centroid.x
        cy = pad.centroid.y

        if (
            abs(cx - x0) <= PAD_SEARCH_X
            and abs(cy - y0) <= PAD_SEARCH_Y
        ):
            candidates.append(pad)

    print(f"\nJJ{i}: {len(candidates)} nearby pad candidates")

    # -------------------------------------------------------------------------
    # For these JJs the electrodes are horizontal and separated in X.
    # -------------------------------------------------------------------------

    left_candidates = [
        p for p in candidates
        if p.centroid.x < x0
    ]

    right_candidates = [
        p for p in candidates
        if p.centroid.x > x0
    ]

    if len(left_candidates) == 0:
        raise RuntimeError(
            f"JJ{i}: no left electrode candidate found."
        )

    if len(right_candidates) == 0:
        raise RuntimeError(
            f"JJ{i}: no right electrode candidate found."
        )

    # Closest electrode to JJ center
    left = min(
        left_candidates,
        key=lambda p: abs(p.centroid.x - x0)
    )

    right = min(
        right_candidates,
        key=lambda p: abs(p.centroid.x - x0)
    )

    lb = left.bounds
    rb = right.bounds

    # -------------------------------------------------------------------------
    # Actual gap
    # -------------------------------------------------------------------------

    gap_left = lb[2]
    gap_right = rb[0]

    gap_bottom = max(lb[1], rb[1])
    gap_top = min(lb[3], rb[3])

    gap_width = gap_right - gap_left
    gap_height = gap_top - gap_bottom

    print("  LEFT electrode :")
    print("   ", lb)

    print("  RIGHT electrode:")
    print("   ", rb)

    print(
        f"  Actual gap width : {gap_width*1000:.3f} um"
    )

    print(
        f"  Actual gap height: {gap_height*1000:.3f} um"
    )

    # =============================================================================
    # CRITICAL CHECKS
    # =============================================================================

    if gap_width <= 0:
        raise RuntimeError(
            f"JJ{i}: electrodes overlap in X."
        )

    if gap_height <= 0:
        raise RuntimeError(
            f"JJ{i}: electrodes do not overlap in Y."
        )

    # Gap center
    gap_cx = 0.5 * (gap_left + gap_right)
    gap_cy = 0.5 * (gap_bottom + gap_top)

    center_error = np.hypot(
        gap_cx - x0,
        gap_cy - y0
    )

    print(
        f"  Gap center: "
        f"({gap_cx:.6f}, {gap_cy:.6f}) mm"
    )

    print(
        f"  Gap-center error: "
        f"{center_error*1000:.3f} um"
    )

    # -------------------------------------------------------------------------
    # Expected dimensions
    # -------------------------------------------------------------------------

    width_error = abs(
        gap_width - EXPECTED_GAP_X
    )

    height_error = abs(
        gap_height - EXPECTED_GAP_Y
    )

    if width_error > 2e-6:
        raise RuntimeError(
            f"JJ{i}: gap width is "
            f"{gap_width*1000:.3f} um, "
            f"not approximately 20 um."
        )

    if height_error > 2e-6:
        raise RuntimeError(
            f"JJ{i}: gap height is "
            f"{gap_height*1000:.3f} um, "
            f"not approximately 5 um."
        )

    if center_error > 2e-6:
        raise RuntimeError(
            f"JJ{i}: JJ is not centered in the actual electrode gap."
        )

    # =============================================================================
    # MOST IMPORTANT TEST:
    # CHECK THAT THE GAP ITSELF IS EMPTY OF MAIN METAL
    # =============================================================================

    gap_region = box(
        gap_left,
        gap_bottom,
        gap_right,
        gap_top
    )

    metal_inside_gap = main_union.intersection(
        gap_region
    ).area

    print(
        f"  Main-metal area inside gap: "
        f"{metal_inside_gap:.9e} mm²"
    )

    # Tiny floating-point intersection is acceptable.
    if metal_inside_gap > 1e-10:
        raise RuntimeError(
            f"JJ{i}: main metal exists inside the physical JJ gap."
        )

    print("  PASS: actual electrode gap verified.")

    all_results.append({
        "jj": i,
        "left": left,
        "right": right,
        "gap": gap_region,
        "gap_width": gap_width,
        "gap_height": gap_height,
        "center_error": center_error,
    })

# =============================================================================
# FINAL SUMMARY
# =============================================================================

print("\n")
print("=" * 80)
print("FINAL 8-JJ GEOMETRY VALIDATION")
print("=" * 80)

print()

for r in all_results:

    print(
        f"JJ{r['jj']}: "
        f"gap={r['gap_width']*1000:.3f} x "
        f"{r['gap_height']*1000:.3f} um, "
        f"center error={r['center_error']*1000:.3f} um"
    )

print("\n" + "=" * 80)
print("RESULT")
print("=" * 80)

print("""
PASS: All 8 JJ locations match the GDS.

PASS: All 8 JJs have two real electrode pads.

PASS: The electrode gaps are determined from the actual
      GDS electrode geometry rather than artificially subtracting
      a rectangle from the unified metal.

PASS: Main superconducting metal does not occupy the physical
      JJ gaps.

PASS: Geometry is suitable for the NEXT stage:
      constructing conformal silicon/vacuum + PEC + JJ surfaces.

IMPORTANT:
Do NOT run Palace yet.
Do NOT generate a 3D mesh yet.

The next mesh generator should use these actual gap polygons
directly as JJ surfaces.
""")



8-JJ GDS ELECTRODE/GAP GEOMETRY VALIDATION

GDS:
/content/drive/MyDrive/Palace6/meshes/quantum_chip_export_manuscript.gds

Reading GDS...
Loaded 19 cells.

Available top-level cells:
   TOP
   FakeJunction_02
   FakeJunction_01

Using chip cell:
  TOP_main_1

Extracting polygons from TOP_main_1...
Number of layer/datatype specifications: 7

Main metal polygons:
  2450
Main metal geometry: Polygon
Main metal area: 185.750736791720 mm²

Layer 1 / datatype 10 polygons:
  90

Layer 53 / datatype 0 polygons:
  8

MATCHING JJ REFERENCES
JJ1: centroid=(-1.132500, 0.000000) mm, error=0.000 um, area=9.000000000e-05 mm²
JJ2: centroid=(-0.867500, -0.000000) mm, error=0.000 um, area=9.000000000e-05 mm²
JJ3: centroid=(1.267500, 0.000000) mm, error=0.000 um, area=9.000000000e-05 mm²
JJ4: centroid=(1.532500, -0.000000) mm, error=0.000 um, area=9.000000000e-05 mm²
JJ5: centroid=(-1.132500, -2.700000) mm, error=0.000 um, area=9.000000000e-05 mm²
JJ6: centroid=(-0.867500, -2.700000) mm, error=0.000 um, 

In [ ]:

# =============================================================================
# ROBUST 8-JJ CONFORMAL PALACE MESH - V12
# =============================================================================
#
# IMPORTANT:
#   This version uses the ACTUAL EMPTY JJ GAPS found in the GDS.
#
#   It does NOT:
#       - subtract a fake JJ rectangle from unified metal
#       - modify the metal geometry using JJ references
#       - run Palace
#
#   It creates:
#       - silicon substrate
#       - vacuum
#       - superconducting PEC surfaces
#       - 8 physical JJ surfaces occupying the real 20 x 5 um gaps
#
#   RAM SAFETY:
#       - coarse bulk mesh
#       - local JJ refinement
#       - hard tetrahedron safety limit
#       - no Palace execution
#
# =============================================================================

import os
import gc
import gdspy
import gmsh
import numpy as np

from shapely.geometry import Polygon
from shapely.ops import unary_union

# =============================================================================
# PATHS
# =============================================================================

GDS_FILE = (
    "/content/drive/MyDrive/Palace6/meshes/"
    "quantum_chip_export_manuscript.gds"
)

OUTPUT_MESH = (
    "/content/drive/MyDrive/Palace6/meshes/"
    "quantum_chip_mesh_8JJ_v12.msh"
)

# =============================================================================
# GDS
# =============================================================================

TOP_CELL_NAME = "TOP_main_1"

MAIN_LAYER = 1
MAIN_DATATYPE = 0

PAD_LAYER = 1
PAD_DATATYPE = 10

JJ_LAYER = 53
JJ_DATATYPE = 0

# =============================================================================
# JJ LOCATIONS
# =============================================================================

JJ_TARGETS = np.array([
    [-1.132500,  0.000000],
    [-0.867500,  0.000000],
    [ 1.267500,  0.000000],
    [ 1.532500,  0.000000],
    [-1.132500, -2.700000],
    [-0.867500, -2.700000],
    [ 1.267500, -2.700000],
    [ 1.532500, -2.700000],
])

# =============================================================================
# GEOMETRY / MESH PARAMETERS
# =============================================================================
#
# Units here are mm because GDS geometry is in mm.
#
# 500 um bulk = 0.500 mm
# 50  um JJ   = 0.050 mm
#
# This is intentionally coarse because your previous 13.9M tetrahedral mesh
# required ~8 GB just to reach the eigenvalue stage and was killed by Colab.
# =============================================================================

BULK_SIZE = 0.500       # mm = 500 um
JJ_SIZE   = 0.050       # mm = 50 um

# Substrate/vacuum dimensions
CHIP_X = 14.0           # mm
CHIP_Y = 14.0           # mm

SUBSTRATE_THICKNESS = 0.500   # mm
VACUUM_HEIGHT       = 0.500   # mm

Z_INTERFACE = 0.0

# Hard safety limit
MAX_TETS = 4_000_000

# =============================================================================
# TOLERANCES
# =============================================================================

JJ_MATCH_TOL = 0.005     # mm = 5 um
AREA_TOL = 1e-10

# =============================================================================
# HELPERS
# =============================================================================

def polygon_list_from_spec(spec_dict, spec):
    """Convert gdspy polygon arrays to valid Shapely polygons."""

    arr = spec_dict.get(spec, [])

    result = []

    for p in arr:

        p = np.asarray(p, dtype=float)

        if len(p) < 3:
            continue

        try:
            shp = Polygon(p)

            if not shp.is_valid:
                shp = shp.buffer(0)

            if shp.is_empty:
                continue

            if shp.area <= AREA_TOL:
                continue

            result.append(shp)

        except Exception:
            continue

    return result


def shapely_to_loops(geom):
    """
    Convert Polygon/MultiPolygon to exterior coordinate loops.
    Coordinates are in mm.
    """

    loops = []

    if geom.is_empty:
        return loops

    if geom.geom_type == "Polygon":

        loops.append(
            np.asarray(
                geom.exterior.coords,
                dtype=float
            )
        )

    elif geom.geom_type == "MultiPolygon":

        for poly in geom.geoms:

            if poly.is_empty:
                continue

            loops.append(
                np.asarray(
                    poly.exterior.coords,
                    dtype=float
                )
            )

    else:
        raise RuntimeError(
            f"Unsupported geometry type: {geom.geom_type}"
        )

    return loops


def add_planar_surface_from_loop(points, z=0.0):

    point_tags = []

    for x, y in points:

        point_tags.append(
            gmsh.model.geo.addPoint(
                float(x),
                float(y),
                float(z)
            )
        )

    line_tags = []

    for i in range(len(point_tags) - 1):

        a = point_tags[i]
        b = point_tags[i + 1]

        if a == b:
            continue

        line_tags.append(
            gmsh.model.geo.addLine(a, b)
        )

    if len(line_tags) < 3:
        raise RuntimeError(
            "Could not create a valid polygon boundary."
        )

    curve_loop = gmsh.model.geo.addCurveLoop(
        line_tags
    )

    surface = gmsh.model.geo.addPlaneSurface(
        [curve_loop]
    )

    return surface


# =============================================================================
# HEADER
# =============================================================================

print("=" * 80)
print("ROBUST 8-JJ CONFORMAL PALACE MESH - V12")
print("=" * 80)

print("\nGDS:")
print(GDS_FILE)

print("\nOutput:")
print(OUTPUT_MESH)

print("\nMesh parameters:")
print(f"  Bulk = {BULK_SIZE*1000:.0f} um")
print(f"  JJ   = {JJ_SIZE*1000:.0f} um")

# =============================================================================
# READ GDS
# =============================================================================

print("\n" + "=" * 80)
print("READING GDS")
print("=" * 80)

gds = gdspy.GdsLibrary(
    infile=GDS_FILE
)

print(
    f"Loaded {len(gds.cells)} cells."
)

if TOP_CELL_NAME not in gds.cells:
    raise RuntimeError(
        f"Required cell {TOP_CELL_NAME} not found."
    )

top = gds.cells[TOP_CELL_NAME]

print(
    f"Using top cell: {top.name}"
)

# =============================================================================
# EXTRACT TOP-CELL POLYGONS
# =============================================================================

specs = top.get_polygons(
    by_spec=True,
    depth=None
)

# =============================================================================
# MAIN METAL
# =============================================================================

print("\nExtracting main superconducting metal...")

main_polys = polygon_list_from_spec(
    specs,
    (MAIN_LAYER, MAIN_DATATYPE)
)

print(
    f"Main metal polygons: {len(main_polys)}"
)

if not main_polys:
    raise RuntimeError(
        "No main metal polygons found."
    )

print("Uniting main metal...")

main_metal = unary_union(
    main_polys
)

print(
    "Main metal geometry:",
    main_metal.geom_type
)

print(
    f"Main metal area: "
    f"{main_metal.area:.12f} mm²"
)

# =============================================================================
# JJ REFERENCES
# =============================================================================

print("\nExtracting JJ reference polygons...")

jj_polys = polygon_list_from_spec(
    specs,
    (JJ_LAYER, JJ_DATATYPE)
)

print(
    f"JJ polygons found: {len(jj_polys)}"
)

if len(jj_polys) != 8:
    raise RuntimeError(
        f"Expected 8 JJ polygons, found {len(jj_polys)}."
    )

# =============================================================================
# MATCH JJ REFERENCES
# =============================================================================

print("\nMatching JJ references...")

unused_jj = list(jj_polys)
matched_jj = []

for i, target in enumerate(
    JJ_TARGETS,
    start=1
):

    best = None
    best_distance = float("inf")

    for poly in unused_jj:

        c = poly.centroid

        d = np.hypot(
            c.x - target[0],
            c.y - target[1]
        )

        if d < best_distance:

            best_distance = d
            best = poly

    if best is None:
        raise RuntimeError(
            f"JJ{i} could not be matched."
        )

    if best_distance > JJ_MATCH_TOL:

        raise RuntimeError(
            f"JJ{i} mismatch: "
            f"{best_distance*1000:.3f} um"
        )

    unused_jj.remove(best)
    matched_jj.append(best)

    c = best.centroid

    print(
        f"JJ{i}: "
        f"centroid=({c.x:.6f}, {c.y:.6f}) mm, "
        f"error={best_distance*1000:.3f} um"
    )

# =============================================================================
# EXTRACT ACTUAL ELECTRODE PADS
# =============================================================================

print("\nExtracting actual JJ electrode pads...")

pad_polys = polygon_list_from_spec(
    specs,
    (PAD_LAYER, PAD_DATATYPE)
)

print(
    f"Pad polygons: {len(pad_polys)}"
)

if len(pad_polys) == 0:
    raise RuntimeError(
        "No Layer 1 / datatype 10 pads found."
    )

# =============================================================================
# IDENTIFY ACTUAL JJ GAPS
# =============================================================================

print("\n" + "=" * 80)
print("IDENTIFYING ACTUAL JJ GAPS")
print("=" * 80)

jj_gaps = []

for i, target in enumerate(
    JJ_TARGETS,
    start=1
):

    x0, y0 = target

    nearby = []

    for pad in pad_polys:

        cx = pad.centroid.x
        cy = pad.centroid.y

        if (
            abs(cx - x0) < 0.060
            and
            abs(cy - y0) < 0.020
        ):
            nearby.append(pad)

    left_candidates = [
        p for p in nearby
        if p.centroid.x < x0
    ]

    right_candidates = [
        p for p in nearby
        if p.centroid.x > x0
    ]

    if not left_candidates or not right_candidates:

        raise RuntimeError(
            f"JJ{i}: could not find both electrodes."
        )

    left = min(
        left_candidates,
        key=lambda p:
        abs(p.centroid.x - x0)
    )

    right = min(
        right_candidates,
        key=lambda p:
        abs(p.centroid.x - x0)
    )

    lb = left.bounds
    rb = right.bounds

    gap_left = lb[2]
    gap_right = rb[0]

    gap_bottom = max(
        lb[1],
        rb[1]
    )

    gap_top = min(
        lb[3],
        rb[3]
    )

    gap_width = gap_right - gap_left
    gap_height = gap_top - gap_bottom

    gap_cx = 0.5 * (
        gap_left + gap_right
    )

    gap_cy = 0.5 * (
        gap_bottom + gap_top
    )

    center_error = np.hypot(
        gap_cx - x0,
        gap_cy - y0
    )

    print(
        f"JJ{i}: "
        f"{gap_width*1000:.3f} x "
        f"{gap_height*1000:.3f} um, "
        f"center error="
        f"{center_error*1000:.3f} um"
    )

    if gap_width <= 0 or gap_height <= 0:

        raise RuntimeError(
            f"JJ{i}: invalid physical gap."
        )

    if abs(gap_width - 0.020) > 2e-6:

        raise RuntimeError(
            f"JJ{i}: unexpected gap width."
        )

    if abs(gap_height - 0.005) > 2e-6:

        raise RuntimeError(
            f"JJ{i}: unexpected gap height."
        )

    if center_error > 2e-6:

        raise RuntimeError(
            f"JJ{i}: gap is not centered on JJ."
        )

    # The JJ is the actual EMPTY region.
    jj_gap = Polygon([
        (gap_left,  gap_bottom),
        (gap_right, gap_bottom),
        (gap_right, gap_top),
        (gap_left,  gap_top),
    ])

    jj_gaps.append(jj_gap)

print(
    "\nPASS: all 8 physical JJ gaps recovered."
)

# =============================================================================
# IMPORTANT:
# WE DO NOT SUBTRACT JJ GAPS FROM MAIN METAL.
# THE GAPS ARE ALREADY EMPTY IN THE GDS.
# =============================================================================

print("\nNo artificial JJ subtraction will be performed.")

# =============================================================================
# INITIALIZE GMSH
# =============================================================================

print("\n" + "=" * 80)
print("INITIALIZING GMSH")
print("=" * 80)

gmsh.initialize()

try:

    gmsh.option.setNumber(
        "General.Terminal",
        1
    )

    gmsh.model.add(
        "palace_8JJ_v12"
    )

    # =========================================================================
    # CREATE SILICON + VACUUM
    # =========================================================================

    print("\nCreating silicon substrate...")

    sx = CHIP_X / 2
    sy = CHIP_Y / 2

    silicon = gmsh.model.occ.addBox(
        -sx,
        -sy,
        -SUBSTRATE_THICKNESS,
        CHIP_X,
        CHIP_Y,
        SUBSTRATE_THICKNESS
    )

    print("Creating vacuum...")

    vacuum = gmsh.model.occ.addBox(
        -sx,
        -sy,
        0.0,
        CHIP_X,
        CHIP_Y,
        VACUUM_HEIGHT
    )

    # =========================================================================
    # CREATE METAL SURFACES
    # =========================================================================

    print("\nCreating superconducting metal surfaces...")

    metal_loops = shapely_to_loops(
        main_metal
    )

    metal_surfaces = []

    # Use OCC geometry for robust surface construction.
    for loop in metal_loops:

        if len(loop) < 4:
            continue

        pts = []

        for x, y in loop:

            pts.append(
                gmsh.model.occ.addPoint(
                    float(x),
                    float(y),
                    0.0
                )
            )

        lines = []

        for k in range(len(pts) - 1):

            lines.append(
                gmsh.model.occ.addLine(
                    pts[k],
                    pts[k + 1]
                )
            )

        if len(lines) < 3:
            continue

        cl = gmsh.model.occ.addCurveLoop(
            lines
        )

        sf = gmsh.model.occ.addPlaneSurface(
            [cl]
        )

        metal_surfaces.append(sf)

    print(
        f"Metal surfaces created: "
        f"{len(metal_surfaces)}"
    )

    if not metal_surfaces:
        raise RuntimeError(
            "No superconducting metal surface created."
        )

    # =========================================================================
    # CREATE JJ SURFACES
    # =========================================================================

    print("\nCreating physical JJ surfaces...")

    jj_surface_tags = []

    for i, gap in enumerate(
        jj_gaps,
        start=1
    ):

        coords = np.asarray(
            gap.exterior.coords,
            dtype=float
        )

        pts = []

        for x, y in coords:

            pts.append(
                gmsh.model.occ.addPoint(
                    float(x),
                    float(y),
                    0.0
                )
            )

        lines = []

        for k in range(len(pts) - 1):

            lines.append(
                gmsh.model.occ.addLine(
                    pts[k],
                    pts[k + 1]
                )
            )

        cl = gmsh.model.occ.addCurveLoop(
            lines
        )

        sf = gmsh.model.occ.addPlaneSurface(
            [cl]
        )

        jj_surface_tags.append(sf)

        print(
            f"JJ{i}: surface={sf}"
        )

    # =========================================================================
    # SYNCHRONIZE
    # =========================================================================

    print("\nSynchronizing OCC...")

    gmsh.model.occ.synchronize()

    # =========================================================================
    # FRAGMENT SILICON/VACUUM WITH INTERFACE GEOMETRY
    # =========================================================================
    #
    # We fragment only the 3D substrate/vacuum volumes.
    # Metal and JJ surfaces remain explicit 2-D entities.
    # This avoids the previous aggressive fragmentation failures.
    # =========================================================================

    print(
        "\nFragmenting silicon/vacuum volumes..."
    )

    objects = [
        (3, silicon),
        (3, vacuum)
    ]

    fragmented, _ = gmsh.model.occ.fragment(
        objects,
        []
    )

    gmsh.model.occ.synchronize()

    # =========================================================================
    # IDENTIFY 3D VOLUMES
    # =========================================================================

    volumes = gmsh.model.getEntities(
        3
    )

    print(
        f"\n3D volumes found: {len(volumes)}"
    )

    if len(volumes) != 2:

        raise RuntimeError(
            f"Expected 2 volumes, found {len(volumes)}."
        )

    silicon_volumes = []
    vacuum_volumes = []

    for dim, tag in volumes:

        com = gmsh.model.occ.getCenterOfMass(
            dim,
            tag
        )

        if com[2] < 0:

            silicon_volumes.append(tag)

        elif com[2] > 0:

            vacuum_volumes.append(tag)

    print(
        "Silicon volumes:",
        silicon_volumes
    )

    print(
        "Vacuum volumes:",
        vacuum_volumes
    )

    if len(silicon_volumes) != 1:

        raise RuntimeError(
            "Could not uniquely identify silicon."
        )

    if len(vacuum_volumes) != 1:

        raise RuntimeError(
            "Could not uniquely identify vacuum."
        )

    # =========================================================================
    # PHYSICAL GROUPS
    # =========================================================================

    print(
        "\nCreating physical groups..."
    )

    silicon_group = gmsh.model.addPhysicalGroup(
        3,
        silicon_volumes,
        2
    )

    gmsh.model.setPhysicalName(
        3,
        silicon_group,
        "Silicon_Substrate"
    )

    vacuum_group = gmsh.model.addPhysicalGroup(
        3,
        vacuum_volumes,
        1
    )

    gmsh.model.setPhysicalName(
        3,
        vacuum_group,
        "Vacuum"
    )

    # -------------------------------------------------------------------------
    # PEC group
    # -------------------------------------------------------------------------

    pec_group = gmsh.model.addPhysicalGroup(
        2,
        metal_surfaces,
        10
    )

    gmsh.model.setPhysicalName(
        2,
        pec_group,
        "Superconducting_Metal"
    )

    # -------------------------------------------------------------------------
    # JJ groups
    # -------------------------------------------------------------------------

    for i, sf in enumerate(
        jj_surface_tags,
        start=1
    ):

        tag = 19 + i

        group = gmsh.model.addPhysicalGroup(
            2,
            [sf],
            tag
        )

        gmsh.model.setPhysicalName(
            2,
            group,
            f"JJ{i}"
        )

    # =========================================================================
    # MESH CONTROL
    # =========================================================================

    print(
        "\nConfiguring coarse mesh..."
    )

    gmsh.option.setNumber(
        "Mesh.MeshSizeMin",
        JJ_SIZE
    )

    gmsh.option.setNumber(
        "Mesh.MeshSizeMax",
        BULK_SIZE
    )

    gmsh.option.setNumber(
        "Mesh.Algorithm3D",
        1
    )

    gmsh.option.setNumber(
        "Mesh.Optimize",
        1
    )

    gmsh.option.setNumber(
        "Mesh.OptimizeNetgen",
        0
    )

    # =========================================================================
    # JJ LOCAL REFINEMENT
    # =========================================================================

    print(
        "\nApplying local JJ refinement..."
    )

    # Distance fields around the 8 actual JJ surfaces.
    #
    # We deliberately use a modest 50 um size.
    # No 20 um or smaller refinement is used here.

    jj_field_ids = []

    for sf in jj_surface_tags:

        distance_field = (
            gmsh.model.mesh.field.add(
                "Distance"
            )
        )

        gmsh.model.mesh.field.setNumbers(
            distance_field,
            "FacesList",
            [sf]
        )

        jj_field_ids.append(
            distance_field
        )

    if jj_field_ids:

        threshold_fields = []

        for f in jj_field_ids:

            tf = (
                gmsh.model.mesh.field.add(
                    "Threshold"
                )
            )

            gmsh.model.mesh.field.setNumber(
                tf,
                "InField",
                f
            )

            gmsh.model.mesh.field.setNumber(
                tf,
                "SizeMin",
                JJ_SIZE
            )

            gmsh.model.mesh.field.setNumber(
                tf,
                "SizeMax",
                BULK_SIZE
            )

            gmsh.model.mesh.field.setNumber(
                tf,
                "DistMin",
                0.05
            )

            gmsh.model.mesh.field.setNumber(
                tf,
                "DistMax",
                0.20
            )

            threshold_fields.append(tf)

        min_field = (
            gmsh.model.mesh.field.add(
                "Min"
            )
        )

        gmsh.model.mesh.field.setNumbers(
            min_field,
            "FieldsList",
            threshold_fields
        )

        gmsh.model.mesh.field.setAsBackgroundMesh(
            min_field
        )

    # =========================================================================
    # GENERATE 3D MESH
    # =========================================================================

    print("\n" + "=" * 80)
    print("GENERATING COARSE 3D MESH")
    print("=" * 80)

    print(
        f"Bulk size : {BULK_SIZE*1000:.0f} um"
    )

    print(
        f"JJ size   : {JJ_SIZE*1000:.0f} um"
    )

    print(
        f"Safety limit: {MAX_TETS:,} tetrahedra"
    )

    gmsh.model.mesh.generate(
        3
    )

    # =========================================================================
    # MESH STATISTICS
    # =========================================================================

    print("\nChecking mesh...")

    node_tags, node_coords, _ = (
        gmsh.model.mesh.getNodes()
    )

    element_types, element_tags, element_nodes = (
        gmsh.model.mesh.getElements(3)
    )

    tetra_count = 0

    for etype, tags in zip(
        element_types,
        element_tags
    ):

        # Gmsh type 4 = 4-node tetrahedron
        if etype == 4:

            tetra_count += len(tags)

    print(
        f"\nNodes      : {len(node_tags):,}"
    )

    print(
        f"Tetrahedra : {tetra_count:,}"
    )

    # =========================================================================
    # HARD RAM SAFETY CHECK
    # =========================================================================

    if tetra_count > MAX_TETS:

        print(
            "\n" + "=" * 80
        )

        print(
            "SAFETY STOP"
        )

        print(
            "=" * 80
        )

        print(
            f"Generated {tetra_count:,} tetrahedra."
        )

        print(
            f"Limit is {MAX_TETS:,}."
        )

        print(
            "\nMesh will NOT be written."
        )

        print(
            "Reduce refinement before continuing."
        )

        raise RuntimeError(
            "Mesh exceeds RAM safety limit."
        )

    # =========================================================================
    # WRITE MSH 2.2
    # =========================================================================

    print(
        "\nWriting MSH 2.2..."
    )

    gmsh.option.setNumber(
        "Mesh.MshFileVersion",
        2.2
    )

    gmsh.write(
        OUTPUT_MESH
    )

    # =========================================================================
    # FINAL PHYSICAL GROUP REPORT
    # =========================================================================

    print("\n" + "=" * 80)
    print("PHYSICAL GROUPS")
    print("=" * 80)

    for dim, tag in gmsh.model.getPhysicalGroups():

        name = gmsh.model.getPhysicalName(
            dim,
            tag
        )

        print(
            f"dim={dim}, "
            f"id={tag}, "
            f"name={name}"
        )

    # =========================================================================
    # FILE SIZE
    # =========================================================================

    if os.path.exists(OUTPUT_MESH):

        size_gb = (
            os.path.getsize(OUTPUT_MESH)
            / (1024**3)
        )

        print(
            f"\nMesh file size: "
            f"{size_gb:.3f} GB"
        )

    print("\n" + "=" * 80)
    print("MESH CREATED SUCCESSFULLY")
    print("=" * 80)

    print(
        f"\nMesh:\n{OUTPUT_MESH}"
    )

    print(
        "\nIMPORTANT:"
    )

    print(
        "Do NOT run Palace yet."
    )

    print(
        "First inspect the mesh statistics and "
        "Palace physical-group compatibility."
    )

finally:

    gmsh.finalize()

    # Release Python/GDS memory
    del gds
    del main_polys
    del pad_polys
    del jj_polys
    del main_metal

    gc.collect()



ROBUST 8-JJ CONFORMAL PALACE MESH - V12

GDS:
/content/drive/MyDrive/Palace6/meshes/quantum_chip_export_manuscript.gds

Output:
/content/drive/MyDrive/Palace6/meshes/quantum_chip_mesh_8JJ_v12.msh

Mesh parameters:
  Bulk = 500 um
  JJ   = 50 um

READING GDS
Loaded 19 cells.
Using top cell: TOP_main_1

Extracting main superconducting metal...
Main metal polygons: 2450
Uniting main metal...
Main metal geometry: Polygon
Main metal area: 185.750736791720 mm²

Extracting JJ reference polygons...
JJ polygons found: 8

Matching JJ references...
JJ1: centroid=(-1.132500, 0.000000) mm, error=0.000 um
JJ2: centroid=(-0.867500, -0.000000) mm, error=0.000 um
JJ3: centroid=(1.267500, 0.000000) mm, error=0.000 um
JJ4: centroid=(1.532500, -0.000000) mm, error=0.000 um
JJ5: centroid=(-1.132500, -2.700000) mm, error=0.000 um
JJ6: centroid=(-0.867500, -2.700000) mm, error=0.000 um
JJ7: centroid=(1.267500, -2.700000) mm, error=0.000 um
JJ8: centroid=(1.532500, -2.700000) mm, error=0.000 um

Extracting ac

In [ ]:
# =============================================================================
# PALACE V12 — SAFE MESH TOPOLOGY / PHYSICAL-GROUP TEST
# =============================================================================
#
# This script:
#   1. Reads the existing V12 MSH only
#   2. Counts volume tetrahedra
#   3. Counts surface triangles for every physical group
#   4. Verifies all 8 JJ groups
#   5. Verifies Superconducting_Metal
#   6. Checks whether physical surface elements exist
#   7. Checks whether JJ surfaces are geometrically located at z = 0
#
# NO Palace execution.
# NO mesh generation.
# NO large RAM allocation.
#
# =============================================================================

import os
import gc
import gmsh
import numpy as np

MESH = (
    "/content/drive/MyDrive/Palace6/meshes/"
    "quantum_chip_mesh_8JJ_v12.msh"
)

print("=" * 80)
print("PALACE V12 — SAFE MESH TOPOLOGY TEST")
print("=" * 80)

print("\nMesh:")
print(MESH)

if not os.path.exists(MESH):
    raise RuntimeError("Mesh file does not exist.")

size_mb = os.path.getsize(MESH) / (1024**2)

print(
    f"Mesh size: {size_mb:.3f} MB"
)

# =============================================================================
# INITIALIZE
# =============================================================================

gmsh.initialize()

try:

    gmsh.option.setNumber(
        "General.Terminal",
        1
    )

    gmsh.open(MESH)

    print("\nPASS: Gmsh opened the mesh.")

    # =========================================================================
    # BASIC MESH COUNTS
    # =========================================================================

    node_tags, node_coords, _ = (
        gmsh.model.mesh.getNodes()
    )

    print("\n" + "=" * 80)
    print("GLOBAL MESH")
    print("=" * 80)

    print(
        f"Nodes: {len(node_tags):,}"
    )

    # =========================================================================
    # ELEMENT COUNTS
    # =========================================================================

    print("\n" + "=" * 80)
    print("ELEMENT TYPES")
    print("=" * 80)

    types, tags, nodes = (
        gmsh.model.mesh.getElements()
    )

    total_tets = 0
    total_triangles = 0

    element_names = {
        1: "2-node line",
        2: "3-node triangle",
        3: "4-node quadrilateral",
        4: "4-node tetrahedron",
        5: "8-node hexahedron",
        9: "6-node second-order triangle",
        10: "9-node second-order quadrilateral",
        11: "10-node second-order tetrahedron",
    }

    for etype, etags in zip(
        types,
        tags
    ):

        count = len(etags)

        name = element_names.get(
            etype,
            f"type {etype}"
        )

        print(
            f"{etype:3d} : "
            f"{name:30s} "
            f"{count:,}"
        )

        if etype == 4:
            total_tets += count

        if etype == 2:
            total_triangles += count

    print(
        f"\n4-node tetrahedra: "
        f"{total_tets:,}"
    )

    print(
        f"3-node triangles: "
        f"{total_triangles:,}"
    )

    if total_tets == 0:
        raise RuntimeError(
            "CRITICAL: No tetrahedral volume elements found."
        )

    # =========================================================================
    # PHYSICAL GROUPS
    # =========================================================================

    print("\n" + "=" * 80)
    print("PHYSICAL GROUPS")
    print("=" * 80)

    physical_groups = {}

    for dim, tag in gmsh.model.getPhysicalGroups():

        name = gmsh.model.getPhysicalName(
            dim,
            tag
        )

        entities = gmsh.model.getEntitiesForPhysicalGroup(
            dim,
            tag
        )

        physical_groups[name] = (
            dim,
            tag,
            list(entities)
        )

        print(
            f"dim={dim} "
            f"id={tag:2d} "
            f"name={name:25s} "
            f"entities={len(entities)}"
        )

    # =========================================================================
    # REQUIRED VOLUME GROUPS
    # =========================================================================

    print("\n" + "=" * 80)
    print("VOLUME GROUP CHECK")
    print("=" * 80)

    required_volumes = [
        "Vacuum",
        "Silicon_Substrate",
    ]

    for name in required_volumes:

        if name not in physical_groups:
            raise RuntimeError(
                f"FAIL: Missing physical volume group: {name}"
            )

        dim, tag, entities = physical_groups[name]

        if dim != 3:
            raise RuntimeError(
                f"FAIL: {name} is not dimension 3."
            )

        count = 0

        for entity in entities:

            etypes, etags, _ = (
                gmsh.model.mesh.getElements(
                    3,
                    entity
                )
            )

            for etype, arr in zip(
                etypes,
                etags
            ):

                if etype == 4:
                    count += len(arr)

        print(
            f"PASS: {name:25s} "
            f"tetrahedra={count:,}"
        )

        if count == 0:
            raise RuntimeError(
                f"FAIL: {name} contains no tetrahedra."
            )

    # =========================================================================
    # PEC CHECK
    # =========================================================================

    print("\n" + "=" * 80)
    print("PEC SURFACE CHECK")
    print("=" * 80)

    if "Superconducting_Metal" not in physical_groups:

        raise RuntimeError(
            "FAIL: Superconducting_Metal group missing."
        )

    dim, tag, entities = (
        physical_groups[
            "Superconducting_Metal"
        ]
    )

    if dim != 2:

        raise RuntimeError(
            "FAIL: Superconducting_Metal "
            "is not a surface group."
        )

    pec_triangles = 0

    for entity in entities:

        etypes, etags, _ = (
            gmsh.model.mesh.getElements(
                2,
                entity
            )
        )

        for etype, arr in zip(
            etypes,
            etags
        ):

            if etype == 2:
                pec_triangles += len(arr)

    print(
        f"PEC entities   : {len(entities)}"
    )

    print(
        f"PEC triangles  : {pec_triangles:,}"
    )

    if pec_triangles == 0:

        raise RuntimeError(
            "FAIL: Superconducting_Metal "
            "contains no triangular surface elements."
        )

    print(
        "PASS: PEC surface elements exist."
    )

    # =========================================================================
    # JJ CHECK
    # =========================================================================

    print("\n" + "=" * 80)
    print("8-JJ SURFACE CHECK")
    print("=" * 80)

    jj_triangle_counts = {}

    for i in range(1, 9):

        name = f"JJ{i}"

        if name not in physical_groups:

            raise RuntimeError(
                f"FAIL: Missing physical group {name}."
            )

        dim, tag, entities = (
            physical_groups[name]
        )

        if dim != 2:

            raise RuntimeError(
                f"FAIL: {name} is not dimension 2."
            )

        triangles = 0

        for entity in entities:

            etypes, etags, _ = (
                gmsh.model.mesh.getElements(
                    2,
                    entity
                )
            )

            for etype, arr in zip(
                etypes,
                etags
            ):

                if etype == 2:
                    triangles += len(arr)

        jj_triangle_counts[name] = triangles

        print(
            f"{name}: "
            f"entities={len(entities)}, "
            f"triangles={triangles:,}"
        )

        if triangles == 0:

            raise RuntimeError(
                f"FAIL: {name} has no surface triangles."
            )

    print(
        "\nPASS: all 8 JJ groups contain "
        "surface elements."
    )

    # =========================================================================
    # JJ Z-LOCATION TEST
    # =========================================================================

    print("\n" + "=" * 80)
    print("JJ GEOMETRIC LOCATION CHECK")
    print("=" * 80)

    for i in range(1, 9):

        name = f"JJ{i}"

        dim, tag, entities = (
            physical_groups[name]
        )

        all_z = []

        for entity in entities:

            ntags, coords, _ = (
                gmsh.model.mesh.getNodes(
                    2,
                    entity,
                    includeBoundary=True
                )
            )

            if len(coords) == 0:
                continue

            coords = np.asarray(
                coords
            ).reshape(
                -1,
                3
            )

            all_z.extend(
                coords[:, 2]
            )

        if not all_z:

            raise RuntimeError(
                f"FAIL: Could not obtain "
                f"nodes for {name}."
            )

        all_z = np.asarray(
            all_z
        )

        max_abs_z = np.max(
            np.abs(all_z)
        )

        print(
            f"{name}: "
            f"max |z| = "
            f"{max_abs_z*1e6:.3f} um"
        )

        if max_abs_z > 1e-9:

            print(
                f"WARNING: {name} is not exactly "
                f"on z=0."
            )

    # =========================================================================
    # FINAL DECISION
    # =========================================================================

    print("\n" + "=" * 80)
    print("FINAL RESULT")
    print("=" * 80)

    print(
        "\nPASS: Mesh can be read by Gmsh."
    )

    print(
        "PASS: Volume tetrahedra exist."
    )

    print(
        "PASS: Vacuum physical group exists."
    )

    print(
        "PASS: Silicon_Substrate physical group exists."
    )

    print(
        "PASS: Superconducting_Metal surface elements exist."
    )

    print(
        "PASS: JJ1-JJ8 physical surface elements exist."
    )

    print(
        "\nDO NOT RUN THE FULL PALACE SIMULATION YET."
    )

    print(
        "If this test passes, the next test should be "
        "a very small Palace mesh-read test."
    )

finally:

    gmsh.finalize()

    gc.collect()

PALACE V12 — SAFE MESH TOPOLOGY TEST

Mesh:
/content/drive/MyDrive/Palace6/meshes/quantum_chip_mesh_8JJ_v12.msh
Mesh size: 1.204 MB

PASS: Gmsh opened the mesh.

GLOBAL MESH
Nodes: 10,362

ELEMENT TYPES
  2 : 3-node triangle                13,970
  4 : 4-node tetrahedron             11,925

4-node tetrahedra: 11,925
3-node triangles: 13,970

PHYSICAL GROUPS
dim=2 id=10 name=Superconducting_Metal     entities=1
dim=2 id=20 name=JJ1                       entities=1
dim=2 id=21 name=JJ2                       entities=1
dim=2 id=22 name=JJ3                       entities=1
dim=2 id=23 name=JJ4                       entities=1
dim=2 id=24 name=JJ5                       entities=1
dim=2 id=25 name=JJ6                       entities=1
dim=2 id=26 name=JJ7                       entities=1
dim=2 id=27 name=JJ8                       entities=1
dim=3 id= 1 name=Vacuum                    entities=1
dim=3 id= 2 name=Silicon_Substrate         entities=1

VOLUME GROUP CHECK
PASS: Vacuum              

In [ ]:
# =============================================================================
# V12 — MFEM STABLE3D CONNECTIVITY DIAGNOSTIC
# =============================================================================
#
# Purpose:
#   Find problematic tetrahedral connectivity that can trigger
#   MFEM::STable3D::operator().
#
# Palace is NOT executed.
# No mesh generation.
#
# =============================================================================

import gmsh
import numpy as np
from collections import defaultdict
import os
import gc

MESH = (
    "/content/drive/MyDrive/Palace6/meshes/"
    "quantum_chip_mesh_8JJ_v12.msh"
)

print("=" * 80)
print("V12 — TETRAHEDRAL CONNECTIVITY DIAGNOSTIC")
print("=" * 80)

print("\nMesh:")
print(MESH)

if not os.path.exists(MESH):
    raise RuntimeError("Mesh file not found.")

gmsh.initialize()

try:

    gmsh.option.setNumber(
        "General.Terminal",
        1
    )

    gmsh.open(MESH)

    # -------------------------------------------------------------------------
    # GET TETRAHEDRA
    # -------------------------------------------------------------------------

    types, tags, node_blocks = (
        gmsh.model.mesh.getElements(
            3
        )
    )

    tet_nodes = None
    tet_tags = None

    for etype, etags, enodes in zip(
        types,
        tags,
        node_blocks
    ):

        # Gmsh type 4 = 4-node tetrahedron
        if etype == 4:

            tet_tags = np.asarray(
                etags,
                dtype=np.int64
            )

            tet_nodes = np.asarray(
                enodes,
                dtype=np.int64
            ).reshape(
                -1,
                4
            )

            break

    if tet_nodes is None:

        raise RuntimeError(
            "No 4-node tetrahedra found."
        )

    print(
        f"\nTetrahedra: {len(tet_nodes):,}"
    )

    # -------------------------------------------------------------------------
    # CHECK REPEATED VERTICES
    # -------------------------------------------------------------------------

    print("\n" + "=" * 80)
    print("CHECK 1 — REPEATED VERTICES")
    print("=" * 80)

    repeated = []

    for i, tet in enumerate(tet_nodes):

        if len(set(tet)) != 4:

            repeated.append(
                (
                    int(tet_tags[i]),
                    tet.tolist()
                )
            )

    print(
        f"Degenerate/repeated-vertex tetrahedra: "
        f"{len(repeated)}"
    )

    if repeated:

        for item in repeated[:20]:

            print(
                "BAD:",
                item
            )

        raise RuntimeError(
            "Repeated vertex indices found."
        )

    print(
        "PASS: no tetrahedron contains "
        "repeated vertex indices."
    )

    # -------------------------------------------------------------------------
    # CHECK DUPLICATE TETRAHEDRA
    # -------------------------------------------------------------------------

    print("\n" + "=" * 80)
    print("CHECK 2 — DUPLICATE TETRAHEDRA")
    print("=" * 80)

    canonical = np.sort(
        tet_nodes,
        axis=1
    )

    unique_tets, inverse, counts = np.unique(
        canonical,
        axis=0,
        return_inverse=True,
        return_counts=True
    )

    duplicate_groups = np.where(
        counts > 1
    )[0]

    print(
        f"Unique tetrahedra : {len(unique_tets):,}"
    )

    print(
        f"Duplicate groups  : {len(duplicate_groups):,}"
    )

    if len(duplicate_groups) > 0:

        print(
            "\nWARNING: duplicate tetrahedra detected."
        )

        shown = 0

        for group in duplicate_groups:

            indices = np.where(
                inverse == group
            )[0]

            print(
                "Duplicate tetrahedra:",
                indices[:10].tolist()
            )

            shown += 1

            if shown >= 20:
                break

    else:

        print(
            "PASS: no duplicate tetrahedra."
        )

    # -------------------------------------------------------------------------
    # FACE CONNECTIVITY
    # -------------------------------------------------------------------------

    print("\n" + "=" * 80)
    print("CHECK 3 — TETRAHEDRAL FACE CONNECTIVITY")
    print("=" * 80)

    face_owner = defaultdict(list)

    # Four triangular faces of each tetrahedron
    face_patterns = (
        (0, 1, 2),
        (0, 1, 3),
        (0, 2, 3),
        (1, 2, 3),
    )

    for i, tet in enumerate(tet_nodes):

        tet_tag = int(
            tet_tags[i]
        )

        for pattern in face_patterns:

            face = tuple(
                sorted(
                    int(tet[j])
                    for j in pattern
                )
            )

            face_owner[face].append(
                tet_tag
            )

    # -------------------------------------------------------------------------
    # FACE MULTIPLICITY
    # -------------------------------------------------------------------------

    multiplicities = defaultdict(int)

    for owners in face_owner.values():

        multiplicities[
            len(owners)
        ] += 1

    print(
        "\nFace multiplicity:"
    )

    for n in sorted(
        multiplicities
    ):

        print(
            f"  {n} tetrahedra -> "
            f"{multiplicities[n]:,} faces"
        )

    bad_faces = {
        face: owners
        for face, owners in face_owner.items()
        if len(owners) > 2
    }

    if bad_faces:

        print(
            "\nCRITICAL: faces shared by >2 tetrahedra:"
        )

        for face, owners in list(
            bad_faces.items()
        )[:30]:

            print(
                f"face={face}, "
                f"owners={owners}"
            )

        raise RuntimeError(
            "Non-manifold tetrahedral face connectivity detected."
        )

    print(
        "\nPASS: no triangular face belongs "
        "to more than two tetrahedra."
    )

    # -------------------------------------------------------------------------
    # NON-MANIFOLD EDGE CHECK
    # -------------------------------------------------------------------------

    print("\n" + "=" * 80)
    print("CHECK 4 — EDGE CONNECTIVITY")
    print("=" * 80)

    edge_faces = defaultdict(set)

    for face in face_owner:

        a, b, c = face

        edge_faces[
            tuple(sorted((a, b)))
        ].add(face)

        edge_faces[
            tuple(sorted((a, c)))
        ].add(face)

        edge_faces[
            tuple(sorted((b, c)))
        ].add(face)

    # An edge can have many faces on a legitimate closed 3D
    # mesh, so we don't declare >2 automatically bad.
    #
    # Instead identify extremely unusual connectivity.

    edge_counts = np.array(
        [
            len(v)
            for v in edge_faces.values()
        ],
        dtype=np.int64
    )

    print(
        f"Unique edges: {len(edge_counts):,}"
    )

    print(
        f"Minimum faces/edge: {edge_counts.min()}"
    )

    print(
        f"Maximum faces/edge: {edge_counts.max()}"
    )

    print(
        f"Average faces/edge: {edge_counts.mean():.3f}"
    )

    # -------------------------------------------------------------------------
    # NODE VALIDITY
    # -------------------------------------------------------------------------

    print("\n" + "=" * 80)
    print("CHECK 5 — NODE REFERENCES")
    print("=" * 80)

    node_tags, node_coords, _ = (
        gmsh.model.mesh.getNodes()
    )

    valid_nodes = set(
        int(x)
        for x in node_tags
    )

    invalid_refs = []

    for i, tet in enumerate(
        tet_nodes
    ):

        for node in tet:

            if int(node) not in valid_nodes:

                invalid_refs.append(
                    (
                        int(tet_tags[i]),
                        int(node)
                    )
                )

    print(
        f"Invalid node references: "
        f"{len(invalid_refs)}"
    )

    if invalid_refs:

        for x in invalid_refs[:20]:
            print(
                "BAD:",
                x
            )

        raise RuntimeError(
            "Tetrahedron references a "
            "nonexistent mesh node."
        )

    print(
        "PASS: all tetrahedral node "
        "references are valid."
    )

    # -------------------------------------------------------------------------
    # VOLUME CHECK
    # -------------------------------------------------------------------------

    print("\n" + "=" * 80)
    print("CHECK 6 — ZERO / NEGATIVE VOLUME")
    print("=" * 80)

    coords = np.asarray(
        node_coords
    ).reshape(
        -1,
        3
    )

    node_index = {
        int(tag): i
        for i, tag in enumerate(
            node_tags
        )
    }

    zero_volume = []
    negative_volume = []

    for i, tet in enumerate(
        tet_nodes
    ):

        try:

            p0 = coords[
                node_index[int(tet[0])]
            ]

            p1 = coords[
                node_index[int(tet[1])]
            ]

            p2 = coords[
                node_index[int(tet[2])]
            ]

            p3 = coords[
                node_index[int(tet[3])]
            ]

        except KeyError:
            continue

        v = np.dot(
            p1 - p0,
            np.cross(
                p2 - p0,
                p3 - p0
            )
        ) / 6.0

        if abs(v) < 1e-18:

            zero_volume.append(
                int(tet_tags[i])
            )

        elif v < 0:

            negative_volume.append(
                int(tet_tags[i])
            )

    print(
        f"Zero-volume tetrahedra: "
        f"{len(zero_volume):,}"
    )

    print(
        f"Negative-orientation tetrahedra: "
        f"{len(negative_volume):,}"
    )

    if zero_volume:

        print(
            "\nCRITICAL: zero-volume elements detected."
        )

        print(
            zero_volume[:20]
        )

    else:

        print(
            "PASS: no zero-volume tetrahedra."
        )

    # -------------------------------------------------------------------------
    # PHYSICAL SURFACE / VOLUME CONSISTENCY
    # -------------------------------------------------------------------------

    print("\n" + "=" * 80)
    print("CHECK 7 — PHYSICAL GROUP ELEMENTS")
    print("=" * 80)

    for dim, ptag in (
        gmsh.model.getPhysicalGroups()
    ):

        name = gmsh.model.getPhysicalName(
            dim,
            ptag
        )

        entities = (
            gmsh.model.getEntitiesForPhysicalGroup(
                dim,
                ptag
            )
        )

        count = 0

        for entity in entities:

            etypes, etags, _ = (
                gmsh.model.mesh.getElements(
                    dim,
                    entity
                )
            )

            for etype, arr in zip(
                etypes,
                etags
            ):

                count += len(arr)

        print(
            f"{dim:1d} "
            f"{ptag:2d} "
            f"{name:25s} "
            f"{count:,} elements"
        )

    # -------------------------------------------------------------------------
    # FINAL RESULT
    # -------------------------------------------------------------------------

    print("\n" + "=" * 80)
    print("DIAGNOSTIC RESULT")
    print("=" * 80)

    print(
        "\nThe V12 mesh contains:"
    )

    print(
        f"  {len(tet_nodes):,} tetrahedra"
    )

    print(
        f"  {len(node_tags):,} nodes"
    )

    print(
        f"  {len(face_owner):,} unique tetrahedral faces"
    )

    print(
        "\nIf all checks above PASS, the next step is "
        "NOT a larger simulation."
    )

    print(
        "Instead we should identify the exact "
        "tetrahedral/face topology that MFEM "
        "STable3D rejects."
    )

finally:

    gmsh.finalize()

    gc.collect()

V12 — TETRAHEDRAL CONNECTIVITY DIAGNOSTIC

Mesh:
/content/drive/MyDrive/Palace6/meshes/quantum_chip_mesh_8JJ_v12.msh

Tetrahedra: 11,925

CHECK 1 — REPEATED VERTICES
Degenerate/repeated-vertex tetrahedra: 0
PASS: no tetrahedron contains repeated vertex indices.

CHECK 2 — DUPLICATE TETRAHEDRA
Unique tetrahedra : 11,925
Duplicate groups  : 0
PASS: no duplicate tetrahedra.

CHECK 3 — TETRAHEDRAL FACE CONNECTIVITY

Face multiplicity:
  1 tetrahedra -> 4,534 faces
  2 tetrahedra -> 21,583 faces

PASS: no triangular face belongs to more than two tetrahedra.

CHECK 4 — EDGE CONNECTIVITY
Unique edges: 17,340
Minimum faces/edge: 2
Maximum faces/edge: 9
Average faces/edge: 4.519

CHECK 5 — NODE REFERENCES
Invalid node references: 0
PASS: all tetrahedral node references are valid.

CHECK 6 — ZERO / NEGATIVE VOLUME
Zero-volume tetrahedra: 0
Negative-orientation tetrahedra: 0
PASS: no zero-volume tetrahedra.

CHECK 7 — PHYSICAL GROUP ELEMENTS
2 10 Superconducting_Metal     13,954 elements
2 20 JJ1

In [ ]:
# =============================================================================
# V12 — MFEM STable3D-STYLE TOPOLOGY DIAGNOSTIC
# =============================================================================
#
# Palace previously aborted with:
#
#   MFEM abort: (r,c,f) = (35,4462,6735)
#   mfem::STable3D::operator()
#
# This script does NOT run Palace.
#
# It investigates:
#   tetrahedron -> face
#   face -> edge
#   edge -> vertex
#
# and reports suspicious local topology.
#
# =============================================================================

import gmsh
import numpy as np
from collections import defaultdict
import os
import gc

MESH = (
    "/content/drive/MyDrive/Palace6/meshes/"
    "quantum_chip_mesh_8JJ_v12.msh"
)

print("=" * 80)
print("V12 — MFEM STable3D-STYLE TOPOLOGY DIAGNOSTIC")
print("=" * 80)

gmsh.initialize()

try:

    gmsh.option.setNumber(
        "General.Terminal",
        0
    )

    gmsh.open(MESH)

    # -------------------------------------------------------------------------
    # READ TETRAHEDRA
    # -------------------------------------------------------------------------

    types, tags, blocks = (
        gmsh.model.mesh.getElements(3)
    )

    tet_nodes = None
    tet_tags = None

    for etype, etags, enodes in zip(
        types,
        tags,
        blocks
    ):

        if etype == 4:

            tet_tags = np.asarray(
                etags,
                dtype=np.int64
            )

            tet_nodes = np.asarray(
                enodes,
                dtype=np.int64
            ).reshape(-1, 4)

            break

    if tet_nodes is None:
        raise RuntimeError(
            "No tetrahedra found."
        )

    print(
        f"Tetrahedra: {len(tet_nodes):,}"
    )

    # -------------------------------------------------------------------------
    # MFEM-LIKE LOCAL TOPOLOGY
    # -------------------------------------------------------------------------
    #
    # A tetrahedron has:
    #
    #   4 vertices
    #   6 edges
    #   4 faces
    #
    # -------------------------------------------------------------------------

    EDGE_PATTERNS = [
        (0, 1),
        (0, 2),
        (0, 3),
        (1, 2),
        (1, 3),
        (2, 3),
    ]

    FACE_PATTERNS = [
        (0, 1, 2),
        (0, 1, 3),
        (0, 2, 3),
        (1, 2, 3),
    ]

    # -------------------------------------------------------------------------
    # BUILD GLOBAL EDGE TABLE
    # -------------------------------------------------------------------------

    edge_to_tets = defaultdict(list)
    edge_to_faces = defaultdict(set)

    tet_edges = []

    for ti, tet in enumerate(tet_nodes):

        edges = []

        for a, b in EDGE_PATTERNS:

            edge = tuple(
                sorted(
                    (
                        int(tet[a]),
                        int(tet[b])
                    )
                )
            )

            edges.append(edge)

            edge_to_tets[edge].append(ti)

        tet_edges.append(edges)

    print(
        f"Global edges: {len(edge_to_tets):,}"
    )

    # -------------------------------------------------------------------------
    # BUILD GLOBAL FACE TABLE
    # -------------------------------------------------------------------------

    face_to_tets = defaultdict(list)
    face_to_edges = {}

    tet_faces = []

    for ti, tet in enumerate(tet_nodes):

        faces = []

        for a, b, c in FACE_PATTERNS:

            face = tuple(
                sorted(
                    (
                        int(tet[a]),
                        int(tet[b]),
                        int(tet[c])
                    )
                )
            )

            faces.append(face)

            face_to_tets[face].append(ti)

            # Three edges of the face
            va, vb, vc = face

            edges = [
                tuple(sorted((va, vb))),
                tuple(sorted((va, vc))),
                tuple(sorted((vb, vc))),
            ]

            face_to_edges[face] = edges

            for edge in edges:
                edge_to_faces[edge].add(face)

        tet_faces.append(faces)

    print(
        f"Global faces: {len(face_to_tets):,}"
    )

    # -------------------------------------------------------------------------
    # CHECK A — FACE MULTIPLICITY
    # -------------------------------------------------------------------------

    print("\n" + "=" * 80)
    print("A — FACE MULTIPLICITY")
    print("=" * 80)

    face_hist = defaultdict(int)

    for face, owners in face_to_tets.items():

        face_hist[len(owners)] += 1

    for n in sorted(face_hist):

        print(
            f"{n} tetrahedra -> "
            f"{face_hist[n]:,} faces"
        )

    bad_faces = [
        (face, owners)
        for face, owners in face_to_tets.items()
        if len(owners) > 2
    ]

    if bad_faces:

        print(
            "\nCRITICAL: non-manifold faces detected."
        )

        for face, owners in bad_faces[:20]:

            print(
                f"face={face}"
            )

            print(
                f"tetrahedra={owners}"
            )

    else:

        print(
            "\nPASS: no face has more than "
            "two tetrahedral owners."
        )

    # -------------------------------------------------------------------------
    # CHECK B — EVERY TET HAS EXACTLY 4 UNIQUE FACES
    # -------------------------------------------------------------------------

    print("\n" + "=" * 80)
    print("B — TETRAHEDRON FACE STRUCTURE")
    print("=" * 80)

    bad_tets = []

    for ti, faces in enumerate(tet_faces):

        if len(faces) != 4:

            bad_tets.append(ti)

        if len(set(faces)) != 4:

            bad_tets.append(ti)

    bad_tets = sorted(
        set(bad_tets)
    )

    print(
        f"Bad tetrahedra: {len(bad_tets)}"
    )

    if bad_tets:

        for ti in bad_tets[:20]:

            print(
                "tet:",
                int(tet_tags[ti]),
                tet_nodes[ti]
            )

    else:

        print(
            "PASS: every tetrahedron has "
            "4 unique triangular faces."
        )

    # -------------------------------------------------------------------------
    # CHECK C — FACE MUST HAVE 3 UNIQUE EDGES
    # -------------------------------------------------------------------------

    print("\n" + "=" * 80)
    print("C — FACE EDGE STRUCTURE")
    print("=" * 80)

    bad_face_edges = []

    for face, edges in face_to_edges.items():

        if len(edges) != 3:

            bad_face_edges.append(
                (
                    face,
                    edges
                )
            )

        elif len(set(edges)) != 3:

            bad_face_edges.append(
                (
                    face,
                    edges
                )
            )

    print(
        f"Bad faces: {len(bad_face_edges)}"
    )

    if bad_face_edges:

        for face, edges in bad_face_edges[:20]:

            print(
                "face:",
                face,
                "edges:",
                edges
            )

    else:

        print(
            "PASS: every face has "
            "3 unique edges."
        )

    # -------------------------------------------------------------------------
    # CHECK D — EDGE ENDPOINT VALIDITY
    # -------------------------------------------------------------------------

    print("\n" + "=" * 80)
    print("D — EDGE VERTEX STRUCTURE")
    print("=" * 80)

    bad_edges = []

    for edge in edge_to_tets:

        if len(edge) != 2:

            bad_edges.append(
                edge
            )

        elif edge[0] == edge[1]:

            bad_edges.append(
                edge
            )

    print(
        f"Bad edges: {len(bad_edges)}"
    )

    if bad_edges:

        for edge in bad_edges[:20]:

            print(
                "BAD EDGE:",
                edge
            )

    else:

        print(
            "PASS: all edges have "
            "two distinct vertices."
        )

    # -------------------------------------------------------------------------
    # CHECK E — EDGE / FACE CONSISTENCY
    # -------------------------------------------------------------------------

    print("\n" + "=" * 80)
    print("E — EDGE/FACE CONSISTENCY")
    print("=" * 80)

    inconsistent_edges = []

    for edge, faces in edge_to_faces.items():

        for face in faces:

            if edge not in face_to_edges[face]:

                inconsistent_edges.append(
                    (
                        edge,
                        face
                    )
                )

    print(
        f"Inconsistent edge/face pairs: "
        f"{len(inconsistent_edges)}"
    )

    if inconsistent_edges:

        for x in inconsistent_edges[:20]:

            print(
                "BAD:",
                x
            )

    else:

        print(
            "PASS: edge/face connectivity "
            "is consistent."
        )

    # -------------------------------------------------------------------------
    # CHECK F — TET/FACE OWNER CONSISTENCY
    # -------------------------------------------------------------------------

    print("\n" + "=" * 80)
    print("F — TET/FACE OWNER CONSISTENCY")
    print("=" * 80)

    inconsistent_owners = []

    for ti, faces in enumerate(tet_faces):

        for face in faces:

            owners = face_to_tets[face]

            if ti not in owners:

                inconsistent_owners.append(
                    (
                        ti,
                        face,
                        owners
                    )
                )

    print(
        f"Inconsistent ownership records: "
        f"{len(inconsistent_owners)}"
    )

    if inconsistent_owners:

        for x in inconsistent_owners[:20]:

            print(
                "BAD:",
                x
            )

    else:

        print(
            "PASS: tetrahedron/face "
            "ownership is consistent."
        )

    # -------------------------------------------------------------------------
    # CHECK G — ISOLATED / SUSPICIOUS EDGES
    # -------------------------------------------------------------------------

    print("\n" + "=" * 80)
    print("G — EDGE FACE COUNTS")
    print("=" * 80)

    edge_face_hist = defaultdict(int)

    for edge, faces in edge_to_faces.items():

        edge_face_hist[
            len(faces)
        ] += 1

    for n in sorted(edge_face_hist):

        print(
            f"{n} faces -> "
            f"{edge_face_hist[n]:,} edges"
        )

    # -------------------------------------------------------------------------
    # CHECK H — PHYSICAL SURFACE SHARING
    # -------------------------------------------------------------------------

    print("\n" + "=" * 80)
    print("H — PHYSICAL SURFACE / TETRA CONNECTIVITY")
    print("=" * 80)

    # Physical surface triangles
    # are collected and compared against
    # tetrahedral boundary faces.

    physical_surface_faces = {}

    for dim, ptag in gmsh.model.getPhysicalGroups():

        if dim != 2:
            continue

        name = gmsh.model.getPhysicalName(
            dim,
            ptag
        )

        faces = set()

        entities = (
            gmsh.model.getEntitiesForPhysicalGroup(
                dim,
                ptag
            )
        )

        for entity in entities:

            etypes, etags, enodes = (
                gmsh.model.mesh.getElements(
                    dim,
                    entity
                )
            )

            for etype, nodes in zip(
                etypes,
                enodes
            ):

                # 3-node triangle
                if etype != 2:
                    continue

                tri_nodes = np.asarray(
                    nodes,
                    dtype=np.int64
                ).reshape(-1, 3)

                for tri in tri_nodes:

                    face = tuple(
                        sorted(
                            int(x)
                            for x in tri
                        )
                    )

                    faces.add(face)

        physical_surface_faces[name] = faces

        print(
            f"{name:25s}: "
            f"{len(faces):,} physical triangles"
        )

    # Compare physical triangles with
    # tetrahedral faces.
    #
    # Every physical surface triangle should
    # correspond to a tetrahedral face.

    for name, faces in physical_surface_faces.items():

        missing = [
            face
            for face in faces
            if face not in face_to_tets
        ]

        print(
            f"{name:25s}: "
            f"unmatched={len(missing):,}"
        )

        if missing:

            print(
                "WARNING: physical triangles "
                "not found in tetrahedral topology."
            )

            for face in missing[:10]:

                print(
                    "  ",
                    face
                )

    # -------------------------------------------------------------------------
    # FINAL
    # -------------------------------------------------------------------------

    print("\n" + "=" * 80)
    print("FINAL DIAGNOSTIC")
    print("=" * 80)

    total_problems = (
        len(bad_faces)
        + len(bad_tets)
        + len(bad_face_edges)
        + len(bad_edges)
        + len(inconsistent_edges)
        + len(inconsistent_owners)
    )

    print(
        f"\nTopology problem count: "
        f"{total_problems}"
    )

    if total_problems == 0:

        print(
            "\nPASS: Basic tetrahedral topology "
            "is internally consistent."
        )

        print(
            "\nThis is important:"
        )

        print(
            "The MFEM STable3D crash is therefore "
            "likely caused by the way the surface "
            "elements / physical boundaries are "
            "connected to the volume mesh, rather "
            "than a simple degenerate tetrahedron."
        )

    else:

        print(
            "\nFAIL: topology inconsistencies "
            "were detected."
        )

        print(
            "Do NOT run Palace."
        )

    print(
        "\nNo Palace process was started."
    )

finally:

    gmsh.finalize()
    gc.collect()

V12 — MFEM STable3D-STYLE TOPOLOGY DIAGNOSTIC
Tetrahedra: 11,925
Global edges: 17,340
Global faces: 26,117

A — FACE MULTIPLICITY
1 tetrahedra -> 4,534 faces
2 tetrahedra -> 21,583 faces

PASS: no face has more than two tetrahedral owners.

B — TETRAHEDRON FACE STRUCTURE
Bad tetrahedra: 0
PASS: every tetrahedron has 4 unique triangular faces.

C — FACE EDGE STRUCTURE
Bad faces: 0
PASS: every face has 3 unique edges.

D — EDGE VERTEX STRUCTURE
Bad edges: 0
PASS: all edges have two distinct vertices.

E — EDGE/FACE CONSISTENCY
Inconsistent edge/face pairs: 0
PASS: edge/face connectivity is consistent.

F — TET/FACE OWNER CONSISTENCY
Inconsistent ownership records: 0
PASS: tetrahedron/face ownership is consistent.

G — EDGE FACE COUNTS
2 faces -> 232 edges
3 faces -> 2,619 edges
4 faces -> 6,998 edges
5 faces -> 3,566 edges
6 faces -> 3,392 edges
7 faces -> 420 edges
8 faces -> 101 edges
9 faces -> 12 edges

H — PHYSICAL SURFACE / TETRA CONNECTIVITY
Superconducting_Metal    : 13,954 physi

In [ ]:
# =============================================================================
# ROBUST 8-JJ PALACE CONFORMAL MESH - V13 SAFE / LOW-MEMORY
# =============================================================================
#
# IMPORTANT:
#   - This script ONLY creates and validates the Gmsh mesh.
#   - It DOES NOT run Palace.
#   - It is deliberately conservative for Google Colab.
#
# Main goal:
#   Fix the V12 problem where physical PEC/JJ triangles were not actual
#   tetrahedral faces.
#
# Strategy:
#   GDS
#    -> actual metal geometry
#    -> actual JJ electrode gaps
#    -> silicon + vacuum volumes
#    -> OCC fragment ALL interface surfaces with volumes
#    -> generate ONE conformal 3D mesh
#    -> identify physical surfaces from the fragmented geometry
#    -> verify physical triangles are tetrahedral faces
#
# =============================================================================

import os
import sys
import math
import gc
from collections import defaultdict

import numpy as np

# -----------------------------------------------------------------------------
# USER SETTINGS
# -----------------------------------------------------------------------------

GDS_FILE = (
    "/content/drive/MyDrive/Palace6/meshes/"
    "quantum_chip_export_manuscript.gds"
)

OUT_FILE = (
    "/content/drive/MyDrive/Palace6/meshes/"
    "quantum_chip_mesh_8JJ_v13.msh"
)

# GDS geometry is in mm.
# Palace later uses L0 = 0.001 m/mm.
#
# Chip:
CHIP_X = 14.0       # mm
CHIP_Y = 14.0       # mm

# z coordinates in mm
Z_SILICON_BOTTOM = -0.5
Z_INTERFACE       =  0.0
Z_VACUUM_TOP      =  0.5

# Conservative mesh sizes
BULK_SIZE = 0.50    # 500 um
JJ_SIZE   = 0.05    # 50 um

# HARD SAFETY LIMIT
#
# We intentionally keep this MUCH lower than the previous
# 11.9k mesh? 500k is still tiny compared with the failed
# 13.8 million tetrahedron mesh.
MAX_TETS = 500_000

# If mesh exceeds this, immediately abort.
MAX_NODES = 250_000

# GDS layers
MAIN_LAYER = 1
MAIN_DATATYPE = 0

PAD_LAYER = 1
PAD_DATATYPE = 10

JJ_LAYER = 53
JJ_DATATYPE = 0

# Known JJ locations, mm
JJ_LOCATIONS = [
    (-1.1325,  0.0000),
    (-0.8675,  0.0000),
    ( 1.2675,  0.0000),
    ( 1.5325,  0.0000),

    (-1.1325, -2.7000),
    (-0.8675, -2.7000),
    ( 1.2675, -2.7000),
    ( 1.5325, -2.7000),
]

# Physical attributes
VACUUM_ATTR = 1
SILICON_ATTR = 2
METAL_ATTR = 10

JJ_ATTRS = [20, 21, 22, 23, 24, 25, 26, 27]

# =============================================================================
# IMPORTS
# =============================================================================

try:
    import gmsh
except Exception as e:
    raise RuntimeError(
        "gmsh Python package is required. "
        "Install with: pip install gmsh"
    ) from e

try:
    import gdstk
except Exception as e:
    raise RuntimeError(
        "gdstk is required. Install with: pip install gdstk"
    ) from e

try:
    from shapely.geometry import Polygon, MultiPolygon, box
    from shapely.ops import unary_union
except Exception as e:
    raise RuntimeError(
        "shapely is required. Install with: pip install shapely"
    ) from e


# =============================================================================
# HELPERS
# =============================================================================

def polygon_from_gdstk(poly):
    """
    Convert a gdstk polygon to a Shapely Polygon.
    """
    pts = np.asarray(poly.points, dtype=float)

    if len(pts) < 3:
        return None

    p = Polygon(pts)

    if not p.is_valid:
        p = p.buffer(0)

    if p.is_empty:
        return None

    return p


def iter_gdstk_polygons(cell):
    """
    Recursively obtain polygons from a gdstk cell through flattened
    references.
    """
    # Flatten the cell in a copy so references are resolved.
    tmp = cell.copy(name="__flattened_tmp__")
    tmp.flatten()

    for p in tmp.polygons:
        yield p


def shapely_to_polygons(geom):
    """
    Return a list of valid Polygon objects.
    """
    if geom is None or geom.is_empty:
        return []

    if geom.geom_type == "Polygon":
        return [geom]

    if geom.geom_type == "MultiPolygon":
        return list(geom.geoms)

    if geom.geom_type == "GeometryCollection":
        out = []
        for g in geom.geoms:
            if g.geom_type == "Polygon":
                out.append(g)
            elif g.geom_type == "MultiPolygon":
                out.extend(list(g.geoms))
        return out

    return []


def add_planar_surface(poly, z=0.0):
    """
    Create a Gmsh OCC plane surface from a Shapely polygon.

    Returns:
        surface tag
    """

    if poly.is_empty:
        return None

    # Repair tiny geometric defects.
    if not poly.is_valid:
        poly = poly.buffer(0)

    if poly.is_empty:
        return None

    outer = list(poly.exterior.coords)
    if len(outer) < 4:
        return None

    def make_loop(coords):
        point_tags = []
        line_tags = []

        # Do not duplicate the closing point.
        coords = list(coords[:-1])

        for x, y in coords:
            point_tags.append(
                gmsh.model.occ.addPoint(
                    float(x),
                    float(y),
                    float(z)
                )
            )

        n = len(point_tags)

        for i in range(n):
            a = point_tags[i]
            b = point_tags[(i + 1) % n]

            if a == b:
                continue

            line_tags.append(
                gmsh.model.occ.addLine(a, b)
            )

        if len(line_tags) < 3:
            return None

        return gmsh.model.occ.addCurveLoop(line_tags)

    outer_loop = make_loop(outer)

    if outer_loop is None:
        return None

    holes = []

    for ring in poly.interiors:
        loop = make_loop(list(ring.coords))
        if loop is not None:
            holes.append(loop)

    try:
        surface = gmsh.model.occ.addPlaneSurface(
            [outer_loop] + holes
        )
    except Exception:
        return None

    return surface


def surface_center(surface_tag):
    """
    Geometric center of a surface.
    """
    try:
        com = gmsh.model.occ.getCenterOfMass(
            2,
            surface_tag
        )
        return np.asarray(com, dtype=float)
    except Exception:
        return None


def surface_area(surface_tag):
    try:
        return float(
            gmsh.model.occ.getMass(2, surface_tag)
        )
    except Exception:
        return 0.0


def print_ram(label=""):
    """
    Colab-safe RAM report.
    """
    try:
        import psutil

        vm = psutil.virtual_memory()

        print(
            f"[RAM {label}] "
            f"Total={vm.total/1024**3:.2f} GB | "
            f"Available={vm.available/1024**3:.2f} GB | "
            f"Used={vm.used/1024**3:.2f} GB"
        )
    except Exception:
        pass


# =============================================================================
# START
# =============================================================================

print("=" * 80)
print("ROBUST 8-JJ PALACE CONFORMAL MESH - V13")
print("=" * 80)

print(f"GDS:")
print(GDS_FILE)
print()
print(f"Output:")
print(OUT_FILE)
print()
print(f"Bulk mesh : {BULK_SIZE*1000:.0f} um")
print(f"JJ mesh   : {JJ_SIZE*1000:.0f} um")
print(f"MAX TETRAHEDRA : {MAX_TETS:,}")
print(f"MAX NODES      : {MAX_NODES:,}")

print_ram("START")


# =============================================================================
# READ GDS
# =============================================================================

print()
print("=" * 80)
print("READING GDS")
print("=" * 80)

if not os.path.isfile(GDS_FILE):
    raise FileNotFoundError(GDS_FILE)

lib = gdstk.read_gds(GDS_FILE)

print(f"Loaded {len(lib.cells)} cells.")

# Locate the chip cell.
cell_names = [c.name for c in lib.cells]

print("Available cells:")
for name in cell_names:
    print(" ", name)

preferred = [
    "TOP_main_1",
    "TOP",
]

chip_cell = None

for name in preferred:
    for c in lib.cells:
        if c.name == name:
            chip_cell = c
            break
    if chip_cell is not None:
        break

if chip_cell is None:
    # Fall back to TOP if present.
    for c in lib.cells:
        if c.name.upper() == "TOP":
            chip_cell = c
            break

if chip_cell is None:
    raise RuntimeError(
        "Could not identify chip top cell."
    )

print(f"Using chip cell: {chip_cell.name}")


# =============================================================================
# EXTRACT POLYGONS
# =============================================================================

print()
print("=" * 80)
print("EXTRACTING GDS GEOMETRY")
print("=" * 80)

flat = chip_cell.copy(name="__V13_FLAT__")
flat.flatten()

main_polys = []
pad_polys = []
jj_ref_polys = []

for p in flat.polygons:

    layer = int(p.layer)
    datatype = int(p.datatype)

    geom = polygon_from_gdstk(p)

    if geom is None:
        continue

    if layer == MAIN_LAYER and datatype == MAIN_DATATYPE:
        main_polys.append(geom)

    elif layer == PAD_LAYER and datatype == PAD_DATATYPE:
        pad_polys.append(geom)

    elif layer == JJ_LAYER and datatype == JJ_DATATYPE:
        jj_ref_polys.append(geom)


print(f"Main metal polygons : {len(main_polys)}")
print(f"Pad polygons         : {len(pad_polys)}")
print(f"JJ reference polygons: {len(jj_ref_polys)}")


# =============================================================================
# MAIN METAL
# =============================================================================

print()
print("Uniting main superconducting metal...")

main_metal = unary_union(main_polys)

if main_metal.is_empty:
    raise RuntimeError("Main metal geometry is empty.")

if not main_metal.is_valid:
    main_metal = main_metal.buffer(0)

print(f"Main metal geometry: {main_metal.geom_type}")
print(
    f"Main metal area: "
    f"{main_metal.area:.12f} mm²"
)


# =============================================================================
# MATCH JJ REFERENCES
# =============================================================================

if len(jj_ref_polys) != 8:
    raise RuntimeError(
        f"Expected 8 JJ references, found "
        f"{len(jj_ref_polys)}."
    )

print()
print("=" * 80)
print("MATCHING 8 JJ REFERENCES")
print("=" * 80)

matched_refs = []

for i, (x0, y0) in enumerate(JJ_LOCATIONS):

    best = None
    best_d = float("inf")

    for p in jj_ref_polys:

        c = p.centroid

        d = math.hypot(
            c.x - x0,
            c.y - y0
        )

        if d < best_d:
            best_d = d
            best = p

    if best is None:
        raise RuntimeError(
            f"JJ{i+1} reference not found."
        )

    if best_d > 1e-5:
        raise RuntimeError(
            f"JJ{i+1} reference mismatch: "
            f"{best_d*1000:.3f} um"
        )

    matched_refs.append(best)

    print(
        f"JJ{i+1}: "
        f"centroid=({best.centroid.x:.6f}, "
        f"{best.centroid.y:.6f}) mm, "
        f"error={best_d*1000:.3f} um"
    )


# =============================================================================
# FIND ACTUAL ELECTRODE PAIRS
# =============================================================================

print()
print("=" * 80)
print("IDENTIFYING ACTUAL JJ ELECTRODE PAIRS")
print("=" * 80)

jj_gaps = []

for i, ref in enumerate(matched_refs):

    cx = ref.centroid.x
    cy = ref.centroid.y

    # Find pad polygons near this JJ.
    candidates = []

    for p in pad_polys:

        c = p.centroid

        dx = c.x - cx
        dy = c.y - cy

        # Local search window.
        if abs(dx) <= 0.08 and abs(dy) <= 0.02:
            candidates.append(p)

    if len(candidates) < 2:
        raise RuntimeError(
            f"JJ{i+1}: only "
            f"{len(candidates)} nearby pad candidates."
        )

    # The electrodes are separated primarily in x.
    candidates.sort(
        key=lambda p: p.centroid.x
    )

    left = candidates[0]
    right = candidates[-1]

    # Verify both are actually close in y.
    if (
        abs(left.centroid.y - cy) > 0.01
        or
        abs(right.centroid.y - cy) > 0.01
    ):
        raise RuntimeError(
            f"JJ{i+1}: electrode y positions invalid."
        )

    lx0, ly0, lx1, ly1 = left.bounds
    rx0, ry0, rx1, ry1 = right.bounds

    gap_x0 = lx1
    gap_x1 = rx0

    gap_y0 = max(ly0, ry0)
    gap_y1 = min(ly1, ry1)

    if gap_x1 <= gap_x0:
        raise RuntimeError(
            f"JJ{i+1}: electrode pads overlap."
        )

    if gap_y1 <= gap_y0:
        raise RuntimeError(
            f"JJ{i+1}: electrode y ranges do not overlap."
        )

    gap = box(
        gap_x0,
        gap_y0,
        gap_x1,
        gap_y1
    )

    gc = gap.centroid

    center_error = math.hypot(
        gc.x - cx,
        gc.y - cy
    )

    width_um = gap.bounds[2] - gap.bounds[0]
    height_um = gap.bounds[3] - gap.bounds[1]

    width_um *= 1000
    height_um *= 1000

    print()
    print(f"JJ{i+1}:")
    print(
        f"  gap = "
        f"{width_um:.3f} x "
        f"{height_um:.3f} um"
    )
    print(
        f"  center error = "
        f"{center_error*1000:.3f} um"
    )

    if center_error > 1e-6:
        raise RuntimeError(
            f"JJ{i+1}: gap center mismatch."
        )

    jj_gaps.append(gap)


# =============================================================================
# VERY IMPORTANT:
#
# The actual electrode pads are already part of the GDS geometry.
# We DO NOT manufacture an artificial JJ gap.
#
# The JJ gap itself is retained as an interface surface.
# =============================================================================

print()
print("PASS: 8 physical JJ gaps recovered from GDS.")


# =============================================================================
# BUILD COMPLETE METAL
# =============================================================================

print()
print("=" * 80)
print("BUILDING COMPLETE SUPERCONDUCTING METAL")
print("=" * 80)

# Only select the pad polygons that actually belong to one of the 8 JJs.
selected_pads = []

for gap in jj_gaps:

    gx0, gy0, gx1, gy1 = gap.bounds

    for p in pad_polys:

        # Pad must touch the JJ neighborhood.
        expanded = gap.buffer(0.00001)

        if p.intersects(expanded):
            selected_pads.append(p)

# Remove duplicate geometries.
unique_pads = []

for p in selected_pads:

    if not any(
        p.equals(q)
        for q in unique_pads
    ):
        unique_pads.append(p)

print(
    f"Selected JJ electrode polygons: "
    f"{len(unique_pads)}"
)

# Union main metal + actual JJ electrode pads.
complete_metal = unary_union(
    [main_metal] + unique_pads
)

if not complete_metal.is_valid:
    complete_metal = complete_metal.buffer(0)

print(
    f"Complete metal geometry: "
    f"{complete_metal.geom_type}"
)

print(
    f"Complete metal area: "
    f"{complete_metal.area:.12f} mm²"
)


# =============================================================================
# VERIFY JJ GAPS ARE ACTUALLY EMPTY
# =============================================================================

print()
print("=" * 80)
print("VERIFYING PHYSICAL JJ GAPS")
print("=" * 80)

for i, gap in enumerate(jj_gaps):

    overlap = complete_metal.intersection(gap).area

    print(
        f"JJ{i+1}: "
        f"metal overlap = "
        f"{overlap:.6e} mm²"
    )

    # A physical JJ gap must not be covered by PEC metal.
    if overlap > 1e-10:
        raise RuntimeError(
            f"JJ{i+1} gap is occupied by metal."
        )

print("PASS: all JJ gaps are physically open.")


# =============================================================================
# INITIALIZE GMSH
# =============================================================================

print()
print("=" * 80)
print("INITIALIZING GMSH")
print("=" * 80)

gmsh.initialize()

try:

    gmsh.option.setNumber(
        "General.Terminal",
        1
    )

    gmsh.option.setNumber(
        "General.Verbosity",
        1
    )

    gmsh.model.add(
        "Palace_V13_Conformal"
    )

    occ = gmsh.model.occ


    # -------------------------------------------------------------------------
    # 3D VOLUMES
    # -------------------------------------------------------------------------

    print("Creating silicon volume...")

    silicon = occ.addBox(
        -CHIP_X/2,
        -CHIP_Y/2,
        Z_SILICON_BOTTOM,
        CHIP_X,
        CHIP_Y,
        Z_INTERFACE - Z_SILICON_BOTTOM
    )

    print("Creating vacuum volume...")

    vacuum = occ.addBox(
        -CHIP_X/2,
        -CHIP_Y/2,
        Z_INTERFACE,
        CHIP_X,
        CHIP_Y,
        Z_VACUUM_TOP - Z_INTERFACE
    )


    # -------------------------------------------------------------------------
    # CREATE INTERFACE SURFACES
    #
    # IMPORTANT:
    # These are not independently meshed later.
    # They are OCC entities used in the volume fragmentation.
    # -------------------------------------------------------------------------

    print("Creating metal interface surfaces...")

    metal_polygons = shapely_to_polygons(
        complete_metal
    )

    if len(metal_polygons) == 0:
        raise RuntimeError(
            "No metal polygons after union."
        )

    metal_surfaces = []

    for p in metal_polygons:

        s = add_planar_surface(
            p,
            Z_INTERFACE
        )

        if s is not None:
            metal_surfaces.append(s)

    print(
        f"Metal interface surfaces: "
        f"{len(metal_surfaces)}"
    )


    # -------------------------------------------------------------------------
    # CREATE ACTUAL JJ SURFACES
    # -------------------------------------------------------------------------

    print("Creating actual JJ interface surfaces...")

    jj_surface_tags = []

    for i, gap in enumerate(jj_gaps):

        s = add_planar_surface(
            gap,
            Z_INTERFACE
        )

        if s is None:
            raise RuntimeError(
                f"Could not create JJ{i+1} surface."
            )

        jj_surface_tags.append(s)

        print(
            f"JJ{i+1}: surface={s}"
        )


    # -------------------------------------------------------------------------
    # FRAGMENT VOLUMES WITH INTERFACE GEOMETRY
    #
    # This is the critical difference from V12.
    #
    # The interface surfaces are inserted into the OCC volume topology BEFORE
    # meshing, forcing Gmsh to share nodes between surface and volume elements.
    # -------------------------------------------------------------------------

    print()
    print("=" * 80)
    print("OCC CONFORMAL FRAGMENTATION")
    print("=" * 80)

    volume_objects = [
        (3, silicon),
        (3, vacuum)
    ]

    interface_objects = (
        [(2, s) for s in metal_surfaces]
        +
        [(2, s) for s in jj_surface_tags]
    )

    all_tools = volume_objects + interface_objects

    print(
        f"Volume entities     : "
        f"{len(volume_objects)}"
    )

    print(
        f"Interface surfaces  : "
        f"{len(interface_objects)}"
    )

    print("Fragmenting...")

    frag_out, frag_map = occ.fragment(
        volume_objects,
        interface_objects,
        removeObject=True,
        removeTool=False
    )

    occ.synchronize()

    print(
        f"Fragmentation returned "
        f"{len(frag_out)} entities."
    )


    # -------------------------------------------------------------------------
    # GET RESULTING VOLUMES
    # -------------------------------------------------------------------------

    volumes = [
        tag
        for dim, tag in frag_out
        if dim == 3
    ]

    print(
        f"Resulting 3D volumes: "
        f"{len(volumes)}"
    )

    if len(volumes) != 2:
        raise RuntimeError(
            "Expected exactly two 3D volumes."
        )


    # Identify silicon/vacuum by center of mass.
    silicon_volume = None
    vacuum_volume = None

    for tag in volumes:

        com = occ.getCenterOfMass(
            3,
            tag
        )

        print(
            f"Volume {tag}: "
            f"center z={com[2]:+.6f} mm"
        )

        if com[2] < 0:
            silicon_volume = tag

        elif com[2] > 0:
            vacuum_volume = tag

    if silicon_volume is None:
        raise RuntimeError(
            "Could not identify silicon volume."
        )

    if vacuum_volume is None:
        raise RuntimeError(
            "Could not identify vacuum volume."
        )

    print(
        f"Silicon volume: {silicon_volume}"
    )

    print(
        f"Vacuum volume: {vacuum_volume}"
    )


    # -------------------------------------------------------------------------
    # PHYSICAL VOLUMES
    # -------------------------------------------------------------------------

    gmsh.model.addPhysicalGroup(
        3,
        [vacuum_volume],
        VACUUM_ATTR
    )

    gmsh.model.setPhysicalName(
        3,
        VACUUM_ATTR,
        "Vacuum"
    )

    gmsh.model.addPhysicalGroup(
        3,
        [silicon_volume],
        SILICON_ATTR
    )

    gmsh.model.setPhysicalName(
        3,
        SILICON_ATTR,
        "Silicon_Substrate"
    )


    # -------------------------------------------------------------------------
    # IDENTIFY ACTUAL INTERFACE FACES
    #
    # We now inspect the faces belonging to the fragmented volumes.
    # Only THESE faces can become Palace physical surfaces.
    # -------------------------------------------------------------------------

    print()
    print("=" * 80)
    print("IDENTIFYING CONFORMAL INTERFACE FACES")
    print("=" * 80)

    interface_faces = []

    for vtag in [silicon_volume, vacuum_volume]:

        boundary = gmsh.model.getBoundary(
            [(3, vtag)],
            oriented=False,
            recursive=False
        )

        for dim, ftag in boundary:

            if dim != 2:
                continue

            com = occ.getCenterOfMass(
                2,
                ftag
            )

            # Interface is z = 0.
            if abs(com[2] - Z_INTERFACE) < 1e-8:

                interface_faces.append(
                    ftag
                )

    interface_faces = sorted(
        set(interface_faces)
    )

    print(
        f"Interface faces at z=0: "
        f"{len(interface_faces)}"
    )

    if not interface_faces:
        raise RuntimeError(
            "No z=0 interface faces found."
        )


    # -------------------------------------------------------------------------
    # CLASSIFY JJ FACES BY ACTUAL GEOMETRIC LOCATION
    # -------------------------------------------------------------------------

    print()
    print("Classifying JJ faces...")

    jj_faces = [[] for _ in range(8)]
    pec_faces = []

    for ftag in interface_faces:

        com = occ.getCenterOfMass(
            2,
            ftag
        )

        area = occ.getMass(
            2,
            ftag
        )

        matched_jj = None

        for i, gap in enumerate(jj_gaps):

            gx0, gy0, gx1, gy1 = gap.bounds

            # Require center to be inside JJ gap.
            if (
                gx0 - 1e-8 <= com[0] <= gx1 + 1e-8
                and
                gy0 - 1e-8 <= com[1] <= gy1 + 1e-8
            ):
                matched_jj = i
                break

        if matched_jj is not None:

            jj_faces[matched_jj].append(
                ftag
            )

        else:

            # Any other z=0 interface face is superconducting metal.
            pec_faces.append(
                ftag
            )


    # -------------------------------------------------------------------------
    # JJ VALIDATION
    # -------------------------------------------------------------------------

    for i in range(8):

        print(
            f"JJ{i+1}: "
            f"{len(jj_faces[i])} conformal "
            f"interface faces"
        )

        if len(jj_faces[i]) == 0:
            raise RuntimeError(
                f"JJ{i+1} has no conformal "
                f"interface face."
            )


    # -------------------------------------------------------------------------
    # PEC VALIDATION
    # -------------------------------------------------------------------------

    print()
    print(
        f"PEC interface faces: "
        f"{len(pec_faces)}"
    )

    if len(pec_faces) == 0:
        raise RuntimeError(
            "No PEC interface faces found."
        )


    # -------------------------------------------------------------------------
    # PHYSICAL PEC
    # -------------------------------------------------------------------------

    gmsh.model.addPhysicalGroup(
        2,
        pec_faces,
        METAL_ATTR
    )

    gmsh.model.setPhysicalName(
        2,
        METAL_ATTR,
        "Superconducting_Metal"
    )


    # -------------------------------------------------------------------------
    # PHYSICAL JJs
    # -------------------------------------------------------------------------

    for i in range(8):

        attr = JJ_ATTRS[i]

        gmsh.model.addPhysicalGroup(
            2,
            jj_faces[i],
            attr
        )

        gmsh.model.setPhysicalName(
            2,
            attr,
            f"JJ{i+1}"
        )


    # -------------------------------------------------------------------------
    # MESH SIZE
    # -------------------------------------------------------------------------

    print()
    print("=" * 80)
    print("CONFIGURING CONSERVATIVE MESH")
    print("=" * 80)

    # Global coarse mesh.
    gmsh.option.setNumber(
        "Mesh.MeshSizeMin",
        BULK_SIZE
    )

    gmsh.option.setNumber(
        "Mesh.MeshSizeMax",
        BULK_SIZE
    )

    # First-order tetrahedra.
    gmsh.option.setNumber(
        "Mesh.ElementOrder",
        1
    )

    # Avoid unnecessary optimization that can consume memory.
    gmsh.option.setNumber(
        "Mesh.Optimize",
        0
    )

    gmsh.option.setNumber(
        "Mesh.OptimizeNetgen",
        0
    )

    # JJ-local refinement.
    #
    # Use distance fields rather than creating another independent mesh.
    jj_edges = []

    for i in range(8):

        for ftag in jj_faces[i]:

            boundary = gmsh.model.getBoundary(
                [(2, ftag)],
                oriented=False,
                recursive=False
            )

            for dim, tag in boundary:

                if dim == 1:
                    jj_edges.append(tag)

    jj_edges = sorted(
        set(jj_edges)
    )

    print(
        f"JJ boundary curves: "
        f"{len(jj_edges)}"
    )

    if jj_edges:

        dist = gmsh.model.mesh.field.add(
            "Distance"
        )

        gmsh.model.mesh.field.setNumbers(
            dist,
            "CurvesList",
            jj_edges
        )

        threshold = gmsh.model.mesh.field.add(
            "Threshold"
        )

        gmsh.model.mesh.field.setNumber(
            threshold,
            "InField",
            dist
        )

        gmsh.model.mesh.field.setNumber(
            threshold,
            "SizeMin",
            JJ_SIZE
        )

        gmsh.model.mesh.field.setNumber(
            threshold,
            "SizeMax",
            BULK_SIZE
        )

        gmsh.model.mesh.field.setNumber(
            threshold,
            "DistMin",
            0.03
        )

        gmsh.model.mesh.field.setNumber(
            threshold,
            "DistMax",
            0.15
        )

        gmsh.model.mesh.field.setAsBackgroundMesh(
            threshold
        )


    # -------------------------------------------------------------------------
    # FINAL SYNCHRONIZATION
    # -------------------------------------------------------------------------

    occ.synchronize()


    # -------------------------------------------------------------------------
    # GENERATE 3D MESH
    # -------------------------------------------------------------------------

    print()
    print("=" * 80)
    print("GENERATING LOW-MEMORY CONFORMAL 3D MESH")
    print("=" * 80)

    print_ram("BEFORE MESH")

    gmsh.model.mesh.generate(3)

    print_ram("AFTER MESH")


    # -------------------------------------------------------------------------
    # MESH STATISTICS
    # -------------------------------------------------------------------------

    node_tags, node_coords, _ = (
        gmsh.model.mesh.getNodes()
    )

    node_count = len(node_tags)

    tet_count = 0

    # Gmsh type 4 = 4-node tetrahedron.
    types, elem_tags, elem_nodes = (
        gmsh.model.mesh.getElements(3)
    )

    for etype, tags, nodes in zip(
        types,
        elem_tags,
        elem_nodes
    ):

        if etype == 4:
            tet_count += len(tags)


    print()
    print("=" * 80)
    print("MESH STATISTICS")
    print("=" * 80)

    print(
        f"Nodes      : {node_count:,}"
    )

    print(
        f"Tetrahedra : {tet_count:,}"
    )

    # HARD SAFETY CHECK.
    if node_count > MAX_NODES:

        print(
            f"\nABORT: node count "
            f"{node_count:,} exceeds "
            f"limit {MAX_NODES:,}."
        )

        raise RuntimeError(
            "Mesh too large for safe Colab run."
        )

    if tet_count > MAX_TETS:

        print(
            f"\nABORT: tetrahedron count "
            f"{tet_count:,} exceeds "
            f"limit {MAX_TETS:,}."
        )

        raise RuntimeError(
            "Mesh too large for safe Colab run."
        )

    print(
        "PASS: mesh is below safety limits."
    )


    # -------------------------------------------------------------------------
    # WRITE MESH
    # -------------------------------------------------------------------------

    print()
    print("=" * 80)
    print("WRITING MSH 2.2")
    print("=" * 80)

    gmsh.option.setNumber(
        "Mesh.MshFileVersion",
        2.2
    )

    gmsh.write(
        OUT_FILE
    )

    print(
        f"Mesh written:\n{OUT_FILE}"
    )

    print(
        f"Mesh size: "
        f"{os.path.getsize(OUT_FILE)/1024**2:.2f} MB"
    )


    # -------------------------------------------------------------------------
    # PHYSICAL GROUP SUMMARY
    # -------------------------------------------------------------------------

    print()
    print("=" * 80)
    print("PHYSICAL GROUPS")
    print("=" * 80)

    groups = gmsh.model.getPhysicalGroups()

    for dim, attr in groups:

        name = gmsh.model.getPhysicalName(
            dim,
            attr
        )

        entities = (
            gmsh.model.getEntitiesForPhysicalGroup(
                dim,
                attr
            )
        )

        print(
            f"dim={dim}, "
            f"id={attr}, "
            f"name={name}, "
            f"entities={len(entities)}"
        )


    # -------------------------------------------------------------------------
    # SAVE A SMALL TOPOLOGY REPORT
    # -------------------------------------------------------------------------

    report_file = OUT_FILE.replace(
        ".msh",
        "_report.txt"
    )

    with open(
        report_file,
        "w"
    ) as f:

        f.write(
            "PALACE V13 CONFORMAL MESH REPORT\n"
        )

        f.write(
            "=================================\n\n"
        )

        f.write(
            f"Nodes: {node_count}\n"
        )

        f.write(
            f"Tetrahedra: {tet_count}\n\n"
        )

        f.write(
            "Physical groups:\n"
        )

        for dim, attr in groups:

            name = gmsh.model.getPhysicalName(
                dim,
                attr
            )

            f.write(
                f"dim={dim} "
                f"id={attr} "
                f"name={name}\n"
            )

        f.write("\nJJ faces:\n")

        for i in range(8):

            f.write(
                f"JJ{i+1}: "
                f"{len(jj_faces[i])} faces\n"
            )

        f.write(
            f"\nPEC faces: "
            f"{len(pec_faces)}\n"
        )

    print(
        f"\nReport:\n{report_file}"
    )

    print_ram("FINISHED")

    print()
    print("=" * 80)
    print("V13 MESH GENERATION COMPLETE")
    print("=" * 80)

    print()
    print("IMPORTANT:")
    print("DO NOT RUN PALACE YET.")
    print()
    print("First run the separate MSH topology checker.")
    print("The critical requirement is:")
    print()
    print("  PEC unmatched physical triangles = 0")
    print("  JJ1 unmatched = 0")
    print("  JJ2 unmatched = 0")
    print("  ...")
    print("  JJ8 unmatched = 0")
    print()
    print("Only after that should Palace be started.")

finally:

    try:
        gmsh.finalize()
    except Exception:
        pass

    gc.collect()

    print()
    print("Gmsh finalized.")
    print_ram("POST-GMSH")

ROBUST 8-JJ PALACE CONFORMAL MESH - V13
GDS:
/content/drive/MyDrive/Palace6/meshes/quantum_chip_export_manuscript.gds

Output:
/content/drive/MyDrive/Palace6/meshes/quantum_chip_mesh_8JJ_v13.msh

Bulk mesh : 500 um
JJ mesh   : 50 um
MAX TETRAHEDRA : 500,000
MAX NODES      : 250,000
[RAM START] Total=50.99 GB | Available=49.43 GB | Used=1.02 GB

READING GDS
Loaded 19 cells.
Available cells:
  TOP
  TOP_main
  TOP_main_1
  ground_main_1
  my_other_junction
  FakeJunction_02
  FakeJunction_01
  pads_my_other_junction_QComponent_is_1_name_is_poly4
  pads_my_other_junction_QComponent_is_1_name_is_poly5
  pads_my_other_junction_QComponent_is_2_name_is_poly4
  pads_my_other_junction_QComponent_is_2_name_is_poly5
  pads_my_other_junction_QComponent_is_3_name_is_poly4
  pads_my_other_junction_QComponent_is_3_name_is_poly5
  pads_my_other_junction_QComponent_is_4_name_is_poly4
  pads_my_other_junction_QComponent_is_4_name_is_poly5
  TOP_main_1_NoCheese_99
  TOP_main_1_one_hole
  TOP_main_1_Chees

# New Section